# Spectral Joint Flow (SJFPO) on Kaggle (offline)

Train the spectral flow policy on MuJoCo with internet disabled. The cells
below recreate the full source tree (`src/`, `scripts/train.py`, `configs/`)
next to this notebook via `%%writefile`, install deps from the attached
wheels dataset, then run `scripts/train.py` (eval runs inside training and
lands in `eval_history.csv`).

## 0. Attachments and secrets

1. Attach your wheels dataset (the pinned `requirements-linux.txt` bundle)
   and set `WHEELS_DIR` in the install cell to its `/kaggle/input/...` path.
2. Set the `WANDB_API_KEY` secret so wandb login happens automatically
   (no manual paste). Training also runs without it.
3. Run all cells top to bottom.

In [ ]:
# Recreate the source tree next to the notebook (writefile needs the dirs).
!mkdir -p src/sjfpo scripts configs

In [ ]:
# Detect runtime and load secrets from the right store.
import os


def detect_env():
    try:
        if "google.colab" in str(get_ipython()):
            return "colab"
    except NameError:
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    return "local"


ENV_NAME = detect_env()
print(f"Environment: {ENV_NAME}")


def load_secret(key_name):
    try:
        if ENV_NAME == "colab":
            from google.colab import userdata

            return userdata.get(key_name) or None
        if ENV_NAME == "kaggle":
            from kaggle_secrets import UserSecretsClient

            return UserSecretsClient().get_secret(key_name) or None
        return os.getenv(key_name)
    except Exception as e:
        print(f"load_secret({key_name}) failed: {e}")
        return None

In [ ]:
%%writefile src/sjfpo/__init__.py

In [ ]:
%%writefile src/sjfpo/analysis.py
"""Benchmark analysis: bootstrap CIs, IQM, superiority, normalization, plots, CSV."""

from __future__ import annotations

import csv
import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import matplotlib

matplotlib.use("Agg")  # Non-interactive backend for tests
import matplotlib.pyplot as plt
import numpy as np

from sjfpo.config import RunSpec


class AnalysisError(Exception):
    """Analysis cannot proceed (missing/invalid data)."""

    pass


def bootstrap_ci(
    data: np.ndarray,
    n_boot: int = 2000,
    ci: float = 0.95,
    seed: int = 42,
) -> tuple[float, float, float]:
    if len(data) == 0:
        raise ValueError("data must not be empty")

    rng = np.random.default_rng(seed)
    n = len(data)
    boot_means = np.empty(n_boot)

    for i in range(n_boot):
        indices = rng.integers(0, n, size=n)
        boot_sample = data[indices]
        boot_means[i] = np.mean(boot_sample)

    alpha = (1 - ci) / 2
    ci_low = np.percentile(boot_means, 100 * alpha)
    ci_high = np.percentile(boot_means, 100 * (1 - alpha))
    mean = np.mean(data)

    return float(mean), float(ci_low), float(ci_high)


def compute_iqm(returns_per_seed: list[float]) -> float:
    if not returns_per_seed:
        raise ValueError("returns_per_seed must not be empty")

    arr = np.array(returns_per_seed, dtype=float)
    n = len(arr)

    if n < 4:
        return float(np.mean(arr))

    sorted_arr = np.sort(arr)
    trim_count = n // 4

    if trim_count == 0:
        return float(np.mean(arr))

    trimmed = sorted_arr[trim_count : n - trim_count]
    return float(np.mean(trimmed))


def is_superior(ci_low: float, ci_high: float) -> bool:
    return ci_low > 0


def normalize_cross_env(
    results_by_env: dict[str, dict[str, list[float]]],
) -> dict[str, dict[str, list[float]]]:
    normalized = {}

    for env_name, methods in results_by_env.items():
        all_final_returns = []
        for method_returns in methods.values():
            if method_returns:
                all_final_returns.append(method_returns[-1])

        if not all_final_returns:
            normalized[env_name] = {m: [] for m in methods}
            continue

        max_return = max(all_final_returns)
        if max_return == 0:
            max_return = 1.0

        normalized[env_name] = {}
        for method, returns in methods.items():
            normalized[env_name][method] = [r / max_return for r in returns]

    return normalized


def load_eval_histories(run_dir: Path, *, eval_steps: int) -> list[dict[str, Any]]:
    """Scan run dirs for eval_history.csv files.      Returns [{method, env, seed, global_step, return}]; global_step is     ep"""
    histories = []

    for subdir in run_dir.iterdir():
        if not subdir.is_dir():
            continue

        parts = subdir.name.split("_")
        if len(parts) < 3:
            continue

        method = parts[0]
        env = "_".join(parts[1:-1])
        seed_str = parts[-1].removeprefix("seed")
        try:
            seed = int(seed_str)
        except ValueError:
            continue

        csv_path = subdir / "eval_history.csv"
        if not csv_path.exists():
            continue

        with open(csv_path, newline="") as f:
            reader = csv.DictReader(f)
            missing = {"episode", "return"} - set(reader.fieldnames or [])
            if missing:
                raise AnalysisError(f"{csv_path} missing columns: {sorted(missing)}")
            for row in reader:
                histories.append(
                    {
                        "method": method,
                        "env": env,
                        "seed": seed,
                        "global_step": int(row["episode"]) * eval_steps,
                        "return": float(row["return"]),
                    }
                )

    return histories


@dataclass(frozen=True)
class AnalysisReport:
    per_env_curves: dict[str, Path]
    aggregate_png: Path
    summary_csv: Path
    superiority: dict[str, bool]
    excluded_diverged: list[str]
    summary_stats: dict[tuple[str, str], tuple[float, float, float]] = field(
        default_factory=dict  # (method, env) -> (iqm, ci_low, ci_high)
    )


def plot_per_env_curves(
    env_name: str,
    curves_by_method: dict[str, dict[str, Any]],
    output_dir: Path,
) -> Path:
    """Learning curves with bootstrap-CI shading for one env."""
    output_dir.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(10, 6))

    colors = {"sjfpo": "#1f77b4"}

    for method, data in curves_by_method.items():
        eval_points = data["eval_points"]
        returns_per_seed = data["returns_per_seed"]

        if not returns_per_seed:
            continue

        returns_array = np.array(returns_per_seed)
        mean_returns = np.mean(returns_array, axis=0)

        ci_low = np.empty(len(eval_points))
        ci_high = np.empty(len(eval_points))
        for i in range(len(eval_points)):
            point_returns = returns_array[:, i]
            if len(point_returns) >= 2:
                _, low, high = bootstrap_ci(point_returns, n_boot=500, ci=0.95, seed=42)
                ci_low[i] = low
                ci_high[i] = high
            else:
                ci_low[i] = mean_returns[i]
                ci_high[i] = mean_returns[i]

        color = colors.get(method, "#000000")
        ax.plot(
            eval_points, mean_returns, label=method.upper(), color=color, linewidth=2
        )
        ax.fill_between(eval_points, ci_low, ci_high, color=color, alpha=0.2)

    ax.set_xlabel("Global Step")
    ax.set_ylabel("Return")
    ax.set_title(f"Learning Curves: {env_name}")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax.ticklabel_format(style="sci", axis="x", scilimits=(0, 0))

    plt.tight_layout()
    png_path = output_dir / f"{env_name}_curves.png"
    plt.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    return png_path


def plot_aggregate(
    aggregate_data: dict[str, dict[str, list[float]]],
    output_dir: Path,
) -> Path:
    """Normalized cross-env bar chart with bootstrap-CI error bars."""
    output_dir.mkdir(parents=True, exist_ok=True)

    methods = set()
    for env_data in aggregate_data.values():
        methods.update(env_data.keys())
    methods = sorted(methods)

    method_means = {}
    method_cis = {}

    for method in methods:
        all_returns = []
        for env_data in aggregate_data.values():
            if method in env_data:
                all_returns.extend(env_data[method])

        if all_returns:
            mean, ci_low, ci_high = bootstrap_ci(
                np.array(all_returns), n_boot=2000, ci=0.95, seed=42
            )
            method_means[method] = mean
            method_cis[method] = (ci_low, ci_high)
        else:
            method_means[method] = 0.0
            method_cis[method] = (0.0, 0.0)

    fig, ax = plt.subplots(figsize=(8, 6))

    colors = {"sjfpo": "#1f77b4"}
    x_pos = np.arange(len(methods))
    means = [method_means[m] for m in methods]
    ci_lows = [method_cis[m][0] for m in methods]
    ci_highs = [method_cis[m][1] for m in methods]

    bars = ax.bar(
        x_pos,
        means,
        yerr=[
            np.array(means) - np.array(ci_lows),
            np.array(ci_highs) - np.array(means),
        ],
        capsize=5,
        color=[colors.get(m, "#000000") for m in methods],
        edgecolor="black",
        linewidth=0.5,
    )

    ax.set_xticks(x_pos)
    ax.set_xticklabels([m.upper() for m in methods])
    ax.set_ylabel("Normalized Return")
    ax.set_title("Aggregate Cross-Environment Comparison")
    ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    png_path = output_dir / "aggregate_comparison.png"
    plt.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    return png_path


def write_summary_csv(report: AnalysisReport, output_dir: Path) -> None:
    """Summary CSV (method,env,iqm,ci_low,ci_high,status) from report cells."""
    output_dir.mkdir(parents=True, exist_ok=True)
    csv_path = output_dir / "results_summary.csv"

    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["method", "env", "iqm", "ci_low", "ci_high", "status"])
        for (method, env), (iqm, ci_low, ci_high) in sorted(
            report.summary_stats.items()
        ):
            status = "complete"
            if any(
                d.startswith(f"{method}_{env}_seed") for d in report.excluded_diverged
            ):
                status = "diverged"
            writer.writerow([method, env, iqm, ci_low, ci_high, status])


def analyze_results(
    matrix: tuple[RunSpec, ...],
    base_dir: Path,
    output_dir: Path,
    *,
    eval_steps: int,
) -> AnalysisReport:
    """Run the full analysis: histories → curves → aggregate → plots → CSV."""
    output_dir.mkdir(parents=True, exist_ok=True)

    histories = load_eval_histories(base_dir, eval_steps=eval_steps)

    if not histories:
        raise AnalysisError("no eval histories")

    from collections import defaultdict

    grouped = defaultdict(lambda: defaultdict(list))

    for h in histories:
        key = (h["method"], h["env"], h["seed"])
        grouped[key]["returns"].append(h["return"])
        grouped[key]["eval_points"].append(h["global_step"])

    excluded_diverged = []
    for subdir in base_dir.iterdir():
        if not subdir.is_dir():
            continue
        manifest_path = subdir / "diverged.json"
        if manifest_path.exists():
            with open(manifest_path) as f:
                manifest = json.load(f)
            if manifest.get("diverged", False):
                excluded_diverged.append(subdir.name)

    per_env_curves = {}
    curves_by_env_method = defaultdict(lambda: defaultdict(list))

    for (method, env, seed), data in grouped.items():
        if f"{method}_{env}_{seed}" in excluded_diverged:
            continue
        curves_by_env_method[env][method].append(
            {
                "seed": seed,
                "eval_points": data["eval_points"],
                "returns": data["returns"],
            }
        )

    for env, methods in curves_by_env_method.items():
        curves_by_method = {}
        for method, seeds_data in methods.items():
            if seeds_data:
                curves_by_method[method] = {
                    "eval_points": seeds_data[0]["eval_points"],
                    "returns_per_seed": [d["returns"] for d in seeds_data],
                }
        if curves_by_method:
            png_path = plot_per_env_curves(env, curves_by_method, output_dir)
            per_env_curves[env] = png_path

    final_returns_by_env = defaultdict(lambda: defaultdict(list))
    for (method, env, seed), data in grouped.items():
        if f"{method}_{env}_{seed}" in excluded_diverged:
            continue
        if data["returns"]:
            final_returns_by_env[env][method].append(data["returns"][-1])

    summary_stats: dict[tuple[str, str], tuple[float, float, float]] = {}
    for env, methods in final_returns_by_env.items():
        for method, returns in methods.items():
            _, ci_low, ci_high = bootstrap_ci(
                np.array(returns, dtype=float), n_boot=2000, ci=0.95, seed=42
            )
            summary_stats[(method, env)] = (compute_iqm(returns), ci_low, ci_high)

    normalized = normalize_cross_env(final_returns_by_env)

    aggregate_png = plot_aggregate(normalized, output_dir)

    superiority = {}
    for method in ["sjfpo"]:
        baseline_methods = [m for m in ["sjfpo"] if m != method]
        if not baseline_methods:
            superiority[method] = False
            continue

        method_returns = []
        for env_data in final_returns_by_env.values():
            if method in env_data:
                method_returns.extend(env_data[method])

        if not method_returns:
            superiority[method] = False
            continue

        best_baseline_returns = []
        for baseline in baseline_methods:
            baseline_returns = []
            for env_data in final_returns_by_env.values():
                if baseline in env_data:
                    baseline_returns.extend(env_data[baseline])
            if baseline_returns:
                best_baseline_returns.append((np.mean(baseline_returns), baseline))

        if not best_baseline_returns:
            superiority[method] = False
            continue

        best_baseline_returns.sort(reverse=True)
        best_baseline_method = best_baseline_returns[0][1]

        baseline_returns = []
        for env_data in final_returns_by_env.values():
            if best_baseline_method in env_data:
                baseline_returns.extend(env_data[best_baseline_method])

        if not baseline_returns:
            superiority[method] = False
            continue

        min_len = min(len(method_returns), len(baseline_returns))
        if min_len < 2:
            superiority[method] = False
            continue

        diff = np.array(method_returns[:min_len]) - np.array(baseline_returns[:min_len])
        _, ci_low, ci_high = bootstrap_ci(diff, n_boot=2000, ci=0.95, seed=42)
        superiority[method] = is_superior(ci_low, ci_high)

    summary_csv = output_dir / "results_summary.csv"
    write_summary_csv(
        AnalysisReport(
            per_env_curves=per_env_curves,
            aggregate_png=aggregate_png,
            summary_csv=summary_csv,
            superiority=superiority,
            excluded_diverged=excluded_diverged,
        ),
        output_dir,
    )

    return AnalysisReport(
        per_env_curves=per_env_curves,
        aggregate_png=aggregate_png,
        summary_csv=summary_csv,
        superiority=superiority,
        excluded_diverged=excluded_diverged,
        summary_stats=summary_stats,
    )

In [ ]:
%%writefile src/sjfpo/checkpointing.py
"""Policy checkpoint IO (HuggingFace style): save_pretrained/from_pretrained.

Layout of a saved checkpoint directory:

* ``config.json``                — LeRobot ``PreTrainedConfig`` (unwrapped keys)
* ``model.safetensors``          — policy weights (unwrapped keys)
* ``optimizer.pt``               — AdamW state_dict (pickled, weights-only)
* ``counters.json``              — ``{global_step, step_count}``
* ``rng.pt``                     — torch / cuda RNG state (optional, when
                                    ``save_rng=True``)

Optimizer, counters, and RNG files use temporary files followed by atomic
replacement. Policy model/config serialization follows the LeRobot/Hugging
Face ``save_pretrained`` path and is not claimed to be atomic as a complete
multi-file checkpoint generation.
"""

from __future__ import annotations

import contextlib
import json
import os
import tempfile
from pathlib import Path

import torch
from huggingface_hub.constants import CONFIG_NAME, SAFETENSORS_SINGLE_FILE
from lerobot.utils.constants import CHECKPOINTS_DIR
from sjfpo.modeling_sjfpo import SJFPOPolicy  # lazy: import-light module

OPTIMIZER_FILE = "optimizer.pt"
COUNTERS_FILE = "counters.json"
RNG_FILE = "rng.pt"


class ResumeError(RuntimeError):
    """Unresumable checkpoint (missing/corrupt files)."""


# ---------------------------------------------------------------------------
# torch.compile unwrapping
# ---------------------------------------------------------------------------


def _iter_compiled_children(
    policy,
) -> list[tuple[str, torch.nn.Module, torch.nn.Module]]:
    """Return ``(name, compiled_module, inner_module)`` for every top-level
    submodule of ``policy`` wrapped by ``torch.compile``.

    Only top-level children are inspected; ``torch.compile`` is applied
    per-module via ``maybe_compile`` and never nested in this codebase.
    """
    out = []
    for name, child in policy.named_children():
        inner = getattr(child, "_orig_mod", None)
        if isinstance(inner, torch.nn.Module):
            out.append((name, child, inner))
    return out


@contextlib.contextmanager
def unwrapped_policy(policy):
    """Expose a compiled policy's eager submodules for checkpoint IO.

    Swaps every ``torch.compile`` wrapper (``X._orig_mod``) for its inner
    module so ``state_dict()`` emits unwrapped keys. Restores the wrappers on
    exit, even on exception.
    """
    swapped = _iter_compiled_children(policy)
    for name, _compiled, inner in swapped:
        setattr(policy, name, inner)
    try:
        yield policy
    finally:
        for name, compiled, _inner in swapped:
            setattr(policy, name, compiled)


# ---------------------------------------------------------------------------
# Atomic write helpers
# ---------------------------------------------------------------------------


def _atomic_write_bytes(path: Path, data: bytes) -> None:
    with tempfile.NamedTemporaryFile(
        mode="wb", dir=path.parent, prefix=f".{path.name}.", delete=False
    ) as tmp:
        tmp.write(data)
        tmp.flush()
        os.fsync(tmp.fileno())
        tmp_name = tmp.name
    os.replace(tmp_name, path)


def _atomic_torch_save(path: Path, obj) -> None:
    with tempfile.NamedTemporaryFile(
        mode="wb", dir=path.parent, prefix=f".{path.name}.", delete=False
    ) as tmp:
        tmp_name = tmp.name
    try:
        torch.save(obj, tmp_name)
        os.replace(tmp_name, path)
    except Exception:
        with contextlib.suppress(FileNotFoundError):
            os.unlink(tmp_name)
        raise


# ---------------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------------


def save_policy_checkpoint(
    dir_path: str | os.PathLike,
    *,
    policy,
    optimizer: torch.optim.Optimizer | None = None,
    actor_optimizer: torch.optim.Optimizer | None = None,
    critic_optimizer: torch.optim.Optimizer | None = None,
    global_step: int,
    ema_state: dict | None = None,
    save_rng: bool = True,
) -> None:
    """Save through LeRobot's ``HubMixin`` (+ optimizer.pt, counters.json).

    ``model.safetensors`` holds **unwrapped** eager keys (torch.compile
    wrappers are swapped out for the duration of ``save_pretrained``), so a
    fresh, uncompiled ``SJFPOFlowModel`` in :func:`load_policy_checkpoint`
    finds every key it expects.
    """
    from sjfpo.modeling_sjfpo import SJFPOPolicy  # lazy: import-light module

    if not isinstance(policy, SJFPOPolicy):
        raise ValueError(
            f"save_policy_checkpoint needs an SJFOPolicy, got {type(policy).__name__}"
        )
    if int(global_step) < 0:
        raise ValueError(f"global_step={global_step} must be >= 0")
    if (actor_optimizer is None) != (critic_optimizer is None):
        raise ValueError("actor_optimizer and critic_optimizer must be provided together")
    if actor_optimizer is None and optimizer is None:
        raise ValueError("provide actor/critic optimizers or legacy optimizer")

    d = Path(dir_path)
    d.mkdir(parents=True, exist_ok=True)

    # 1) Weights (through LeRobot; unwrapped so keys match a fresh policy).
    # The frozen behavior snapshots are runtime-only and are refreshed before
    # the next rollout, so do not persist their duplicate parameters.
    with unwrapped_policy(policy):
        state_dict = {
            name: tensor
            for name, tensor in policy.state_dict().items()
            if not name.startswith(("old_actor.", "old_conditioner."))
        }
        if ema_state is not None and policy.ema_ready(global_step):
            for name, shadow in ema_state["shadow_params"].items():
                full_name = f"actor.{name}"
                if full_name in state_dict:
                    state_dict[full_name] = shadow.detach().clone()
        policy.save_pretrained(d, state_dict=state_dict)
    if ema_state is not None:
        _atomic_torch_save(d / "ema.pt", ema_state)

    # 2) Optimizer state (pickled, weights-only safe on load).
    if actor_optimizer is not None:
        optimizer_state = {
            "format": 2,
            "actor": actor_optimizer.state_dict(),
            "critic": critic_optimizer.state_dict(),
        }
    else:
        optimizer_state = optimizer.state_dict()
    _atomic_torch_save(d / OPTIMIZER_FILE, optimizer_state)

    # 3) Counters (atomic JSON).
    counters = {
        "global_step": int(global_step),
        "step_count": int(getattr(policy, "step_count", 0)),
    }
    _atomic_write_bytes(
        d / COUNTERS_FILE,
        json.dumps(counters, indent=2).encode("utf-8"),
    )

    # 4) RNG state (torch + cuda if available) — enables bit-reproducible resume.
    if save_rng:
        rng = {"torch": torch.get_rng_state()}
        if torch.cuda.is_available():
            rng["cuda"] = torch.cuda.get_rng_state_all()
        _atomic_torch_save(d / RNG_FILE, rng)


# ---------------------------------------------------------------------------
# Load
# ---------------------------------------------------------------------------


def _load_optimizer(policy, d: Path) -> torch.optim.Optimizer:
    """Rebuild the optimizer with the current config and load saved state.

    Fails loudly on any structural mismatch (wrong number of param groups,
    wrong key set) rather than relying on ``load_state_dict``'s opaque
    error messages.
    """
    groups = policy.get_optim_params()["params"]
    optimizer = policy.config.get_optimizer_preset().build(groups)

    try:
        opt_state = torch.load(
            d / OPTIMIZER_FILE, map_location="cpu", weights_only=True
        )
    except Exception as e:
        raise ResumeError(
            f"failed to unpickle {d / OPTIMIZER_FILE}: {e}. "
            "If the optimizer was subclassed with non-tensor state, re-save "
            "with weights_only=False at your own risk."
        ) from e

    saved_groups = opt_state.get("param_groups", [])
    if len(saved_groups) != len(optimizer.param_groups):
        raise ResumeError(
            f"optimizer param_groups mismatch: checkpoint has "
            f"{len(saved_groups)}, current config expects "
            f"{len(optimizer.param_groups)}. The config used to save differs "
            "from the current one — cannot resume safely."
        )
    try:
        optimizer.load_state_dict(opt_state)
    except Exception as e:
        raise ResumeError(f"optimizer.load_state_dict failed: {e}") from e
    return optimizer


def _load_optimizer_pair(policy, d: Path):
    try:
        payload = torch.load(
            d / OPTIMIZER_FILE, map_location="cpu", weights_only=True
        )
    except Exception as e:
        raise ResumeError(
            f"failed to unpickle {d / OPTIMIZER_FILE}: {e}. "
            "If the optimizer was subclassed with non-tensor state, re-save "
            "with weights_only=False at your own risk."
        ) from e

    if not isinstance(payload, dict) or payload.get("format") != 2:
        return _load_optimizer(policy, d), None

    try:
        actor_state = payload["actor"]
        critic_state = payload["critic"]
    except (KeyError, TypeError) as e:
        raise ResumeError(f"malformed split optimizer payload in {d}: {e}") from e

    actor = policy.config.get_optimizer_preset(lr=policy.config.policy_lr).build(
        policy.get_actor_optim_params()["params"]
    )
    critic = policy.config.get_optimizer_preset(lr=policy.config.critic_lr).build(
        policy.get_critic_optim_params()["params"]
    )
    try:
        actor.load_state_dict(actor_state)
        critic.load_state_dict(critic_state)
    except Exception as e:
        raise ResumeError(f"split optimizer load_state_dict failed: {e}") from e
    return actor, critic


def _load_rng(d: Path) -> None:
    """Restore torch/cuda RNG state if a saved RNG file exists. Best effort."""
    path = d / RNG_FILE
    if not path.is_file():
        return
    try:
        rng = torch.load(path, map_location="cpu", weights_only=True)
    except Exception:
        return  # corrupt RNG file shouldn't block resume
    if "torch" in rng:
        torch.set_rng_state(rng["torch"])
    if "cuda" in rng and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(rng["cuda"])


def load_policy_checkpoint(
    dir_path: str | os.PathLike,
    *,
    device: str | torch.device | None = None,
    restore_rng: bool = True,
    action_low=None,
    action_high=None,
):
    """Load a checkpoint written by :func:`save_policy_checkpoint`.

    Rebuilds every component from ``config.json`` — no caller-supplied dims.
    Environment action bounds are runtime state; pass both ``action_low`` and
    ``action_high`` to restore the current postprocessor mapping.
    New split checkpoints return ``(policy, actor_optimizer, critic_optimizer,
    counters)``; legacy checkpoints return ``(policy, optimizer, counters)``.
    """
    from sjfpo.modeling_sjfpo import SJFPOPolicy  # lazy: import-light module

    d = Path(dir_path)
    missing = [
        f
        for f in (CONFIG_NAME, SAFETENSORS_SINGLE_FILE, OPTIMIZER_FILE, COUNTERS_FILE)
        if not (d / f).is_file()
    ]
    if missing:
        raise ResumeError(f"incomplete policy checkpoint in {d}: missing {missing}")

    # 1) Policy (fresh, uncompiled — state_dict keys are unwrapped by save).
    policy = SJFPOPolicy.from_pretrained(d)
    if device is not None:
        policy.to(device)
    # Frozen snapshots are intentionally omitted from model.safetensors;
    # reconstruct them from the loaded live modules before resuming.
    policy.snapshot_behavior_policy()
    if (action_low is None) != (action_high is None):
        raise ValueError("action_low and action_high must be provided together")
    if action_low is not None:
        policy.set_action_bounds(action_low, action_high)

    # 2) Optimizer state (split format for new checkpoints; legacy stays supported).
    actor_optimizer, critic_optimizer = _load_optimizer_pair(policy, d)

    ema_path = d / "ema.pt"
    if ema_path.is_file():
        policy.load_ema_state_dict(
            torch.load(ema_path, map_location="cpu", weights_only=True)
        )

    # 3) Counters (validated).
    try:
        counters_raw = json.loads((d / COUNTERS_FILE).read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError) as e:
        raise ResumeError(f"corrupt {d / COUNTERS_FILE}: {e}") from e
    try:
        global_step = int(counters_raw["global_step"])
        step_count = int(counters_raw["step_count"])
    except (KeyError, TypeError, ValueError) as e:
        raise ResumeError(f"malformed counters in {d}: {e}") from e
    if global_step < 0 or step_count < 0:
        raise ResumeError(
            f"negative counters in {d}: global_step={global_step}, "
            f"step_count={step_count}"
        )
    policy.step_count = step_count

    # 4) RNG state (best-effort).
    if restore_rng:
        _load_rng(d)

    counters = {"global_step": global_step, "step_count": step_count}
    if critic_optimizer is None:
        return policy, actor_optimizer, counters
    return policy, actor_optimizer, critic_optimizer, counters

In [ ]:
%%writefile src/sjfpo/collector.py
"""Chunked open-loop rollout collector.

Performance notes
-----------------
* Actions are transferred host-side **once per step** (batched over envs) rather
  than once per env per step. The per-step `.numpy()` sync is the single
  biggest cost in the vec collector.
* Observation windows and rewards stay in **fp32** end-to-end; policy data
  follows the execution-device dtype policy. bf16 rewards corrupt the GAE
  target.
* Bounds are validated once per collector call and reduced to
  ``(low, range)``; ``scale_and_clip_chunk`` no longer re-validates.
* Warmup windows share one clone (deque holds N references to the same
  tensor) instead of cloning ``n_obs_steps`` copies; safe because the window
  is only read, never mutated in place.
* Tail truncation writes into the existing reward/done tensors rather than
  constructing a fresh scalar per tail.
"""

from __future__ import annotations

from collections import deque
from collections.abc import Callable
from typing import TypedDict

import numpy as np
import torch

from lerobot.processor.converters import to_tensor
from lerobot.utils.constants import DONE, OBS_STATE, REWARD, TRUNCATED

from sjfpo.configuration_sjfpo import SJFPOConfig

__all__ = ["Transition", "collect_chunks", "collect_chunks_vec"]


# Lazily-constructed config (import cost only on first use).
_DEFAULT_CFG: SJFPOConfig | None = None


def _defaults() -> SJFPOConfig:
    global _DEFAULT_CFG
    if _DEFAULT_CFG is None:
        _DEFAULT_CFG = SJFPOConfig()
    return _DEFAULT_CFG


# Dtypes: observations and rewards are fp32 end-to-end. Only policy outputs
# and policy data use the execution-device dtype policy.
_OBS_DTYPE = torch.float32
_REWARD_DTYPE = torch.float32
_INDEX_DTYPE = torch.int64


Transition = TypedDict(
    "Transition",
    {
        "observation.state": torch.Tensor,  # [T, obs] fp32
        "action": torch.Tensor,  # [H, action_dim] scaled
        "next.observation.state": torch.Tensor,  # [T, obs] fp32
        "next.reward": torch.Tensor,  # [n_exec] fp32
        "next.done": torch.Tensor,  # bool scalar
        "next.truncated": torch.Tensor,  # bool scalar
        "executed_n_action_steps": torch.Tensor,  # int64 scalar
        "clip_count": torch.Tensor,  # int64 scalar
    },
)


class _StepTransition(dict):
    """Transition dict with post-step windows stored outside public keys."""

    def __init__(self, record: Transition, step_next_states: torch.Tensor):
        super().__init__(record)
        # Keep the public eight-key Transition contract while retaining the
        # state window after every executed action for step-level GAE.
        self.step_next_states = step_next_states


class RolloutBuffer:
    """Fixed-capacity CPU storage for chunk and executed-step rollout data.

    The active training path reuses one instance per capacity/device. Chunks
    are written by the collector; ``SJFPOPolicy.generate_rollout`` reads views
    of the used prefix and transfers those contiguous tensors to the device.
    """

    def __init__(
        self,
        *,
        max_chunks: int,
        max_steps: int,
        obs_steps: int,
        obs_dim: int,
        action_horizon: int,
        action_dim: int,
        device: torch.device | str = "cpu",
        pin_memory: bool = False,
    ):
        if max_chunks < 1 or max_steps < 1:
            raise ValueError("max_chunks and max_steps must be positive")
        if obs_steps < 1 or obs_dim < 1 or action_horizon < 1 or action_dim < 1:
            raise ValueError("observation and action dimensions must be positive")
        self.device = torch.device(device)
        self.max_chunks = max_chunks
        self.max_steps = max_steps
        self.obs_steps = obs_steps
        self.obs_dim = obs_dim
        self.action_horizon = action_horizon
        self.action_dim = action_dim
        self.pin_memory = pin_memory

        def zeros(shape, dtype):
            return torch.zeros(
                *shape, dtype=dtype, device=self.device, pin_memory=self.pin_memory
            )

        self.step_states = zeros((max_steps, obs_steps, obs_dim), _OBS_DTYPE)
        self.step_next_states = zeros(
            (max_steps, obs_steps, obs_dim), _OBS_DTYPE
        )
        self.step_rewards = zeros((max_steps,), _REWARD_DTYPE)
        self.step_terminated = zeros((max_steps,), torch.bool)
        self.step_truncated = zeros((max_steps,), torch.bool)

        self.chunk_actions = zeros(
            (max_chunks, action_horizon, action_dim), _OBS_DTYPE
        )
        self.chunk_starts = zeros((max_chunks,), _INDEX_DTYPE)
        self.chunk_lengths = zeros((max_chunks,), _INDEX_DTYPE)
        self.chunk_done = zeros((max_chunks,), torch.bool)
        self.chunk_truncated = zeros((max_chunks,), torch.bool)
        self.chunk_clip_counts = zeros((max_chunks,), _INDEX_DTYPE)
        self.n_chunks = 0
        self.n_steps = 0

    def capacity_key(self) -> tuple[int, ...]:
        return (
            self.max_chunks,
            self.max_steps,
            self.obs_steps,
            self.obs_dim,
            self.action_horizon,
            self.action_dim,
            self.pin_memory,
        )

    def reset(self) -> None:
        """Reset cursors without clearing the large storage tensors."""
        self.n_chunks = 0
        self.n_steps = 0

    def add_chunk(
        self,
        *,
        state: torch.Tensor,
        action: torch.Tensor,
        rewards: list[float] | np.ndarray | torch.Tensor,
        step_next_states: torch.Tensor,
        done: bool,
        truncated: bool,
        clip_count: int,
    ) -> int:
        if self.n_chunks >= self.max_chunks:
            raise RuntimeError("RolloutBuffer chunk capacity exceeded")
        reward_t = torch.as_tensor(rewards, dtype=_REWARD_DTYPE, device=self.device).reshape(-1)
        n_exec = int(reward_t.numel())
        if n_exec < 1 or self.n_steps + n_exec > self.max_steps:
            raise RuntimeError("RolloutBuffer step capacity exceeded")
        if state.shape != (self.obs_steps, self.obs_dim):
            raise ValueError(f"state shape {tuple(state.shape)} does not fit buffer")
        if action.shape != (self.action_horizon, self.action_dim):
            raise ValueError(f"action shape {tuple(action.shape)} does not fit buffer")
        if tuple(step_next_states.shape) != (n_exec, self.obs_steps, self.obs_dim):
            raise ValueError(
                f"step_next_states shape {tuple(step_next_states.shape)} does not fit buffer"
            )

        chunk_index = self.n_chunks
        offset = self.n_steps
        self.step_states[offset].copy_(state)
        if n_exec > 1:
            self.step_states[offset + 1 : offset + n_exec].copy_(step_next_states[:-1])
        self.step_next_states[offset : offset + n_exec].copy_(step_next_states)
        self.step_rewards[offset : offset + n_exec].copy_(reward_t)
        self.chunk_actions[chunk_index].copy_(action)
        self.chunk_starts[chunk_index] = offset
        self.chunk_lengths[chunk_index] = n_exec
        self.chunk_done[chunk_index] = done
        self.chunk_truncated[chunk_index] = truncated
        self.chunk_clip_counts[chunk_index] = int(clip_count)
        self.step_terminated[offset + n_exec - 1] = done
        self.step_truncated[offset + n_exec - 1] = truncated
        self.n_chunks += 1
        self.n_steps += n_exec
        return chunk_index

    def copy_to(self, other: "RolloutBuffer", *, non_blocking: bool = False) -> None:
        """Copy the active prefix into another buffer on another device."""
        if other.capacity_key()[:6] != self.capacity_key()[:6]:
            raise ValueError("RolloutBuffer capacities do not match")
        other.n_chunks = self.n_chunks
        other.n_steps = self.n_steps
        for name in (
            "step_states",
            "step_next_states",
            "step_rewards",
            "step_terminated",
            "step_truncated",
            "chunk_actions",
            "chunk_starts",
            "chunk_lengths",
            "chunk_done",
            "chunk_truncated",
        ):
            getattr(other, name)[: self.n_steps if name.startswith("step_") else self.n_chunks].copy_(
                getattr(self, name)[: self.n_steps if name.startswith("step_") else self.n_chunks],
                non_blocking=non_blocking,
            )

    def mark_truncated(self, chunk_index: int) -> None:
        if not 0 <= chunk_index < self.n_chunks:
            raise IndexError(f"chunk index {chunk_index} out of range")
        self.chunk_truncated[chunk_index] = True
        start = int(self.chunk_starts[chunk_index])
        length = int(self.chunk_lengths[chunk_index])
        self.step_truncated[start + length - 1] = True

    def active(self) -> dict[str, torch.Tensor]:
        """Return contiguous views of the populated buffer prefix."""
        n_chunks, n_steps = self.n_chunks, self.n_steps
        return {
            "step_states": self.step_states[:n_steps],
            "step_next_states": self.step_next_states[:n_steps],
            "step_rewards": self.step_rewards[:n_steps],
            "step_terminated": self.step_terminated[:n_steps],
            "step_truncated": self.step_truncated[:n_steps],
            "chunk_actions": self.chunk_actions[:n_chunks],
            "chunk_starts": self.chunk_starts[:n_chunks],
            "chunk_lengths": self.chunk_lengths[:n_chunks],
            "chunk_done": self.chunk_done[:n_chunks],
            "chunk_truncated": self.chunk_truncated[:n_chunks],
        }


# ---------------------------------------------------------------------------
# Conversions
# ---------------------------------------------------------------------------


def as_float_tensor(x: object) -> torch.Tensor:
    """Coerce to an fp32 CPU/GPU tensor without forcing a device hop.

    A ``torch.Tensor`` input keeps its device (no implicit ``.cpu()``);
    numpy / Python inputs are lifted to a fresh CPU fp32 tensor.
    """
    if isinstance(x, torch.Tensor):
        return x.detach().to(_OBS_DTYPE)
    return torch.as_tensor(np.asarray(x, dtype=np.float32), dtype=_OBS_DTYPE)


def _bounds_to_tensor(
    low: float | np.ndarray | torch.Tensor,
    high: float | np.ndarray | torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Coerce, validate, and return ``(low, range = high - low)`` on fp32."""
    lo = torch.as_tensor(
        np.asarray(low, dtype=np.float32), dtype=torch.float32
    ).reshape(-1)
    hi = torch.as_tensor(
        np.asarray(high, dtype=np.float32), dtype=torch.float32
    ).reshape(-1)
    if lo.shape != hi.shape:
        raise ValueError(
            f"action_low shape {tuple(lo.shape)} != action_high shape {tuple(hi.shape)}"
        )
    rng = hi - lo
    if not bool((rng > 0).all()):
        raise ValueError("action bounds must satisfy high > low in every dim")
    return lo, rng


def scale_and_clip_chunk(
    proposed: torch.Tensor, low: torch.Tensor, range_: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor, int]:
    """Clamp a proposal to bounds and rescale to [-1, 1]; count clipped elements.

    ``range_`` is the precomputed ``high - low`` (see :func:`_bounds_to_tensor`).
    """
    clip_mask = (proposed < low) | (proposed > low + range_)
    clip_count = int(clip_mask.sum())
    executed = torch.clamp(proposed, min=low, max=low + range_)
    scaled = 2.0 * (executed - low) / range_ - 1.0
    return executed, scaled, clip_count


# ---------------------------------------------------------------------------
# Window helpers
# ---------------------------------------------------------------------------


def build_warmup_window(
    first_obs: torch.Tensor, n_obs_steps: int
) -> deque[torch.Tensor]:
    """Deque of ``n_obs_steps`` identical frames.

    Uses a single clone referenced N times — safe because the window is only
    read, never mutated in place. Appends past the initial fill replace the
    oldest reference, not the shared tensor.
    """
    shared = first_obs.detach().clone()
    return deque((shared for _ in range(n_obs_steps)), maxlen=n_obs_steps)


def reset_chunk_window(env, n_obs_steps: int) -> deque[torch.Tensor]:
    obs, _ = env.reset()
    return build_warmup_window(as_float_tensor(obs), n_obs_steps)


# ---------------------------------------------------------------------------
# Transition builder
# ---------------------------------------------------------------------------


def _bool_tensor(v: bool) -> torch.Tensor:
    return torch.tensor(v, dtype=torch.bool)


def _int_tensor(v: int) -> torch.Tensor:
    return torch.tensor(v, dtype=_INDEX_DTYPE)


def finalize_transition(
    state: torch.Tensor,
    next_state: torch.Tensor,
    scaled: torch.Tensor,
    rewards: list[float] | np.ndarray,
    done_kind: str,
    clip_count: int,
    chunk_size: int,
    n_action_steps: int,
    step_next_states: torch.Tensor,
) -> Transition:
    if scaled.shape[0] != chunk_size:
        raise ValueError(
            f"proposed chunk has {scaled.shape[0]} rows, "
            f"expected chunk_size={chunk_size}"
        )
    if not 1 <= n_action_steps <= chunk_size:
        raise ValueError(
            f"n_action_steps={n_action_steps} must satisfy "
            f"1 <= n <= chunk_size={chunk_size}"
        )
    # Rewards land in fp32 — GAE targets must not be bf16-truncated.
    reward_t = torch.as_tensor(
        np.asarray(rewards, dtype=np.float32), dtype=_REWARD_DTYPE
    )
    n_executed = int(reward_t.shape[0])
    if step_next_states.shape != (n_executed, *next_state.shape):
        raise ValueError(
            f"step_next_states shape {tuple(step_next_states.shape)} != "
            f"{(n_executed, *next_state.shape)}"
        )
    record: Transition = {
        OBS_STATE: state,
        "action": scaled,
        "next.observation.state": next_state,
        REWARD: reward_t,
        DONE: _bool_tensor(done_kind == "terminated"),
        TRUNCATED: _bool_tensor(done_kind == "truncated"),
        "executed_n_action_steps": _int_tensor(n_executed),
        "clip_count": _int_tensor(clip_count),
    }
    return _StepTransition(record, step_next_states)  # type: ignore[return-value]


# ---------------------------------------------------------------------------
# Single-env collector
# ---------------------------------------------------------------------------


def execute_open_loop_steps(
    env,
    executed_np: np.ndarray,  # [H, act_dim] on host — no per-step sync
    window: deque[torch.Tensor],
    executed_total: int,
    total_steps: int,
    n_action_steps: int,
) -> tuple[list[float], str, torch.Tensor]:
    rewards: list[float] = []
    step_next_states: list[torch.Tensor] = []
    done_kind = "continue"
    for i in range(n_action_steps):
        if executed_total >= total_steps:
            break
        obs, reward, terminated, truncated, _info = env.step(executed_np[i])
        executed_total += 1
        rewards.append(float(reward))
        window.append(as_float_tensor(obs))
        step_next_states.append(torch.stack(tuple(window), dim=0))
        if terminated:
            done_kind = "terminated"
            break
        if truncated:
            done_kind = "truncated"
            break
    return rewards, done_kind, torch.stack(step_next_states, dim=0)


def execute_open_loop_steps_into(
    env,
    executed_np: np.ndarray,
    window: torch.Tensor,
    step_windows: torch.Tensor,
    rewards: torch.Tensor,
    executed_total: int,
    total_steps: int,
    n_action_steps: int,
) -> tuple[int, str]:
    """Execute one single-env chunk into preallocated window/reward storage."""
    n_exec = 0
    done_kind = "continue"
    for i in range(n_action_steps):
        if executed_total + n_exec >= total_steps:
            break
        obs, reward, terminated, truncated, _info = env.step(executed_np[i])
        n_exec += 1
        rewards[n_exec - 1] = reward
        window[:-1].copy_(window[1:].clone())
        window[-1].copy_(torch.as_tensor(np.asarray(obs, dtype=np.float32)))
        step_windows[n_exec - 1].copy_(window)
        if terminated:
            done_kind = "terminated"
            break
        if truncated:
            done_kind = "truncated"
            break
    return n_exec, done_kind


def propose_and_execute_chunk(
    policy: Callable[[torch.Tensor], torch.Tensor],
    window: deque[torch.Tensor],
    low: torch.Tensor,
    range_: torch.Tensor,
    env,
    executed_total: int,
    total_steps: int,
    chunk_size: int,
    n_action_steps: int,
) -> tuple[Transition, str]:
    state = torch.stack(tuple(window), dim=0)
    proposed = as_float_tensor(policy(state))
    executed, scaled, clip_count = scale_and_clip_chunk(proposed, low, range_)
    # Single host transfer for the whole chunk, then index on CPU.
    executed_np = executed.detach().cpu().numpy()
    rewards, done_kind, step_next_states = execute_open_loop_steps(
        env, executed_np, window, executed_total, total_steps, n_action_steps
    )
    next_state = torch.stack(tuple(window), dim=0)
    record = finalize_transition(
        state,
        next_state,
        scaled,
        rewards,
        done_kind,
        clip_count,
        chunk_size,
        n_action_steps,
        step_next_states,
    )
    return record, done_kind


def flush_on_collection_error(
    on_error_flush: Callable[[], None] | None,
) -> None:
    if on_error_flush is not None:
        try:
            on_error_flush()
        except Exception:
            pass


# ---------------------------------------------------------------------------
# Vector collector
# ---------------------------------------------------------------------------


def collect_chunks_vec(
    envs,
    policy: Callable[[torch.Tensor], torch.Tensor],
    total_steps: int,
    action_low: float | np.ndarray,
    action_high: float | np.ndarray,
    *,
    n_obs_steps: int | None = None,
    n_action_steps: int,
    chunk_size: int | None = None,
    seed: int | None = None,
    on_error_flush: Callable[[], None] | None = None,
    buffer: RolloutBuffer | None = None,
) -> list[Transition] | RolloutBuffer:
    """Vector twin of :func:`collect_chunks` — same records, same chaining."""
    cfg = _defaults()
    n_obs_steps = cfg.n_obs_steps if n_obs_steps is None else n_obs_steps
    chunk_size = cfg.action_horizon if chunk_size is None else int(chunk_size)

    n = int(envs.num_envs)
    base_target, remainder = divmod(total_steps, n)
    targets = [base_target + int(i < remainder) for i in range(n)]

    low, range_ = _bounds_to_tensor(action_low, action_high)

    if not 1 <= n_action_steps <= chunk_size:
        raise ValueError(
            f"n_action_steps={n_action_steps} must satisfy "
            f"1 <= n <= chunk_size={chunk_size}"
        )
    if seed is None:
        seed = int(torch.randint(0, 2**31, (1,)).item())
    act_dim = int(np.asarray(envs.single_action_space.shape).prod())

    try:
        obs, _ = envs.reset(seed=[seed + i for i in range(n)])
        if buffer is not None:
            obs_dim = int(obs.shape[-1])
            windows = torch.zeros(n, n_obs_steps, obs_dim, dtype=_OBS_DTYPE)
            for i in range(n):
                windows[i].copy_(as_float_tensor(obs[i]).expand_as(windows[i]))
            reward_windows = torch.zeros(n, chunk_size, dtype=_REWARD_DTYPE)
            step_windows = torch.zeros(n, chunk_size, n_obs_steps, obs_dim, dtype=_OBS_DTYPE)
        else:
            windows = [build_warmup_window(as_float_tensor(o), n_obs_steps) for o in obs]
            reward_windows = None
            step_windows = None
        cur: list = [None] * n  # per-env in-flight chunk state
        pos = [0] * n
        rewards: list[list[float]] | None = None if buffer is not None else [[] for _ in range(n)]
        step_states: list[list[torch.Tensor]] | None = (
            None if buffer is not None else [[] for _ in range(n)]
        )
        prev: list[Transition | None] = [None] * n
        prev_done = ["continue"] * n
        executed_total = [0] * n
        awaiting = [False] * n
        records: list[list[Transition]] | None = (
            None if buffer is not None else [[] for _ in range(n)]
        )
        prev_chunk_indices: list[int | None] = [None] * n

        actions_t = torch.zeros((n, act_dim), dtype=torch.float32)
        while any(executed_total[i] < targets[i] for i in range(n)):
            need = [
                i
                for i in range(n)
                if (
                    cur[i] is None
                    and not awaiting[i]
                    and executed_total[i] < targets[i]
                )
            ]
            if need:
                if buffer is not None:
                    states = windows[need].clone()
                else:
                    states = torch.stack(
                        [torch.stack(tuple(windows[i]), dim=0) for i in need]
                    )
                proposed = as_float_tensor(policy(states))
                for k, i in enumerate(need):
                    executed, scaled, clip_count = scale_and_clip_chunk(
                        proposed[k], low, range_
                    )
                    execution_limit = min(
                        n_action_steps, targets[i] - executed_total[i]
                    )
                    cur[i] = (
                        executed,
                        scaled,
                        clip_count,
                        states[k],
                        execution_limit,
                    )
                    pos[i] = 0
                    if rewards is not None:
                        rewards[i] = []
                    if step_states is not None:
                        step_states[i] = []

            # Gather all active actions into a tensor, then ONE host transfer.
            actions_t.zero_()
            any_active = False
            for i in range(n):
                if cur[i] is not None:
                    actions_t[i] = cur[i][0][pos[i]]
                    any_active = True
            if not any_active:
                # All envs over budget or awaiting: nothing left to step.
                break
            actions_np = actions_t.cpu().numpy()

            obs, rew, term, trunc, _ = envs.step(actions_np)
            term_np = np.asarray(term, dtype=bool)
            trunc_np = np.asarray(trunc, dtype=bool)
            done_np = term_np | trunc_np
            rew_np = np.asarray(rew, dtype=np.float32)

            for i in range(n):
                if awaiting[i]:
                    if buffer is not None:
                        windows[i].copy_(as_float_tensor(obs[i]).expand_as(windows[i]))
                    else:
                        windows[i] = build_warmup_window(
                            as_float_tensor(obs[i]), n_obs_steps
                        )
                    awaiting[i] = False
                    continue
                if cur[i] is None:
                    continue
                if buffer is not None:
                    reward_windows[i, pos[i]] = float(rew_np[i])
                    windows[i, :-1].copy_(windows[i, 1:].clone())
                    windows[i, -1].copy_(as_float_tensor(obs[i]))
                    step_windows[i, pos[i]].copy_(windows[i])
                else:
                    rewards[i].append(float(rew_np[i]))
                    windows[i].append(as_float_tensor(obs[i]))
                    step_states[i].append(torch.stack(tuple(windows[i]), dim=0))
                pos[i] += 1
                execution_limit = cur[i][4]
                if done_np[i] or pos[i] >= execution_limit:
                    _executed, scaled, clip_count, replan, _limit = cur[i]
                    n_exec = pos[i]
                    if term_np[i]:
                        done_kind = "terminated"
                    elif trunc_np[i]:
                        done_kind = "truncated"
                    else:
                        done_kind = "continue"
                    next_state = (
                        windows[i]
                        if buffer is not None
                        else torch.stack(tuple(windows[i]), dim=0)
                    )
                    executed_total[i] += n_exec
                    if buffer is not None:
                        prev_chunk_indices[i] = buffer.add_chunk(
                            state=replan,
                            action=scaled,
                            rewards=reward_windows[i, :n_exec],
                            step_next_states=step_windows[i, :n_exec],
                            done=bool(term_np[i]),
                            truncated=bool(trunc_np[i]),
                            clip_count=clip_count,
                        )
                    else:
                        record = finalize_transition(
                            replan,
                            next_state,
                            scaled,
                            rewards[i],
                            done_kind,
                            clip_count,
                            chunk_size,
                            execution_limit,
                            torch.stack(step_states[i], dim=0),
                        )
                        records[i].append(record)
                        prev[i] = record
                    prev_done[i] = done_kind
                    cur[i] = None
                    if done_np[i]:
                        awaiting[i] = True

        # Budget-exhausted tails: mark as truncated so GAE bootstraps the cut.
        for i in range(n):
            if prev_done[i] == "continue" and (
                prev[i] is not None or prev_chunk_indices[i] is not None
            ):
                if buffer is not None:
                    buffer.mark_truncated(prev_chunk_indices[i])  # type: ignore[arg-type]
                else:
                    prev[i][TRUNCATED].fill_(True)  # type: ignore[index]

        if buffer is not None:
            return buffer
        return [r for traj in records for r in traj]  # type: ignore[union-attr]

    except Exception:
        flush_on_collection_error(on_error_flush)
        raise


# ---------------------------------------------------------------------------
# Single-env collector
# ---------------------------------------------------------------------------


def collect_chunks(
    env,
    policy: Callable[[torch.Tensor], torch.Tensor],
    total_steps: int,
    action_low: float | np.ndarray,
    action_high: float | np.ndarray,
    *,
    n_obs_steps: int | None = None,
    n_action_steps: int,
    chunk_size: int | None = None,
    on_error_flush: Callable[[], None] | None = None,
    buffer: RolloutBuffer | None = None,
) -> list[Transition] | RolloutBuffer:
    cfg = _defaults()
    n_obs_steps = cfg.n_obs_steps if n_obs_steps is None else n_obs_steps
    chunk_size = cfg.action_horizon if chunk_size is None else int(chunk_size)

    low, range_ = _bounds_to_tensor(action_low, action_high)

    if not 1 <= n_action_steps <= chunk_size:
        raise ValueError(
            f"n_action_steps={n_action_steps} must satisfy "
            f"1 <= n <= chunk_size={chunk_size}"
        )
    try:
        obs, _ = env.reset()
        window_deque = build_warmup_window(as_float_tensor(obs), n_obs_steps)
        if buffer is not None:
            window = torch.stack(tuple(window_deque), dim=0)
            step_windows = torch.zeros(chunk_size, n_obs_steps, int(window.shape[-1]))
            reward_window = torch.zeros(chunk_size, dtype=_REWARD_DTYPE)
        records: list[Transition] | None = [] if buffer is None else None
        executed_total = 0
        while executed_total < total_steps:
            if buffer is not None:
                state = window.clone()
                proposed = as_float_tensor(policy(state))
                executed, scaled, clip_count = scale_and_clip_chunk(proposed, low, range_)
                executed_np = executed.detach().cpu().numpy()
                n_exec, done_kind = execute_open_loop_steps_into(
                    env,
                    executed_np,
                    window,
                    step_windows,
                    reward_window,
                    executed_total,
                    total_steps,
                    n_action_steps,
                )
                if n_exec < 1:
                    raise RuntimeError("buffer collection executed zero steps")
                chunk_index = buffer.add_chunk(
                    state=state,
                    action=scaled,
                    rewards=reward_window[:n_exec],
                    step_next_states=step_windows[:n_exec],
                    done=done_kind == "terminated",
                    truncated=done_kind == "truncated",
                    clip_count=clip_count,
                )
                if done_kind == "continue":
                    buffer.mark_truncated(chunk_index)
                executed_total += n_exec
                if done_kind != "continue":
                    obs, _ = env.reset()
                    window.copy_(as_float_tensor(obs).expand_as(window))
            else:
                record, done_kind = propose_and_execute_chunk(
                    policy,
                    window_deque,
                    low,
                    range_,
                    env,
                    executed_total,
                    total_steps,
                    chunk_size,
                    n_action_steps,
                )
                executed_total += int(record["executed_n_action_steps"])
                records.append(record)  # type: ignore[union-attr]
                if done_kind != "continue":
                    window_deque = reset_chunk_window(env, n_obs_steps)
        if buffer is not None:
            return buffer
        return records  # type: ignore[return-value]
    except Exception:
        flush_on_collection_error(on_error_flush)
        raise

In [ ]:
%%writefile src/sjfpo/config.py
"""Environment registry and run-matrix helpers for the SJFPO benchmark."""

from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path


IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

ENV_REGISTRY_PATH = Path(__file__).resolve().parents[2] / "configs" / "envs.yaml"

try:
    import yaml

    with open(ENV_REGISTRY_PATH) as f:
        _ENV_REGISTRY = yaml.safe_load(f) or {}
except Exception:
    _ENV_REGISTRY = {}

_REGISTRY_KEYS = list(_ENV_REGISTRY.keys()) if isinstance(_ENV_REGISTRY, dict) else []
ENVS = tuple(k for k in _REGISTRY_KEYS if _ENV_REGISTRY[k].get("tier") == "core")
PROTOTYPE_ENVS = tuple(
    k for k in _REGISTRY_KEYS if _ENV_REGISTRY[k].get("tier") != "core"
)
KNOWN_ENVS = ENVS + PROTOTYPE_ENVS


@dataclass(frozen=True)
class RunSpec:
    method: str
    env: str
    seed: int
    total_steps: int
    eval_steps: int
    save_steps: int


def parse_version_tuple(version: str) -> tuple[int, ...]:
    """Version string -> (major, minor, patch, ...) for comparison."""
    parts: list[int] = []
    for part in version.split("."):
        digits = "".join(ch for ch in part if ch.isdigit())
        parts.append(int(digits) if digits else 0)
    return tuple(parts)


def assert_runtime(allow_fp16: bool = False) -> None:
    """Fail fast on dependency pins and unsupported precision flags.      bf16 *autocast* (fp32 storage, bf16 matmuls) is allow"""
    import importlib.metadata

    gym_version = importlib.metadata.version("gymnasium")
    if parse_version_tuple(gym_version)[:2] < (1, 0):
        raise RuntimeError(
            f"gymnasium {gym_version} has major.minor < 1.0; "
            "this project requires gymnasium[mujoco]>=1.0,<1.3"
        )

    if allow_fp16:
        raise RuntimeError(
            "fp16/bf16 storage requested but unsupported: flow-matching "
            "Euler rollouts and CFM losses require fp32 storage; use "
            "SJFPO_DTYPE=bf16 for bf16 autocast (fp32 params, bf16 matmuls)"
        )


def lookup_env_dims(env_id: str) -> tuple[int, int]:
    """(obs_dim, action_dim) from configs/envs.yaml; KeyError on unknown ids."""
    try:
        import yaml
    except ImportError:
        raise RuntimeError("pyyaml required for --env lookup (pip install pyyaml)")
    with open(ENV_REGISTRY_PATH) as f:
        registry = yaml.safe_load(f)
    try:
        entry = registry[env_id]
    except KeyError:
        raise KeyError(f"unknown env {env_id!r}; add it to {ENV_REGISTRY_PATH}")
    return int(entry["obs_dim"]), int(entry["action_dim"])

In [ ]:
%%writefile src/sjfpo/configuration_sjfpo.py
"""Single home for every SJFPO hyperparameter (LeRobot ``PreTrainedConfig``)."""

from __future__ import annotations

from dataclasses import dataclass

from lerobot.configs import PreTrainedConfig
from lerobot.optim import AdamWConfig

__all__ = ["SJFPOConfig"]


@PreTrainedConfig.register_subclass("sjfpo")
@dataclass
class SJFPOConfig(PreTrainedConfig):
    """SJFPO hyperparameters.

    Notes
    -----
    # ``n_action_steps`` must be set explicitly; ``action_horizon`` controls
      the full predicted chunk length (training) and the executed prefix length (eval).
    * ``num_inference_steps`` is used when sampling with ``noise=False``;
      ``num_euler_steps`` is used when sampling with ``noise=True``.
    """

    # -- Observation / action geometry -------------------------------------
    n_obs_steps: int = 16
    action_horizon: int = 16
    n_action_steps: int = 16  # must be set explicitly; no derived value
    action_perturb_std: float = 0.02
    hidden_dim: int = 128

    # -- Monte-Carlo / rollout ---------------------------------------------
    num_mc_samples: int = 16
    num_euler_steps: int = 16
    num_inference_steps: int = 16
    chunks_per_update: int = 256
    num_epochs: int = 16
    num_minibatches: int = 4
    # 0 derives a stable capacity from chunks_per_update/n_action_steps.
    compile_batch_size: int = 0
    num_envs: int = 64

    # -- Optimization ------------------------------------------------------
    gamma: float = 0.999
    gae_lambda: float = 0.995
    epsilon_clip: float = 0.05
    use_dct_transform: bool = False  # spectral chunk encoding; False = raw action chunk
    # FPO++ trust region: ppo, spo, or aspo (official default).
    trust_region_mode: str = "aspo"
    use_per_bin_clip: bool = False  # legacy diagnostics-only bin clipping
    epsilon_clip_lowfreq: float = 0.02  # tight: low bins carry task intent
    epsilon_clip_highfreq: float = 0.10  # loose: high bins are detail/exploration
    # "mean" (E[(v-v*)^2] over the chunk) keeps the CFM loss O(1) -- ~1.25 for
    # a zero-velocity model on unit noise -- so cfm_clamp below is a real guard.
    # "sqrt" (FPO++'s default) scales the loss by sqrt(horizon*action_dim); at
    # 16x17 that starts the run above cfm_clamp, and clamp() has zero gradient
    # above its max, which silently pins rho at 1 and kills the policy gradient.
    cfm_loss_reduction: str = "mean"  # mean | sum | sqrt
    cfm_clamp: float = 20.0
    cfm_loss_clamp_negative_advantages: bool = True
    cfm_loss_clamp_negative_advantages_max: float = 20.0
    log_ratio_cap: float = 10.0  # official cfm_diff_clamp_max (one-sided)
    advantage_clamp: float = 100.0
    normalize_advantage: bool = True
    use_clipped_value_loss: bool = False
    min_timestep: float = 0.005
    max_timestep: float = 0.995
    timestep_sampling_mode: str = "fpo_beta"
    noise_sampling_mode: str = "normal"
    timestep_inverse_cdf_beta: float = 1.0
    timestep_beta_alpha: float = 2.0
    timestep_beta_beta: float = 2.0

    # -- DiT backbone ------------------------------------------------------
    dit_depth: int = 4
    dit_heads: int = 4
    dit_mlp_ratio: float = 4.0
    time_embed_dim: int = 128
    # Flow time t in [0, 1] is scaled by this before the DDPM sinusoidal ladder
    # (designed for integer steps 0–1000); without it every channel sits at
    # angle <= 1 and t is nearly unresolvable.
    timestep_scale: float = 1000.0

    # -- EMA actor ----------------------------------------------------------
    ema_decay: float = 0.95
    ema_warmup_steps: int = 500

    # -- Losses / optimization --------------------------------------------
    vf_coef: float = 1.0  # 0.5 was too weak for the critic
    grad_clip_norm: float = 1.0

    # -- Chunk selection ---------------------------------------------------
    top_p_spectral_tokens: float = 1.0

    # -- Optimizer ---------------------------------------------------------
    policy_lr: float = 1e-4
    critic_lr: float = 1e-4  # legacy split-checkpoint compatibility
    optimizer_weight_decay: float = 1e-4

    # -- Training loop -----------------------------------------------------
    # No separate warm-up phase; stats initialize naturally during first rollout.
    # -- Blackwell / GPU memory --------------------------------------------
    use_gradient_checkpointing: bool = False
    grad_accum_steps: int = 1  # >1 reduces GPU memory at cost of speed

    # ----------------------------------------------------------------------
    # Validation
    # ----------------------------------------------------------------------

    def __post_init__(self) -> None:
        super().__post_init__()
        self._resolve_n_action_steps()
        self._validate_geometry()
        self._validate_loss_params()
        self._validate_optimizer()
        self._validate_flags()

    def _resolve_n_action_steps(self) -> None:
        """If ``n_action_steps`` is None, use ``action_horizon`` as the default."""
        if self.n_action_steps is None:
            self.n_action_steps = self.action_horizon

    def _validate_geometry(self) -> None:
        if self.n_obs_steps < 1:
            raise ValueError(f"n_obs_steps={self.n_obs_steps} must be >= 1")
        if self.action_horizon < 1:
            raise ValueError(f"action_horizon={self.action_horizon} must be >= 1")
        if not 0 < self.n_action_steps <= self.action_horizon:
            raise ValueError(
                f"n_action_steps={self.n_action_steps} must satisfy "
                f"0 < n <= action_horizon={self.action_horizon}"
            )
        if self.hidden_dim < 1:
            raise ValueError(f"hidden_dim={self.hidden_dim} must be >= 1")
        if self.action_perturb_std < 0:
            raise ValueError("action_perturb_std must be >= 0")
        if not 0.0 <= self.ema_decay < 1.0:
            raise ValueError("ema_decay must be in [0, 1)")
        if self.ema_warmup_steps < 0:
            raise ValueError("ema_warmup_steps must be >= 0")
        if self.hidden_dim % self.dit_heads != 0:
            raise ValueError(
                f"hidden_dim={self.hidden_dim} must be divisible by "
                f"dit_heads={self.dit_heads}"
            )
        if self.dit_depth < 1:
            raise ValueError(f"dit_depth={self.dit_depth} must be >= 1")
        if self.dit_heads < 1:
            raise ValueError(f"dit_heads={self.dit_heads} must be >= 1")
        if self.dit_mlp_ratio <= 0:
            raise ValueError(f"dit_mlp_ratio={self.dit_mlp_ratio} must be > 0")
        if self.time_embed_dim < 1:
            raise ValueError(f"time_embed_dim={self.time_embed_dim} must be >= 1")
        if self.timestep_scale <= 0:
            raise ValueError(f"timestep_scale={self.timestep_scale} must be > 0")

    def _validate_loss_params(self) -> None:
        if not 0.0 < self.gamma <= 1.0:
            raise ValueError(f"gamma={self.gamma} must satisfy 0 < gamma <= 1")
        if not 0.0 < self.gae_lambda <= 1.0:
            raise ValueError(
                f"gae_lambda={self.gae_lambda} must satisfy 0 < lambda <= 1"
            )
        if not 0.0 < self.epsilon_clip < 1.0:
            raise ValueError(
                f"epsilon_clip={self.epsilon_clip} must satisfy 0 < eps < 1"
            )
        if self.use_per_bin_clip:
            lo, hi = self.epsilon_clip_lowfreq, self.epsilon_clip_highfreq
            if not 0.0 < lo <= hi < 1.0:
                raise ValueError(
                    "require 0 < epsilon_clip_lowfreq <= epsilon_clip_highfreq < 1, "
                    f"got ({lo}, {hi})"
                )
        if self.trust_region_mode not in {"ppo", "spo", "aspo"}:
            raise ValueError("trust_region_mode must be 'ppo', 'spo', or 'aspo'")
        if self.cfm_loss_reduction not in {"mean", "sum", "sqrt"}:
            raise ValueError("cfm_loss_reduction must be 'mean', 'sum', or 'sqrt'")
        if self.cfm_clamp <= 0:
            raise ValueError(f"cfm_clamp={self.cfm_clamp} must be > 0")
        if self.cfm_loss_clamp_negative_advantages_max <= 0:
            raise ValueError("cfm_loss_clamp_negative_advantages_max must be > 0")
        if self.log_ratio_cap <= 0:
            raise ValueError(f"log_ratio_cap={self.log_ratio_cap} must be > 0")
        if self.advantage_clamp <= 0:
            raise ValueError(f"advantage_clamp={self.advantage_clamp} must be > 0")
        if not 0.0 <= self.min_timestep < self.max_timestep <= 1.0:
            raise ValueError(
                "require 0 <= min_timestep < max_timestep <= 1, got "
                f"({self.min_timestep}, {self.max_timestep})"
            )
        if self.timestep_sampling_mode not in {
            "uniform",
            "normal",
            "beta",
            "fpo_beta",
        }:
            raise ValueError(
                "timestep_sampling_mode must be 'uniform', 'normal', 'beta', "
                "or 'fpo_beta'"
            )
        if self.noise_sampling_mode not in {"normal", "uniform"}:
            raise ValueError("noise_sampling_mode must be 'normal' or 'uniform'")
        if self.timestep_inverse_cdf_beta <= 0:
            raise ValueError("timestep_inverse_cdf_beta must be > 0")
        if self.timestep_beta_alpha <= 0 or self.timestep_beta_beta <= 0:
            raise ValueError("timestep beta alpha/beta must be > 0")
        if self.vf_coef < 0:
            raise ValueError(f"vf_coef={self.vf_coef} must be >= 0")

    def _validate_optimizer(self) -> None:
        if self.policy_lr <= 0:
            raise ValueError(f"policy_lr={self.policy_lr} must be > 0")
        if self.critic_lr <= 0:
            raise ValueError(f"critic_lr={self.critic_lr} must be > 0")
        if self.optimizer_weight_decay < 0:
            raise ValueError(
                f"optimizer_weight_decay={self.optimizer_weight_decay} must be >= 0"
            )
        if self.grad_clip_norm <= 0:
            raise ValueError(f"grad_clip_norm={self.grad_clip_norm} must be > 0")
        if self.grad_accum_steps < 1:
            raise ValueError(f"grad_accum_steps={self.grad_accum_steps} must be >= 1")
        if self.num_minibatches < 1:
            raise ValueError(f"num_minibatches={self.num_minibatches} must be >= 1")
        if self.chunks_per_update < 1:
            raise ValueError(f"chunks_per_update={self.chunks_per_update} must be >= 1")
        if self.compile_batch_size < 0:
            raise ValueError("compile_batch_size must be >= 0")
        if self.chunks_per_update < self.num_minibatches:
            raise ValueError(
                f"chunks_per_update={self.chunks_per_update} must be >= "
                f"num_minibatches={self.num_minibatches}"
            )
        if self.num_epochs < 1:
            raise ValueError(f"num_epochs={self.num_epochs} must be >= 1")
        if self.num_envs < 1:
            raise ValueError(f"num_envs={self.num_envs} must be >= 1")
        if self.num_mc_samples < 1:
            raise ValueError(f"num_mc_samples={self.num_mc_samples} must be >= 1")

    def _validate_flags(self) -> None:
        if self.num_euler_steps < 1:
            raise ValueError(f"num_euler_steps={self.num_euler_steps} must be >= 1")
        if self.num_inference_steps < 1:
            raise ValueError(
                f"num_inference_steps={self.num_inference_steps} must be >= 1"
            )
        p = self.top_p_spectral_tokens
        if not 0.0 < p <= 1.0:
            raise ValueError(f"top_p_spectral_tokens={p} must satisfy 0 < p <= 1")
        if self.use_per_bin_clip and not self.use_dct_transform:
            raise ValueError(
                "use_per_bin_clip=True requires use_dct_transform=True "
                "(per-bin epsilon needs frequency bins)"
            )

    # ----------------------------------------------------------------------
    # LeRobot hooks
    # ----------------------------------------------------------------------

    def validate_features(self) -> None:
        """Require observation.state + action features (base class skips this)."""
        if self.robot_state_feature is None:
            raise ValueError("SJFPO requires 'observation.state' in input_features.")
        if self.action_feature is None:
            raise ValueError("SJFPO requires 'action' in output_features.")

    def get_optimizer_preset(self, lr: float | None = None) -> AdamWConfig:
        """Return the standard FP32/bf16-compatible AdamW preset."""
        kwargs = dict(
            lr=self.policy_lr if lr is None else float(lr),
            weight_decay=self.optimizer_weight_decay,
            grad_clip_norm=self.grad_clip_norm,
        )
        return AdamWConfig(**kwargs)

    def get_scheduler_preset(self):
        """No LR scheduler by default. Override in a subclass to return one."""
        return None

    # ----------------------------------------------------------------------
    # Dataset windows
    # ----------------------------------------------------------------------

    @property
    def observation_delta_indices(self) -> list[int] | None:
        """Stacked past frames ending at the current step."""
        return list(range(1 - self.n_obs_steps, 1))

    @property
    def action_delta_indices(self) -> list[int]:
        """Action-chunk offsets the dataset loader returns."""
        return list(range(self.action_horizon))

    @property
    def reward_delta_indices(self) -> None:
        """None: rollouts supply executed-step rewards directly."""
        return None

In [ ]:
%%writefile src/sjfpo/dct_utils.py
"""DCT-based Fourier coefficient utilities for FAFM.

Performance notes
-----------------
* DCT bases are bounded LRU caches keyed by ``(shape, device)`` and stored in
  fp32. Low-precision DCT/IDCT contractions run in fp32 before casting back.
* ``extract_dct_coeffs`` caches its ``[M+1, K]`` analysis basis and the
  ``[M+1]`` normalization vector.
* ``decode_coeffs_to_velocity`` shares its cached normalization vector across
  calls and performs the coefficient contraction in fp32.
* ``sample_freq_weighted_noise`` uses a local ``torch.Generator`` when ``key``
  is provided; otherwise it samples from the global RNG.
"""

from __future__ import annotations

import math
from functools import lru_cache

import torch

from sjfpo.utils import resolve_storage_dtype

# sqrt(0.5), used to scale the DC row so each basis is orthonormal.
_INV_SQRT2 = math.sqrt(0.5)


def _normalize_device(device: torch.device | str | None) -> torch.device:
    target = torch.device("cpu") if device is None else torch.device(device)
    if target.type == "cuda" and target.index is None:
        target = torch.device("cuda", torch.cuda.current_device())
    return target


# ---------------------------------------------------------------------------
# Cached DCT-II basis (fp32) keyed by (n, device)
# ---------------------------------------------------------------------------


def _dct_norm(count: int, sequence_length: int, device: torch.device) -> torch.Tensor:
    """Return orthonormal DCT-II normalization for ``count`` frequencies."""
    if count < 1 or sequence_length < 1:
        raise ValueError("count and sequence_length must be positive")
    norm = torch.full(
        (count,), math.sqrt(2.0 / sequence_length), dtype=torch.float32, device=device
    )
    norm[0] *= _INV_SQRT2
    return norm


def _build_dct_basis_f32(n: int, device: torch.device) -> torch.Tensor:
    """Orthonormal DCT-II matrix ``D[k, j]`` of shape ``[n, n]``, fp32.

    ``D[k, j] = c_k · cos(π k (2j+1) / (2n))`` with ``c_0 = sqrt(1/n)`` and
    ``c_k = sqrt(2/n)`` for ``k > 0``. Forward DCT is ``D @ x``; inverse is
    ``Dᵀ @ x``.
    """
    if n < 1:
        raise ValueError(f"n must be positive, got {n}")
    k = torch.arange(n, dtype=torch.float32, device=device).unsqueeze(1)
    t = torch.arange(n, dtype=torch.float32, device=device).unsqueeze(0)
    norm = _dct_norm(n, n, device)
    return norm[:, None] * torch.cos(
        math.pi * (2.0 * t + 1.0) * k / (2.0 * n)
    )


@lru_cache(maxsize=32)
def _get_dct_basis_f32(n: int, device: torch.device) -> torch.Tensor:
    return _build_dct_basis_f32(n, device)


def dct_basis_matrix(
    n: int, device: torch.device | str | None = None
) -> torch.Tensor:
    """Return a mutable copy of the orthonormal DCT-II basis matrix.

    The returned copy uses the device's resolved storage dtype.
    """
    if n < 1:
        raise ValueError(f"n must be positive, got {n}")
    target = _normalize_device(device)
    return _get_dct_basis_f32(n, target).to(
        target, dtype=resolve_storage_dtype(target), copy=True
    )


# ---------------------------------------------------------------------------
# Analysis / synthesis bases for FAFM
# ---------------------------------------------------------------------------

# Analysis: [M+1, K] basis + [M+1] norm, keyed by (K, M, device).
@lru_cache(maxsize=64)
def _get_analysis_basis(
    K: int, M: int, device: torch.device
) -> tuple[torch.Tensor, torch.Tensor]:
    if K < 1:
        raise ValueError(f"K must be positive, got {K}")
    if M < 0 or M >= K:
        raise ValueError(f"M must satisfy 0 <= M < K, got M={M}, K={K}")
    j = torch.arange(K, dtype=torch.float32, device=device)
    k = torch.arange(M + 1, dtype=torch.float32, device=device)
    basis = torch.cos(math.pi * k[:, None] * (j[None, :] + 0.5) / K)
    return basis, _dct_norm(M + 1, K, device)


def extract_dct_coeffs(v_star: torch.Tensor, M: int) -> torch.Tensor:
    """Extract the first ``M+1`` DCT-II coefficients of a velocity sequence.

    ``v_star`` is either ``[K, d]`` or ``[B, K, d]``; returns ``[M+1, d]``
    or ``[B, M+1, d]`` respectively.
    """
    squeeze = False
    if v_star.dim() == 3:
        B, K, d = v_star.shape
        if B < 1:
            raise ValueError(f"v_star batch size must be positive, got {B}")
    elif v_star.dim() == 2:
        K, d = v_star.shape
        v_star = v_star.unsqueeze(0)
        B = 1
        squeeze = True
    else:
        raise ValueError(
            f"v_star must be 2D (K, d) or 3D (B, K, d), got shape {v_star.shape}"
        )
    if d < 1:
        raise ValueError(f"v_star feature dimension must be positive, got {d}")

    basis, norm = _get_analysis_basis(K, M, _normalize_device(v_star.device))
    # Keep the transform accumulation in fp32, then restore the input dtype.
    with torch.autocast(device_type=v_star.device.type, enabled=False):
        coeffs = torch.einsum("mk,bkd->bmd", basis, v_star.float())
    coeffs = (coeffs * norm[None, :, None]).to(v_star.dtype)

    if squeeze:
        coeffs = coeffs.squeeze(0)
    return coeffs


# Synthesis norm depends only on (M, K_original, device).
@lru_cache(maxsize=64)
def _get_synth_norm(M: int, K_original: int, device: torch.device) -> torch.Tensor:
    return _dct_norm(M + 1, K_original, device)


def decode_coeffs_to_velocity(
    coeffs: torch.Tensor,
    tau: torch.Tensor,
    T: float,
    K_original: int,
) -> torch.Tensor:
    """Decode DCT coefficients to velocities at arbitrary physical times.

    ``coeffs`` is ``[M+1, d]`` or ``[B, M+1, d]``. A 1D ``tau`` with shape
    ``[N]`` is shared across the coefficient batch; a 2D ``tau`` with shape
    ``[B, N]`` must use the same batch size as ``coeffs``. The original K
    samples use the periodic grid ``t_j = j*T/K``; queries use the same
    continuous convention and must lie in ``[0, T]``.
    """
    squeeze = False
    if coeffs.dim() == 3:
        B, M_plus_1, d = coeffs.shape
    elif coeffs.dim() == 2:
        M_plus_1, d = coeffs.shape
        coeffs = coeffs.unsqueeze(0)
        B = 1
        squeeze = True
    else:
        raise ValueError(
            f"coeffs must be 2D (M+1, d) or 3D (B, M+1, d), got shape {coeffs.shape}"
        )

    if B < 1 or M_plus_1 < 1 or d < 1:
        raise ValueError(
            "coeffs must have positive batch, frequency, and feature dimensions"
        )
    if K_original < 1:
        raise ValueError(f"K_original must be positive, got {K_original}")
    if not math.isfinite(T) or T <= 0:
        raise ValueError(f"T must be finite and positive, got {T}")

    M = M_plus_1 - 1
    device = _normalize_device(coeffs.device)
    norm = _get_synth_norm(M, K_original, device)
    k = torch.arange(M + 1, dtype=torch.float32, device=device)

    if tau.dim() == 1:
        tau_query = tau.to(device=device, dtype=torch.float32).unsqueeze(0)
        tau_query = tau_query.expand(B, -1)
    elif tau.dim() == 2:
        if tau.shape[0] != B:
            raise ValueError(
                f"tau batch size {tau.shape[0]} must match coeffs batch size {B}"
            )
        tau_query = tau.to(device=device, dtype=torch.float32)
    else:
        raise ValueError(f"tau must be 1D (N,) or 2D (B, N), got shape {tau.shape}")

    if tau_query.numel() == 0:
        raise ValueError("tau must contain at least one query time")
    if not bool(torch.all((tau_query >= 0) & (tau_query <= T))):
        raise ValueError("tau must lie in [0, T]")

    # The original K samples use t_j = j*T/K, so continuous queries map to
    # j = tau*K/T. This is a periodic half-open grid, not an endpoint-inclusive
    # j*T/(K-1) grid.
    j_indices = tau_query * (K_original / T)
    # [B, M+1, N] basis — depends on tau, cannot be cached.
    basis = torch.cos(
        math.pi * k[None, :, None] * (j_indices[:, None, :] + 0.5) / K_original
    )

    # Do the contraction in fp32 for accuracy; cast back at the end. Autocast
    # must be disabled or CUDA would lower this matmul back to bf16.
    with torch.autocast(device_type=device.type, enabled=False):
        v = torch.einsum(
            "bkd,bkn,k->bnd",
            coeffs.to(torch.float32),
            basis,
            norm,
        )
    v = v.to(coeffs.dtype)

    if squeeze:
        v = v.squeeze(0)
    return v


# ---------------------------------------------------------------------------
# Noise sampling
# ---------------------------------------------------------------------------


def sample_freq_weighted_noise(
    M: int,
    d: int,
    T: float,
    B: int = 1,
    sigma: float = 1.0,
    key: int | None = None,
    device: torch.device | str | None = None,
    dtype: torch.dtype | None = None,
) -> torch.Tensor:
    """Frequency-weighted Gaussian noise of shape ``[B, M+1, d]``.

    Spectral envelope ``std_k = sigma / sqrt(1 + (k π / T)²)``. Uses a local
    ``torch.Generator`` when ``key`` is provided, leaving global RNG state
    untouched; otherwise it samples from the global RNG. Reproducibility with
    a key is guaranteed on the same device, not across CPU/CUDA generators.
    ``dtype`` defaults to the device's resolved storage dtype.
    """
    if M < 0 or d < 1 or B < 1:
        raise ValueError("M must be nonnegative; d and B must be positive")
    if not math.isfinite(T) or T <= 0:
        raise ValueError(f"T must be finite and positive, got {T}")
    device = _normalize_device(device)
    out_dtype = resolve_storage_dtype(device) if dtype is None else dtype
    k_idx = torch.arange(M + 1, dtype=torch.float32, device=device)
    omega_k = k_idx * (math.pi / T)
    std_k = sigma / torch.sqrt(1.0 + omega_k**2)  # [M+1]

    if key is not None:
        gen = torch.Generator(device=device).manual_seed(int(key))
        xi = torch.randn(
            B, M + 1, d, dtype=out_dtype, device=device, generator=gen
        )
    else:
        xi = torch.randn(B, M + 1, d, dtype=out_dtype, device=device)

    return (xi.float() * std_k[None, :, None]).to(out_dtype)


# ---------------------------------------------------------------------------
# Matrix application / DCT / IDCT
# ---------------------------------------------------------------------------


def apply_matrix_along_dim(
    x: torch.Tensor, dim: int, mat: torch.Tensor
) -> torch.Tensor:
    """Apply ``mat`` along ``dim``; preserve x's dtype.

    Half/bfloat16 contractions run in fp32 for accuracy. Float64 remains in
    float64.
    """
    dim = dim % x.dim()
    moved = dim != x.dim() - 1
    if moved:
        x = x.movedim(dim, -1)
    if mat.ndim != 2:
        raise ValueError(f"mat must be 2D, got shape {mat.shape}")
    if mat.shape[0] != x.shape[-1]:
        raise ValueError(
            f"mat.shape[0] must match x.shape[{dim}]={x.shape[-1]}, got {mat.shape[0]}"
        )

    compute_dtype = torch.float64 if x.dtype == torch.float64 else torch.float32
    x_compute = x if x.dtype == compute_dtype else x.to(compute_dtype)
    if mat.device != x_compute.device or mat.dtype != compute_dtype:
        mat = mat.to(x_compute.device, dtype=compute_dtype)
    with torch.autocast(device_type=x_compute.device.type, enabled=False):
        out = x_compute @ mat
    if moved:
        out = out.movedim(-1, dim)
    return out.to(x.dtype)


def dct(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """Orthonormal DCT-II along ``dim`` (forward, ``D @ x``)."""
    if x.dim() < 1:
        raise ValueError("DCT input must have at least one dimension")
    n = x.shape[dim % x.dim()]
    if n < 1:
        raise ValueError(f"DCT length must be positive, got {n}")
    basis = _get_dct_basis_f32(n, x.device)
    return apply_matrix_along_dim(x, dim, basis.t())


def idct(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """Orthonormal inverse DCT-II along ``dim`` (``Dᵀ @ x``)."""
    if x.dim() < 1:
        raise ValueError("DCT input must have at least one dimension")
    n = x.shape[dim % x.dim()]
    if n < 1:
        raise ValueError(f"DCT length must be positive, got {n}")
    basis = _get_dct_basis_f32(n, x.device)
    return apply_matrix_along_dim(x, dim, basis)


# ---------------------------------------------------------------------------
# Chunked reward discounting
# ---------------------------------------------------------------------------


@lru_cache(maxsize=128)
def _discount_weights(n: int, gamma: float, device: torch.device) -> torch.Tensor:
    if not 0.0 <= gamma <= 1.0:
        raise ValueError(f"gamma must be in [0, 1], got {gamma}")
    if n == 0:
        return torch.empty(0, dtype=torch.float32, device=device)
    if gamma == 0.0:
        weights = torch.zeros(n, dtype=torch.float32, device=device)
        weights[0] = 1.0
        return weights
    if gamma == 1.0:
        return torch.ones(n, dtype=torch.float32, device=device)
    # RL chunks are far below 2**24, where fp32 arange remains exact.
    pos = torch.arange(n, dtype=torch.float32, device=device)
    return torch.exp(pos * math.log(gamma))


def discounted_chunk_reward(rewards: torch.Tensor, gamma: float) -> torch.Tensor:
    """Discounted sum of a chunk's rewards, then summed over the time axis."""
    weights = _discount_weights(
        rewards.shape[-1], float(gamma), _normalize_device(rewards.device)
    ).to(rewards.dtype)
    return (rewards * weights).sum(dim=-1)

In [ ]:
%%writefile src/sjfpo/evaluator.py
"""SJFPO evaluator — deterministic rollouts, ``eval_history.csv``.

Rollouts call ``env.reset(seed=...)`` and step until termination/truncation
or the step budget is exhausted. Actions are stored per step in fp32;
``action_jerk`` reports mean squared second-difference as a scale-free
smoothness probe.

"""

from __future__ import annotations

import csv
from collections.abc import Callable
from pathlib import Path
from typing import Any

import numpy as np


CSV_FIELDS = ["episode", "return", "length", "success", "jerk"]


# ---------------------------------------------------------------------------
# CSV IO
# ---------------------------------------------------------------------------


def write_eval_csv(
    results: list[dict[str, Any]],
    out_dir: str | Path,
    *,
    append: bool = False,
) -> Path:
    """Write ``eval_history.csv`` with columns ``[episode, return, length, success, jerk]``.

    When ``append=True``, existing rows are preserved and the header is
    only written if the file doesn't already exist.
    """
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    csv_path = out_path / "eval_history.csv"
    file_exists = csv_path.is_file()
    mode = "a" if append and file_exists else "w"
    with open(csv_path, mode, newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        if not (append and file_exists):
            writer.writeheader()
        writer.writerows(results)
    return csv_path


# ---------------------------------------------------------------------------
# Diagnostics
# ---------------------------------------------------------------------------


def action_jerk(actions: np.ndarray) -> float | None:
    """Mean squared second-difference of actions ``[T, D]``; ``None`` when T < 3.

    Scale-free smoothness probe: constant or constant-velocity motion yields
    exactly ``0.0``; larger values indicate jitter between consecutive steps.
    """
    a = np.asarray(actions, dtype=np.float32)
    if a.ndim != 2 or a.shape[0] < 3:
        return None
    second_diff = a[2:] - 2.0 * a[1:-1] + a[:-2]
    return float(np.square(second_diff).mean())


def extract_success(info: dict) -> float | None:
    """Env-reported task success (``info["success"]``); ``None`` when absent."""
    if not isinstance(info, dict) or info.get("success") is None:
        return None
    return float(bool(info["success"]))


def mean_chunk_jerk(actions: np.ndarray, is_pad: np.ndarray) -> float | None:
    """Mean ``action_jerk`` over chunks' executed prefixes.

    ``actions`` is ``[N, H, D]`` real-space chunks and ``is_pad`` is ``[N, H]``
    with True marking padding past the executed prefix. Chunks with fewer
    than 3 executed steps are skipped; returns ``None`` if no chunk
    qualifies.
    """
    a = np.asarray(actions)
    pad = np.asarray(is_pad, dtype=bool)
    if a.ndim != 3 or pad.shape != a.shape[:2]:
        raise ValueError(
            f"expected actions [N, H, D] and is_pad [N, H]; "
            f"got {a.shape} and {pad.shape}"
        )
    jerks: list[float] = []
    for chunk, mask in zip(a, pad):
        n_exec = int((~mask).sum())
        if n_exec >= 3:
            j = action_jerk(chunk[:n_exec])
            if j is not None:
                jerks.append(j)
    return float(np.mean(jerks)) if jerks else None


def check_divergence(
    mean_return: float,
    baseline: float,
    consecutive: int,
    *,
    threshold_frac: float = 0.10,
    patience: int = 3,
) -> tuple[int, bool]:
    """Increment a sub-threshold counter; flag divergence after ``patience`` in a row.

    The baseline must be strictly positive for the check to apply — a
    non-positive baseline disables divergence detection and resets the
    counter.
    """
    if baseline > 0 and mean_return < threshold_frac * baseline:
        consecutive += 1
    else:
        consecutive = 0
    return consecutive, consecutive >= patience


# ---------------------------------------------------------------------------
# Frame rendering
# ---------------------------------------------------------------------------


def render_episode_frames(
    env_id: str,
    policy: Callable[[np.ndarray], np.ndarray],
    seed: int = 0,
    max_steps: int = 1000,
) -> tuple[list[np.ndarray], float]:
    """One deterministic episode's frames ``[T, H, W, C]`` uint8 + return."""
    import gymnasium as gym

    env = gym.make(env_id, render_mode="rgb_array")
    try:
        obs, _ = env.reset(seed=seed)
        if hasattr(policy, "reset"):
            policy.reset()
        frames: list[np.ndarray] = []
        ret = 0.0
        for _ in range(max_steps):
            action = policy(obs)
            obs, reward, terminated, truncated, _ = env.step(action)
            ret += float(reward)
            frames.append(np.asarray(env.render(), dtype=np.uint8))
            if terminated or truncated:
                break
        return frames, ret
    finally:
        env.close()


# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------


class Evaluator:
    """Deterministic evaluator for flat-observation (MuJoCo) environments."""

    def __init__(
        self,
        env: Any,
        eval_seed: int = 0,
        max_episodes: int = 12,
        max_steps: int | None = None,

    ):
        if env is None:
            raise ValueError("Evaluator requires a non-None env")
        if hasattr(env, "observation_space"):
            try:
                import gymnasium as gym
            except ImportError as e:
                raise ImportError(
                    "gymnasium is required for Evaluator obs-contract validation"
                ) from e
            if isinstance(env.observation_space, gym.spaces.Dict):
                raise AssertionError("Evaluator expects flat Box obs")
        if max_episodes < 0:
            raise ValueError(f"max_episodes={max_episodes} must be >= 0")
        if max_steps is not None and max_steps < 0:
            raise ValueError(f"max_steps={max_steps} must be >= 0 or None")
        self.env = env
        self.eval_seed = eval_seed
        self.max_episodes = max_episodes
        self.max_steps = max_steps

    def rollout_single_episode(
        self,
        policy: Callable[[np.ndarray], np.ndarray],
        seed: int,
        episode: int,
        step_budget: int | None,
    ) -> tuple[dict[str, Any], int]:
        """Roll one deterministic episode; return ``(result, steps_taken)``.

        ``step_budget`` (when set) caps the episode at that many steps;
        the returned length reflects the truncated count.
        """
        obs, _ = self.env.reset(seed=seed)
        if hasattr(policy, "reset"):
            policy.reset()
        ep_return = 0.0
        ep_length = 0
        ep_actions: list[np.ndarray] = []
        ep_success: float | None = None
        done = False
        while not done:
            if step_budget is not None and ep_length >= step_budget:
                break
            action = policy(obs)
            obs, reward, terminated, truncated, info = self.env.step(action)
            ep_return += float(reward)
            ep_length += 1
            done = bool(terminated) or bool(truncated)
            ep_actions.append(np.asarray(action, dtype=np.float32).reshape(-1))
            success = extract_success(info)
            if success is not None:
                ep_success = success  # last report wins (terminal info)
        jerk = action_jerk(np.stack(ep_actions)) if ep_actions else None
        return (
            {
                "episode": episode,
                "return": ep_return,
                "length": ep_length,
                "success": ep_success,
                "jerk": jerk,
            },
            ep_length,
        )

    def evaluate(
        self,
        policy: Callable[[np.ndarray], np.ndarray],
        *,
        out_dir: str | Path | None = None,
        append_csv: bool = False,
    ) -> list[dict[str, Any]]:
        """Run deterministic rollouts, optionally writing ``eval_history.csv``."""
        rng = np.random.default_rng(self.eval_seed)
        results: list[dict[str, Any]] = []
        total_steps = 0
        for ep in range(self.max_episodes):
            if self.max_steps is not None and total_steps >= self.max_steps:
                break
            seed = int(rng.integers(0, 2**31))
            budget = None if self.max_steps is None else self.max_steps - total_steps
            result, taken = self.rollout_single_episode(policy, seed, ep, budget)
            results.append(result)
            total_steps += taken
        if out_dir is not None:
            write_eval_csv(results, out_dir, append=append_csv)
        return results

In [ ]:
%%writefile src/sjfpo/fpo_core.py
"""FPO++ core math: CFM losses, per-sample ratios, ASPO trust region, Euler sampler (arXiv 2602.02481 Eq.3-13)."""

from __future__ import annotations

import math
from collections.abc import Callable
from typing import Any

import torch

from sjfpo.configuration_sjfpo import SJFPOConfig
from sjfpo.utils import resolve_storage_dtype


# ---------------------------------------------------------------------------
# Sampling
# ---------------------------------------------------------------------------


def sample_timesteps(
    num_samples: int,
    cfg: SJFPOConfig,
    device: torch.device | str,
    generator: torch.Generator | None,
):
    """Flow times using the explicit configuration and sample shape."""
    lo, hi = cfg.min_timestep, cfg.max_timestep
    mode = cfg.timestep_sampling_mode
    if mode == "uniform":
        return (
            torch.rand(num_samples, device=device, generator=generator) * (hi - lo) + lo
        )
    if mode == "normal":
        mid = 0.5 * (lo + hi)
        std = 0.25 * (hi - lo)
        z = torch.randn(num_samples, device=device, generator=generator)
        return (z * std + mid).clamp(lo, hi)
    if mode == "fpo_beta":
        u = torch.rand(num_samples, device=device, generator=generator)
        beta = cfg.timestep_inverse_cdf_beta
        return lo + (hi - lo) * (
            1.0 - torch.pow(1.0 - u, 1.0 / beta)
        )
    if mode == "beta":
        from lerobot.policies.common.flow_matching import sample_time_beta

        return sample_time_beta(
            num_samples,
            device,
            alpha=cfg.timestep_beta_alpha,
            beta=cfg.timestep_beta_beta,
            scale=hi - lo,
            offset=lo,
        )
    raise ValueError(
        f"mode={mode!r} unknown; expected 'uniform', 'normal', 'beta', or 'fpo_beta'"
    )


def sample_noise(
    shape,
    cfg: SJFPOConfig,
    device: torch.device | str,
    generator: torch.Generator | None,
):
    """MC noise ε of shape ``[*shape]`` using the explicit configuration."""
    mode = cfg.noise_sampling_mode
    if mode == "normal":
        return torch.randn(*shape, device=device, generator=generator)
    if mode == "uniform":
        s = math.sqrt(3.0)
        return (torch.rand(*shape, device=device, generator=generator) * 2.0 - 1.0) * s
    raise ValueError(f"mode={mode!r} unknown; expected 'normal' or 'uniform'")


def run_mc_simulation(
    a: torch.Tensor, eps: torch.Tensor, tau
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, tuple[int, ...], int]:
    """Broadcast action, noise, and per-coordinate timesteps over MC.

    Legacy scalar/[M]/[B,M] tau inputs return ``flatten_mc_timestep`` with shape
    ``[B*M]``. The training path uses per-token tau with shape
    ``[B, M, K, 1]`` and receives ``[B*M, K, 1]``; each token timestep is shared
    across that token's action dimensions.
    """
    b, num_mc = eps.shape[0], eps.shape[1]
    flow_shape = tuple(a.shape[1:])
    if tuple(eps.shape) != (b, num_mc, *flow_shape):
        raise ValueError(f"eps shape {tuple(eps.shape)} != {(b, num_mc, *flow_shape)}")

    # Coerce tau to a device-matched tensor without a silent CPU hop.
    storage_dtype = resolve_storage_dtype(eps.device)
    if isinstance(tau, torch.Tensor):
        t = tau.to(device=eps.device, dtype=storage_dtype)
    else:
        t = torch.as_tensor(tau, dtype=storage_dtype, device=eps.device)

    tokenwise_tau = t.dim() == 4 and tuple(t.shape) == (
        b,
        num_mc,
        flow_shape[0],
        1,
    )
    if tokenwise_tau:
        pass
    elif t.dim() == 0:
        t = t.expand(b, num_mc)
    elif t.dim() == 3 and tuple(t.shape) == (b, num_mc, 1):
        t = t.squeeze(-1)
    elif t.dim() == 1:
        if t.numel() != num_mc:
            raise ValueError(
                f"tau vector must have num_mc_samples={num_mc} entries, got {t.numel()}"
            )
        t = t.expand(b, num_mc)
    elif tuple(t.shape) != (b, num_mc):
        raise ValueError(
            f"tau shape {tuple(t.shape)} must be scalar, [num_mc_samples], "
            f"[B, num_mc_samples], [B, num_mc_samples, 1], or "
            f"[B, num_mc_samples, K, 1]"
        )

    d = a.numel() // b
    a_flat = a.reshape(b, 1, d).expand(b, num_mc, d).reshape(b * num_mc, d)
    if tokenwise_tau:
        flatten_mc_timestep = t.reshape(b * num_mc, flow_shape[0], 1)
    else:
        flatten_mc_timestep = t.reshape(b * num_mc)
    return (
        flatten_mc_timestep,
        eps.reshape(b * num_mc, d),
        a_flat,
        flow_shape,
        num_mc,
    )


# ---------------------------------------------------------------------------
# CFM loss
# ---------------------------------------------------------------------------


def compute_cfm_loss(
    v_hat: torch.Tensor,
    v_star: torch.Tensor,
    cfg: SJFPOConfig,
    reduce_mode: str,
) -> torch.Tensor:
    """CFM squared-velocity loss (paper Eq.7-8).

    The active ``sample`` path follows FPO++'s configurable reduction over the
    flattened action chunk (``mean``, ``sum``, or ``sqrt``; default ``sqrt``).
    ``bin`` remains a legacy diagnostic reduction.
    """
    if v_hat.shape != v_star.shape:
        raise ValueError(
            f"v_hat shape {tuple(v_hat.shape)} != v_star shape {tuple(v_star.shape)}"
        )
    sq = torch.square(v_hat - v_star)

    if reduce_mode == "scalar":
        return sq.mean()

    if reduce_mode == "sample":
        flat = sq.flatten(2)
        if cfg.cfm_loss_reduction == "mean":
            return flat.mean(dim=-1)
        if cfg.cfm_loss_reduction == "sum":
            return flat.sum(dim=-1)
        return flat.sum(dim=-1) / math.sqrt(flat.shape[-1])
    if reduce_mode == "bin":
        # Legacy diagnostic reduction; the official FPO policy uses "sample".
        return sq.sum(dim=-1) / math.sqrt(sq.shape[-1])
    raise ValueError(
        f"reduce_mode={reduce_mode!r} must be 'scalar', 'sample', or 'bin'"
    )


# ---------------------------------------------------------------------------
# ASPO surrogate / policy ratio / policy loss
# ---------------------------------------------------------------------------


def compute_aspo_surrogate(
    rho: torch.Tensor,
    advantage: torch.Tensor,
    cfg: SJFPOConfig,
    epsilon_clip: float | torch.Tensor,
) -> torch.Tensor:
    """FPO++ trust region: PPO clipping, SPO, or asymmetric ASPO.

    ASPO uses PPO for positive advantages and SPO for negative advantages.
    """
    rho_adv = rho * advantage
    rho_clipped = torch.clamp(rho, 1.0 - epsilon_clip, 1.0 + epsilon_clip)
    ppo = torch.min(rho_adv, rho_clipped * advantage)
    spo = rho_adv - (advantage.abs() / (2.0 * epsilon_clip)) * torch.square(rho - 1.0)

    if cfg.trust_region_mode == "ppo":
        return ppo
    if cfg.trust_region_mode == "spo":
        return spo
    return torch.where(advantage > 0, ppo, spo)


def compute_policy_ratio(
    l_old: torch.Tensor,
    l_new: torch.Tensor,
    cfg: SJFPOConfig,
    advantage: torch.Tensor | None = None,
) -> torch.Tensor:
    """Official FPO++ ratio exp(old CFM loss - new CFM loss).

    The official implementation clamps both CFM losses from above, optionally
    clamps the new loss for negative-advantage samples, then applies a
    one-sided STE cap to the log-ratio.
    """
    old = torch.clamp(l_old, max=cfg.cfm_clamp)
    new = torch.clamp(l_new, max=cfg.cfm_clamp)
    if cfg.cfm_loss_clamp_negative_advantages and advantage is not None:
        new = torch.where(
            advantage < 0,
            torch.clamp(new, max=cfg.cfm_loss_clamp_negative_advantages_max),
            new,
        )
    delta = old - new
    safe_delta = delta.clamp(max=cfg.log_ratio_cap)
    # STE: forward uses the capped delta, backward flows through the original.
    delta_st = safe_delta.detach() + delta - delta.detach()
    return torch.exp(delta_st)


def compute_policy_loss(
    l_old: torch.Tensor,
    l_new: torch.Tensor,
    advantage: torch.Tensor,
    cfg: SJFPOConfig,
) -> torch.Tensor:
    """Scalar FPO++ policy objective: MEAN over i of ψ per chunk (Eq.13)."""
    rho = compute_policy_ratio(l_old, l_new, cfg, advantage=advantage)
    return compute_aspo_surrogate(rho, advantage, cfg, cfg.epsilon_clip).mean()


# ---------------------------------------------------------------------------
# Inference
# ---------------------------------------------------------------------------


def sample_action(
    predict_velocity: Callable[[torch.Tensor, torch.Tensor, Any], torch.Tensor],
    shape,
    condition,
    num_inference_steps: int,
    eps: torch.Tensor | None,
    generator: torch.Generator | None,
    device: torch.device | str,
) -> torch.Tensor:
    """Sample the learned flow t=0 (noise) → t=1 (data) with Euler steps."""
    if device is None:
        raise ValueError("device must be provided explicitly")

    grid = torch.linspace(
        0.0,
        1.0,
        num_inference_steps + 1,
        device=device,
        dtype=torch.float32,
    )

    if eps is None:
        eps = torch.randn(*shape, device=device, generator=generator)
    elif device is not None and eps.device != torch.device(device):
        eps = eps.to(device)

    x = eps
    with torch.inference_mode():
        # Precompute dt once; XLA/torch compile friendly.
        dts = grid[1:] - grid[:-1]  # [steps]
        for i in range(num_inference_steps):
            t_current = grid[i].reshape(1)
            v_hat = predict_velocity(x, t_current, condition)
            x = x + dts[i] * v_hat
    return x

In [ ]:
%%writefile src/sjfpo/gae.py
"""Step-level GAE over executed rollout transitions.

The Torch path computes GAE per executed environment step and then selects
the first-step target of each action chunk.
"""

from __future__ import annotations

import torch




@torch.no_grad()
def compute_step_advantages_torch(
    rewards_1d: torch.Tensor,
    values: torch.Tensor,
    next_values: torch.Tensor,
    terminated: torch.Tensor,
    *,
    gamma: float = 0.99,
    lam: float = 0.95,
    truncated: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute step-level GAE over a flat stream of executed transitions.

    Inputs must be ordered by trajectory. A trajectory boundary must be
    indicated by either ``terminated`` or ``truncated``.

    For transition t, the FPO++ timeout handling adjusts the reward before
    computing the TD residual:

        r_adjusted = r_t + gamma * V(s_t)  when truncated

    Then:

        delta_t = r_adjusted + gamma * bootstrap_t - V(s_t)

    where ``bootstrap_t`` is zero for terminated transitions and
    ``V(s_(t+1))`` otherwise. GAE recursion continues across ordinary
    transitions and stops at both terminated and truncated transitions.

    Returns:
        advantages: GAE advantage for every executed step.
        returns: ``values + advantages``, i.e. the GAE value target.
    """
    if not 0.0 <= gamma <= 1.0:
        raise ValueError(f"gamma must be in [0, 1], got {gamma}")
    if not 0.0 <= lam <= 1.0:
        raise ValueError(f"lam must be in [0, 1], got {lam}")

    if rewards_1d.ndim != 1:
        raise ValueError(
            f"rewards_1d must be 1D, got shape {tuple(rewards_1d.shape)}"
        )

    device = values.device
    dtype = torch.float32

    rewards = rewards_1d.to(device=device, dtype=dtype)
    values = values.to(device=device, dtype=dtype).reshape(-1)
    next_values = next_values.to(device=device, dtype=dtype).reshape(-1)
    terminated = terminated.to(device=device, dtype=torch.bool).reshape(-1)

    if truncated is None:
        truncated = torch.zeros_like(terminated)
    else:
        truncated = truncated.to(device=device, dtype=torch.bool).reshape(-1)

    total_steps = rewards.numel()

    if not (
        values.numel()
        == next_values.numel()
        == terminated.numel()
        == truncated.numel()
        == total_steps
    ):
        raise ValueError(
            "step GAE inputs must all have the same length as rewards_1d"
        )

    if total_steps == 0:
        empty = torch.empty(0, device=device, dtype=dtype)
        return empty, empty

    # Match rsl_rl/FPO++: timeouts are handled by adding gamma * V(s_t) to
    # the reward, then the transition is treated as terminal for GAE.
    effective_rewards = torch.where(
        truncated,
        rewards + gamma * values,
        rewards,
    )
    boundary = terminated | truncated
    bootstrap = torch.where(
        boundary,
        torch.zeros_like(next_values),
        next_values,
    )
    delta = effective_rewards + gamma * bootstrap - values

    advantages = torch.empty_like(delta)
    next_advantage = torch.zeros((), device=device, dtype=dtype)

    continuation_factor = torch.as_tensor(
        gamma * lam,
        device=device,
        dtype=dtype,
    )
    cut_recursion = terminated | truncated

    for index in range(total_steps - 1, -1, -1):
        continuation = torch.where(
            cut_recursion[index],
            torch.zeros((), device=device, dtype=dtype),
            continuation_factor,
        )
        next_advantage = delta[index] + continuation * next_advantage
        advantages[index] = next_advantage

    return advantages, advantages + values

In [ ]:
%%writefile src/sjfpo/modeling_sjfpo.py
"""SJFPO policy: flow head + state conditioner + critic.

Batch contract: ``[B, T, obs]`` states, ``[B, H, action_dim]`` action chunks,
frozen GAE advantages/returns from the rollout.

Performance notes
-----------------
* Diagnostics returned by ``forward`` are batched into a single device→host
  transfer (one ``.tolist()``) instead of ~20 ``float()`` syncs per call, and
  the whole diagnostic pass is gated on ``compute_info=True``.
* ``rho_bin`` is computed once per forward, not once per branch.
* Gradient checkpointing is off by default for shallow DiT stacks; when on,
  it actually wraps the actor call.
* ``snapshot_behavior_policy`` copies weights in-place into the frozen
  snapshot instead of ``copy.deepcopy``-ing a ``torch.compile`` wrapper.
"""

from __future__ import annotations

import copy
from collections import deque
from typing import Any

import einx
import numpy as np
import torch
import torch.utils.checkpoint as cp

from lerobot.policies import PreTrainedPolicy
from lerobot.processor.converters import to_tensor
from lerobot.utils.constants import ACTION, OBS_STATE

from sjfpo.configuration_sjfpo import SJFPOConfig
from sjfpo.collector import RolloutBuffer, collect_chunks, collect_chunks_vec
from sjfpo.dct_utils import (
    decode_coeffs_to_velocity,
    extract_dct_coeffs,
)
from sjfpo.fpo_core import (
    compute_aspo_surrogate,
    compute_cfm_loss,
    compute_policy_ratio,
    run_mc_simulation,
    sample_action,
    sample_noise,
    sample_timesteps,
)
from sjfpo.gae import compute_step_advantages_torch
from sjfpo.networks import (
    Critic,
    RunningMeanStd,
    SJFPOFlowModel,
    StateConditioner,
)
from sjfpo.processor_sjfpo import make_sjfpo_pre_post_processors
from sjfpo.utils import (
    autocast_context,
    resolve_storage_dtype,
    to_device_async,
    to_tensor_1d,
)

class SJFPOAborted(RuntimeError):
    """Non-finite training loss (message names the offending term)."""

    def __init__(self, message: str):
        super().__init__(message)
        self.message = message


# ---------------------------------------------------------------------------
# Small helpers (avoid repeating shape arithmetic)
# ---------------------------------------------------------------------------


def _num_spectral_tokens_kept(cfg: SJFPOConfig) -> int:
    """Number of spectral tokens kept per action chunk."""
    return max(1, round(cfg.top_p_spectral_tokens * cfg.action_horizon)) if cfg.use_dct_transform else cfg.action_horizon


def _sample_spectral_eps(
    cfg: SJFPOConfig,
    n: int,
    action_dim: int,
    device: torch.device,
) -> torch.Tensor:
    """Draw MC noise matching the token layout (DCT or raw), on ``device``."""
    shape = (
        n,
        cfg.num_mc_samples,
        (
            _num_spectral_tokens_kept(cfg)
            if cfg.use_dct_transform
            else cfg.action_horizon
        ),
        action_dim,
    )
    return sample_noise(shape, cfg, device, None)


def _mc_x_t(
    flatten_mc_timestep: torch.Tensor,
    flatten_mc_noise: torch.Tensor,
    flatten_mc_action: torch.Tensor,
    flow_shape: tuple[int, ...],
) -> torch.Tensor:
    if flatten_mc_timestep.dim() == 1:
        flatten_mc_timestep = flatten_mc_timestep[:, None].expand(
            -1, flatten_mc_noise.shape[-1]
        )
        flatten_mc_noise = flatten_mc_noise.reshape(flatten_mc_timestep.shape[0], -1)
        flatten_mc_action = flatten_mc_action.reshape(flatten_mc_timestep.shape[0], -1)
    else:
        flatten_mc_timestep = flatten_mc_timestep.expand(-1, -1, flow_shape[-1])
        flatten_mc_noise = flatten_mc_noise.reshape(
            flatten_mc_timestep.shape[0], flow_shape[0], flow_shape[1]
        )
        flatten_mc_action = flatten_mc_action.reshape(
            flatten_mc_timestep.shape[0], flow_shape[0], flow_shape[1]
        )
    return (
        (1.0 - flatten_mc_timestep) * flatten_mc_noise
        + flatten_mc_timestep * flatten_mc_action
    ).reshape(-1, *flow_shape)


def _actor_tau(
    flatten_mc_timestep: torch.Tensor, flow_shape: tuple[int, ...]
) -> torch.Tensor:
    if flatten_mc_timestep.dim() == 1:
        return flatten_mc_timestep
    return flatten_mc_timestep.squeeze(-1)


def _actor_call(actor, x_t, tau, cond, *, use_ckpt: bool):
    """Actor forward, optionally under gradient checkpointing."""
    if use_ckpt:
        return cp.checkpoint(actor, x_t, tau, cond, use_reentrant=False)
    return actor(x_t, tau, cond)


def _unwrap_compiled(m: torch.nn.Module) -> torch.nn.Module:
    """Return the eager module behind a ``torch.compile`` wrapper, if any."""
    return getattr(m, "_orig_mod", m)


# ---------------------------------------------------------------------------
# Policy
# ---------------------------------------------------------------------------


class SJFPOPolicy(PreTrainedPolicy):
    config_class = SJFPOConfig
    name = "sjfpo"

    def __init__(
        self, config: SJFPOConfig, dataset_stats: dict[str, Any] | None = None
    ):
        super().__init__(config, dataset_stats)
        config.validate_features()
        obs_dim = int(config.robot_state_feature.shape[0])
        action_dim = int(config.action_feature.shape[0])

        self.actor = SJFPOFlowModel(
            action_dim,
            config.action_horizon,
            config.hidden_dim,
            config.dit_depth,
            config.dit_heads,
            config.dit_mlp_ratio,
            config.time_embed_dim,
            config.timestep_scale,
        )
        self.conditioner = StateConditioner(obs_dim, config.hidden_dim)
        self.critic = Critic(obs_dim, config.hidden_dim)
        self.obs_normalizer = RunningMeanStd(obs_dim)

        self.register_buffer(
            "action_low", torch.full((action_dim,), -1.0), persistent=False
        )
        self.register_buffer(
            "action_high", torch.full((action_dim,), 1.0), persistent=False
        )

        # Frozen behavior snapshot (PPO π_old); refreshed per collection round.
        self.old_actor = copy.deepcopy(self.actor).requires_grad_(False).eval()
        self.old_conditioner = (
            copy.deepcopy(self.conditioner).requires_grad_(False).eval()
        )
        self._ema_shadow: dict[str, torch.Tensor] = {}
        self._ema_initialized = False
        self._rollout_buffer: RolloutBuffer | None = None
        self._device_rollout_buffer: RolloutBuffer | None = None
        self._rollout_h2d_event: torch.cuda.Event | None = None
        # Fraction of the last forward's CFM samples above ``cfm_clamp``. Those
        # samples get zero gradient from the ratio, so the training loop reads
        # this to abort rather than run on with rho pinned at 1. Left on device
        # so the loop syncs once per rollout, not once per microbatch.
        self.cfm_above_clamp_frac = torch.zeros(())

        self._frame_buffer: deque[torch.Tensor] = deque(maxlen=config.n_obs_steps)
        self._action_buffer: deque[torch.Tensor] = deque()
        self.step_count = 0
        self.preprocessor, self.postprocessor = make_sjfpo_pre_post_processors(self)
        del dataset_stats  # kept for base-class signature compatibility

    # -- module state --------------------------------------------------

    def train(self, mode: bool = True):
        super().train(mode)
        self.old_actor.eval()
        self.old_conditioner.eval()
        return self

    def reset(self):
        self._frame_buffer.clear()
        self._action_buffer.clear()

    @torch.inference_mode()
    def update_observation_stats(self, observations: torch.Tensor) -> None:
        """Update the live running statistics from raw post-step observations."""
        if (
            observations.ndim != 2
            or observations.shape[-1] != self.obs_normalizer.mean.numel()
        ):
            raise ValueError(
                "observations must have shape [N, observation_dim], got "
                f"{tuple(observations.shape)}"
            )
        self.obs_normalizer.update(observations)

    def set_action_bounds(
        self,
        low: np.ndarray | torch.Tensor,
        high: np.ndarray | torch.Tensor,
    ) -> None:
        storage_dtype = resolve_storage_dtype(self.action_low.device)
        low_t = to_tensor_1d(
            low, dtype=storage_dtype, device=self.action_low.device
        ).reshape(-1)
        high_t = to_tensor_1d(
            high, dtype=storage_dtype, device=self.action_high.device
        ).reshape(-1)
        if (
            low_t.shape != self.action_low.shape
            or high_t.shape != self.action_high.shape
        ):
            raise ValueError(
                f"action bounds must match action_dim={self.action_low.numel()}: "
                f"low={tuple(low_t.shape)}, high={tuple(high_t.shape)}"
            )
        if not bool((high_t >= low_t).all()):
            raise ValueError("action_high must be elementwise >= action_low")

        self.action_low.copy_(low_t)
        self.action_high.copy_(high_t)
        self.postprocessor.set_action_bounds(self.action_low, self.action_high)

    def snapshot_behavior_policy(self) -> None:
        """Refresh the frozen π_old snapshot in place.

        Avoids ``copy.deepcopy`` on ``torch.compile`` wrappers
        (``OptimizedModule``), which clones the FX graph + guards and forces a
        fresh compile warmup on the snapshot every rollout. Copying weights
        into the existing modules instead keeps the compiled graph intact.
        """
        actor_old = _unwrap_compiled(self.old_actor)
        actor_live = _unwrap_compiled(self.actor)
        actor_old.load_state_dict(actor_live.state_dict())

        cond_old = _unwrap_compiled(self.old_conditioner)
        cond_live = _unwrap_compiled(self.conditioner)
        cond_old.load_state_dict(cond_live.state_dict())

        self.old_actor.requires_grad_(False)
        self.old_actor.eval()
        self.old_conditioner.requires_grad_(False)
        self.old_conditioner.eval()

    @torch.inference_mode()
    def update_ema(self, global_step: int) -> None:
        """Update the FPO++ actor EMA after an optimizer update."""
        decay = self.config.ema_decay
        if decay <= 0:
            return
        actor = _unwrap_compiled(self.actor)
        current = {
            name: param
            for name, param in actor.named_parameters()
            if param.requires_grad
        }
        if not self._ema_shadow:
            self._ema_shadow = {
                name: param.detach().clone() for name, param in current.items()
            }
        if global_step == self.config.ema_warmup_steps:
            for name, shadow in self._ema_shadow.items():
                shadow.copy_(current[name].detach())
            self._ema_initialized = True
            return
        if global_step > self.config.ema_warmup_steps:
            for name, shadow in self._ema_shadow.items():
                shadow.mul_(decay).add_(current[name].detach(), alpha=1.0 - decay)

    def ema_ready(self, global_step: int) -> bool:
        return (
            self.config.ema_decay > 0
            and self._ema_initialized
            and global_step > self.config.ema_warmup_steps
        )

    def ema_state_dict(self) -> dict:
        return {
            "decay": self.config.ema_decay,
            "initialized": self._ema_initialized,
            "shadow_params": self._ema_shadow,
        }

    def load_ema_state_dict(self, state: dict) -> None:
        self._ema_shadow = {
            name: value.detach().clone()
            for name, value in state["shadow_params"].items()
        }
        self._ema_initialized = bool(state.get("initialized", True))

    def get_actor_optim_params(self) -> dict:
        decay, no_decay = [], []
        for name, p in self.actor.named_parameters():
            short = name.split("_orig_mod.")[-1]
            if "adaLN" in short or short.startswith("out."):
                no_decay.append(p)
            else:
                decay.append(p)
        policy_lr = self.config.policy_lr
        return {
            "params": [
                {"params": decay, "lr": policy_lr},
                {"params": no_decay, "weight_decay": 0.0, "lr": policy_lr},
                {"params": list(self.conditioner.parameters()), "lr": policy_lr},
            ]
        }

    def get_critic_optim_params(self) -> dict:
        return {
            "params": [
                {
                    "params": list(self.critic.parameters()),
                    "lr": self.config.critic_lr,
                }
            ]
        }

    def get_optim_params(self) -> dict:
        """One FPO++-style optimizer group over all trainable policy weights."""
        params = [p for p in self.parameters() if p.requires_grad]
        return {
            "params": [
                {
                    "params": params,
                    "lr": self.config.policy_lr,
                    "weight_decay": self.config.optimizer_weight_decay,
                }
            ]
        }

    # -- velocity helpers ----------------------------------------------

    @staticmethod
    def compute_target_velocity(a: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
        """v* = a − ε broadcast over MC draws: [B, *flow] x [B, m, *flow] → [B, m, *flow]."""
        return a.unsqueeze(1) - eps

    def compute_predicted_velocity(self, x_t, tau, states, old: bool = False):
        conditioner = self.old_conditioner if old else self.conditioner
        net = self.old_actor if old else self.actor
        return net(x_t, tau, conditioner(states))

    # -- observation coercion ------------------------------------------

    def standardize_obs_tensor(self, states: torch.Tensor) -> torch.Tensor:
        states = to_tensor(states)
        states = states.to(dtype=resolve_storage_dtype(states.device))
        if states.dim() == 3:
            return states
        if states.dim() == 2:
            if states.shape[0] == self.config.n_obs_steps:
                return einx.id("t d -> 1 t d", states)
            return einx.id("d -> 1 t d", states, t=self.config.n_obs_steps)
        raise ValueError(
            f"observation.state must be [B, T, obs], [T, obs] or [B, obs]; "
            f"got {tuple(states.shape)}"
        )

    # -- inference -----------------------------------------------------

    def predict_action_chunk(
        self, batch: dict[str, torch.Tensor], **kwargs
    ) -> torch.Tensor:
        """Full action chunk ``[B, H, action_dim]``; ``noise=True`` samples, else zeros."""
        noise = kwargs.get("noise", False)
        cfg = self.config
        device = next(self.actor.parameters()).device

        states = self.standardize_obs_tensor(batch[OBS_STATE])
        states = to_device_async(states, device)
        states = self.preprocessor({OBS_STATE: states})[OBS_STATE]

        num_spectral_tokens_kept = _num_spectral_tokens_kept(cfg)
        action_dim = int(self.config.action_feature.shape[0])
        actor = _unwrap_compiled(self.actor)

        with torch.inference_mode(), autocast_context(device):
            condition = self.conditioner(states)
            shape = (states.shape[0], num_spectral_tokens_kept, action_dim)
            eps = None if noise else torch.zeros(*shape, device=device)
            num_steps = cfg.num_euler_steps if noise else cfg.num_inference_steps
            x0 = sample_action(
                lambda x, t, c: actor(x, t, c),
                shape,
                condition,
                num_inference_steps=num_steps,
                eps=eps,
                generator=None,
                device=device,
            ).float()

        if cfg.use_dct_transform:
            tau_grid = torch.linspace(
                0.0, 1.0, cfg.action_horizon, device=x0.device, dtype=torch.float32
            )
            scaled = decode_coeffs_to_velocity(
                x0, tau=tau_grid, T=1.0, K_original=cfg.action_horizon
            )[:, : cfg.action_horizon, :].clamp(-1.0, 1.0)
        else:
            scaled = x0[:, : cfg.action_horizon, :].clamp(-1.0, 1.0)

        # Match FPO++ action perturbation before postprocessing, so the noise is
        # applied in the policy's normalized action space.
        if self.training and cfg.action_perturb_std > 0:
            scaled = scaled + cfg.action_perturb_std * torch.randn_like(scaled)
        return self.postprocessor({ACTION: scaled})[ACTION]

    def select_action(self, batch: dict[str, torch.Tensor], **kwargs) -> torch.Tensor:
        obs = to_tensor(batch[OBS_STATE])
        obs = obs.to(dtype=resolve_storage_dtype(obs.device))
        if obs.dim() == 1:
            if not self._frame_buffer:
                for _ in range(self.config.n_obs_steps):
                    self._frame_buffer.append(obs.detach().clone())
            else:
                self._frame_buffer.append(obs.detach())
            window = torch.stack(tuple(self._frame_buffer), dim=0)
        elif obs.dim() == 2:
            window = obs
        else:
            raise ValueError(
                f"select_action expects [obs] or [T, obs]; got {tuple(obs.shape)}"
            )
        if not self._action_buffer:
            chunk = self.predict_action_chunk({OBS_STATE: window}, **kwargs)
            n_execute = max(1, min(self.config.n_action_steps, chunk.shape[1]))
            for i in range(n_execute - 1, -1, -1):
                self._action_buffer.append(chunk[0, i])
        return self._action_buffer.popleft()

    # -- rollout -------------------------------------------------------

    def _initial_cfm_losses(
        self,
        states_n: torch.Tensor,
        actions: torch.Tensor,
        tau: torch.Tensor,
        eps: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        cfg = self.config
        token_count = eps.shape[2]
        if cfg.use_dct_transform:
            spectral_action = extract_dct_coeffs(actions, M=token_count - 1)
        else:
            spectral_action = actions
        tau_bn, flatten_mc_noise, flatten_mc_action, flow_shape, num_mc = run_mc_simulation(
            spectral_action, eps, tau
        )
        x_t = _mc_x_t(tau_bn, flatten_mc_noise, flatten_mc_action, flow_shape)
        actor_tau = _actor_tau(tau_bn, flow_shape)
        actor = _unwrap_compiled(self.actor)
        conditioner = _unwrap_compiled(self.conditioner)
        with torch.inference_mode(), autocast_context(actions.device):
            condition = conditioner(states_n)
            condition_flat = einx.id(
                "n d -> n m d", condition, m=num_mc
            ).reshape(num_mc * condition.shape[0], condition.shape[-1])
            v_initial = actor(x_t, actor_tau, condition_flat).float()
        with torch.autocast(device_type=actions.device.type, enabled=False):
            v_star = (spectral_action[:, None] - eps).float()
            v_initial = v_initial.reshape(
                eps.shape[0], num_mc, *flow_shape
            )
            return (
                compute_cfm_loss(v_initial, v_star, cfg, "sample"),
                compute_cfm_loss(v_initial, v_star, cfg, "bin"),
            )

    def generate_rollout(
        self,
        env,
        total_steps: int,
        *,
        action_low: float = -1.0,
        action_high: float = 1.0,
        noisy: bool = True,
    ) -> dict[str, torch.Tensor]:
        """Roll out open-loop chunks; return a forward-ready batch."""
        if total_steps <= 0:
            raise ValueError(f"total_steps={total_steps} must be positive")
        cfg = self.config
        device = next(self.parameters()).device
        storage_dtype = resolve_storage_dtype(device)

        def propose(state: torch.Tensor) -> torch.Tensor:
            chunk = self.predict_action_chunk(
                {OBS_STATE: to_device_async(state, device)}, noise=noisy
            )[0]
            return chunk.cpu()

        def propose_batch(states: torch.Tensor) -> torch.Tensor:
            chunk = self.predict_action_chunk(
                {OBS_STATE: to_device_async(states, device)}, noise=noisy
            )
            return chunk.cpu()

        if self._rollout_h2d_event is not None:
            self._rollout_h2d_event.synchronize()
            self._rollout_h2d_event = None

        num_envs = int(getattr(env, "num_envs", 1))
        action_dim = int(self.config.action_feature.shape[0])
        obs_dim = int(self.config.robot_state_feature.shape[0])
        # A chunk may execute only one step before termination, so the
        # allocation-free upper bound is one chunk per requested step.
        max_chunks = max(1, total_steps)
        buffer_key = (
            max_chunks,
            total_steps,
            cfg.n_obs_steps,
            obs_dim,
            cfg.action_horizon,
            action_dim,
        )
        if (
            self._rollout_buffer is None
            or self._rollout_buffer.capacity_key() != buffer_key
        ):
            self._rollout_buffer = RolloutBuffer(
                max_chunks=max_chunks,
                max_steps=total_steps,
                obs_steps=cfg.n_obs_steps,
                obs_dim=obs_dim,
                action_horizon=cfg.action_horizon,
                action_dim=action_dim,
                pin_memory=device.type == "cuda",
            )
        buffer = self._rollout_buffer
        buffer.reset()

        if num_envs > 1:
            collect_chunks_vec(
                env,
                propose_batch,
                total_steps,
                action_low,
                action_high,
                n_obs_steps=cfg.n_obs_steps,
                n_action_steps=cfg.n_action_steps,
                chunk_size=cfg.action_horizon,
                buffer=buffer,
            )
        else:
            collect_chunks(
                env,
                propose,
                total_steps,
                action_low,
                action_high,
                n_obs_steps=cfg.n_obs_steps,
                n_action_steps=cfg.n_action_steps,
                chunk_size=cfg.action_horizon,
                buffer=buffer,
            )

        if device.type == "cuda":
            if (
                self._device_rollout_buffer is None
                or self._device_rollout_buffer.capacity_key()[:6] != buffer.capacity_key()[:6]
            ):
                self._device_rollout_buffer = RolloutBuffer(
                    max_chunks=max_chunks,
                    max_steps=total_steps,
                    obs_steps=cfg.n_obs_steps,
                    obs_dim=obs_dim,
                    action_horizon=cfg.action_horizon,
                    action_dim=action_dim,
                    device=device,
                )
            buffer.copy_to(self._device_rollout_buffer, non_blocking=True)
            self._rollout_h2d_event = torch.cuda.Event()
            self._rollout_h2d_event.record()
            active = self._device_rollout_buffer.active()
        else:
            active = buffer.active()
        n_chunks = buffer.n_chunks
        n_steps = buffer.n_steps
        num_spectral_tokens_kept = _num_spectral_tokens_kept(cfg)

        lengths = active["chunk_lengths"]
        chunk_starts = active["chunk_starts"]
        step_states = active["step_states"]
        step_next_states = active["step_next_states"]
        all_rewards = active["step_rewards"]
        actions = active["chunk_actions"]
        step_terminated = active["step_terminated"]
        step_truncated = active["step_truncated"]
        states = step_states.index_select(0, chunk_starts)

        with torch.inference_mode():
            # One batched critic pass for both sides of every executed step.
            all_step_states_n = self.preprocessor(
                {OBS_STATE: torch.cat([step_states, step_next_states], dim=0)}
            )[OBS_STATE]
            states_n = all_step_states_n.index_select(0, chunk_starts)
            with autocast_context(device):
                all_step_values = self.critic(all_step_states_n).float()
            total_executed = n_steps
            step_values = all_step_values[:total_executed]
            step_next_values = all_step_values[total_executed:]
            values = step_values.index_select(0, chunk_starts)

        step_advantages, step_returns = compute_step_advantages_torch(
            all_rewards,
            step_values,
            step_next_values,
            step_terminated,
            gamma=cfg.gamma,
            lam=cfg.gae_lambda,
            truncated=step_truncated,
        )
        # Normalize over every executed environment-step advantage, matching
        # rsl_rl/FPO++ storage semantics, before selecting chunk targets.
        if cfg.normalize_advantage and step_advantages.numel() > 0:
            step_advantages = (step_advantages - step_advantages.mean()) / (
                step_advantages.std() + 1e-8
            )
        if step_advantages.numel() > 0 and cfg.advantage_clamp is not None:
            step_advantages = step_advantages.clamp(
                -cfg.advantage_clamp, cfg.advantage_clamp
            )

        # The policy predicts one action chunk, so expose the first executed
        # step's step-GAE target as the chunk-level policy target.
        advantages = step_advantages.index_select(0, chunk_starts)
        returns = step_returns.index_select(0, chunk_starts)
        advantages = advantages.to(storage_dtype)
        returns = returns.to(storage_dtype)

        is_pad = torch.arange(cfg.action_horizon, device=device).unsqueeze(
            0
        ) >= lengths.unsqueeze(1)

        tau = sample_timesteps(
            (n_chunks, cfg.num_mc_samples, num_spectral_tokens_kept),
            cfg,
            device,
            None,
        ).unsqueeze(-1)
        eps = _sample_spectral_eps(cfg, n_chunks, action_dim, device)

        # Single shape check against the layout that will actually be used.
        expected_eps_shape = (n_chunks, cfg.num_mc_samples, num_spectral_tokens_kept, action_dim)
        if tuple(eps.shape) != expected_eps_shape:
            raise ValueError(
                f"MC noise shape {tuple(eps.shape)} != expected "
                f"{expected_eps_shape}"
            )

        initial_cfm_loss, initial_cfm_loss_bin = self._initial_cfm_losses(
            states_n, actions, tau, eps
        )

        return {
            OBS_STATE: states,
            ACTION: actions,
            "action_is_pad": is_pad,
            "advantages": advantages,
            "returns": returns,
            "values": values.detach(),
            # One raw post-step frame per executed environment step. The
            # training loop folds these into the live normalizer after the
            # update; keeping them separate avoids counting warmup/history
            # frames from the overlapping observation windows.
            "normalizer_observations": step_next_states[:, -1, :].detach(),
            # Independent per-chunk/MC CFM draws, reused for old/new losses.
            "mc_tau": tau,
            "mc_eps": eps,
            "initial_cfm_loss": initial_cfm_loss,
            "initial_cfm_loss_bin": initial_cfm_loss_bin,
        }

    # -- training forward ---------------------------------------------

    def forward(
        self,
        batch: dict[str, torch.Tensor],
        reduction: str = "mean",
        compute_info: bool = True,
        check_finite: bool = True,
    ) -> tuple[torch.Tensor, dict | None]:
        """Combined ASPO policy + value loss from a ``generate_rollout`` batch.

        When ``compute_info=False`` the diagnostic reductions are skipped
        entirely; the returned info is ``None``. Training runs without wandb
        should pass ``compute_info=False`` — the info pass is ~25 small
        reductions per minibatch that would otherwise be discarded.
        """
        cfg = self.config
        device = next(self.parameters()).device
        storage_dtype = resolve_storage_dtype(device)

        states = to_device_async(
            to_tensor(batch[OBS_STATE]), device
        ).to(dtype=storage_dtype)
        actions = to_device_async(to_tensor(batch[ACTION]), device).to(
            dtype=storage_dtype
        )
        advantages = to_tensor(batch["advantages"]).to(
            device=device, dtype=storage_dtype
        )
        returns = to_tensor(batch["returns"]).to(device=device, dtype=storage_dtype)
        action_is_pad = to_tensor(
            batch["action_is_pad"], dtype=torch.bool, device=device
        )
        sample_mask = None
        if "sample_mask" in batch:
            sample_mask = to_tensor(
                batch["sample_mask"], dtype=torch.bool, device=device
            )
            if sample_mask.shape != (states.shape[0],):
                raise ValueError(
                    f"sample_mask must have shape {(states.shape[0],)}, "
                    f"got {tuple(sample_mask.shape)}"
                )

        if states.shape[-2] != cfg.n_obs_steps:
            raise ValueError(
                f"state length {states.shape[-2]} != n_obs_steps {cfg.n_obs_steps}"
            )
        if actions.shape[-2] != cfg.action_horizon:
            raise ValueError(
                f"action horizon {actions.shape[-2]} != action_horizon {cfg.action_horizon}"
            )
        expected_action_mask_shape = (states.shape[0], cfg.action_horizon)
        if tuple(action_is_pad.shape) != expected_action_mask_shape:
            raise ValueError(
                f"action_is_pad shape {tuple(action_is_pad.shape)} != "
                f"{(states.shape[0], cfg.action_horizon)}"
            )

        # Treat each selected action chunk as one macro-action for the policy
        # loss. The unexecuted suffix is still part of the chunk selected by
        # the behavior policy, so it must contribute to the FPO ratio when
        # receding-horizon execution is enabled. ``action_is_pad`` records
        # execution bookkeeping; only chunks with no executed action are
        # invalid for the policy update.
        states = self.preprocessor({OBS_STATE: states})[OBS_STATE]

        num_spectral_tokens_kept = _num_spectral_tokens_kept(cfg)
        M_modes = num_spectral_tokens_kept - 1
        # Spectral encoding: first M+1 DCT bins (low frequency) kept; token count = M+1 (derived from top_p * H).
        n = states.shape[0]
        action_dim = actions.shape[-1]

        spectral_action_tokens = (
            extract_dct_coeffs(actions, M=M_modes) if cfg.use_dct_transform else actions
        )

        # Reuse the per-chunk/MC CFM draws stored during rollout.
        has_mc_tau = "mc_tau" in batch
        has_mc_eps = "mc_eps" in batch
        if has_mc_tau != has_mc_eps:
            raise ValueError("mc_tau and mc_eps must be provided together")
        if has_mc_tau:
            tau = to_tensor(batch["mc_tau"], dtype=storage_dtype, device=device)
            expected_tau = (n, cfg.num_mc_samples, num_spectral_tokens_kept, 1)
            if tuple(tau.shape) != expected_tau:
                raise ValueError(
                    f"mc_tau shape {tuple(tau.shape)} != {expected_tau}"
                )
            eps = to_tensor(batch["mc_eps"], dtype=storage_dtype, device=device)
            expected_eps = (n, cfg.num_mc_samples, num_spectral_tokens_kept, action_dim)
            if tuple(eps.shape) != expected_eps:
                raise ValueError(
                    f"stored MC noise shape {tuple(eps.shape)} != {expected_eps}"
                )
        else:
            tau = sample_timesteps(
                (n, cfg.num_mc_samples, num_spectral_tokens_kept),
                cfg,
                device,
                None,
            ).unsqueeze(-1)
            eps = _sample_spectral_eps(cfg, n, action_dim, device)

        # Noisy anchors.
        tau_bn, flatten_mc_noise, flatten_mc_action, flow_shape, num_mc = run_mc_simulation(
            spectral_action_tokens, eps, tau
        )
        x_t_flat = _mc_x_t(tau_bn, flatten_mc_noise, flatten_mc_action, flow_shape)
        actor_tau = _actor_tau(tau_bn, flow_shape)

        # Condition fusion: [n, D] -> [n*m, D].
        with autocast_context(device):
            cond_flat = einx.id(
                "n m d -> (n m) d",
                einx.id("n d -> n m d", self.conditioner(states), m=num_mc),
            )
        stored_l_old = "initial_cfm_loss" in batch
        if stored_l_old:
            l_old = to_tensor(
                batch["initial_cfm_loss"], dtype=torch.float32, device=device
            )
            l_old_bin = to_tensor(
                batch["initial_cfm_loss_bin"], dtype=torch.float32, device=device
            )
            if tuple(l_old.shape) != (n, num_mc):
                raise ValueError(
                    f"initial_cfm_loss shape {tuple(l_old.shape)} != {(n, num_mc)}"
                )
            if tuple(l_old_bin.shape) != (n, num_mc, flow_shape[0]):
                raise ValueError(
                    f"initial_cfm_loss_bin shape {tuple(l_old_bin.shape)} != "
                    f"{(n, num_mc, flow_shape[0])}"
                )
        else:
            with torch.inference_mode(), autocast_context(device):
                old_cond_flat = einx.id(
                    "n m d -> (n m) d",
                    einx.id("n d -> n m d", self.old_conditioner(states), m=num_mc),
                )
                v_old = self.old_actor(x_t_flat, actor_tau, old_cond_flat).float()

        with autocast_context(device):
            if cfg.use_gradient_checkpointing:
                v_new = _actor_call(
                    self.actor, x_t_flat, actor_tau, cond_flat, use_ckpt=True
                )
            else:
                v_new = self.actor(x_t_flat, actor_tau, cond_flat)
            v_new = v_new.float()

        cond_stats = None
        if compute_info:
            # Re-run just the modulation MLPs on one row to grab stats —
            # cheap relative to attention, and only on info-requesting steps.
            with torch.inference_mode(), autocast_context(device):
                _, cond_stats = self.actor(
                    x_t_flat[:1], actor_tau[:1], cond_flat[:1], return_stats=True
                )

        # CFM losses under target velocities (fp32 for stability).
        with torch.autocast(device_type=device.type, enabled=False):
            v_star_f = self.compute_target_velocity(spectral_action_tokens, eps).float()
            v_new_r_f = v_new.reshape(n, num_mc, *flow_shape)

            if not stored_l_old:
                v_old_r_f = v_old.reshape(n, num_mc, *flow_shape)
                l_old_bin = compute_cfm_loss(v_old_r_f, v_star_f, cfg, "bin")
                l_old = compute_cfm_loss(v_old_r_f, v_star_f, cfg, "sample")
            adv_mc = advantages.float().unsqueeze(1)
            l_new_bin = compute_cfm_loss(v_new_r_f, v_star_f, cfg, "bin")
            rho_bin = compute_policy_ratio(l_old_bin, l_new_bin, cfg)  # diagnostics only

            l_new = compute_cfm_loss(v_new_r_f, v_star_f, cfg, "sample")
            rho = compute_policy_ratio(l_old, l_new, cfg, advantage=adv_mc)
            self.cfm_above_clamp_frac = (l_new > cfg.cfm_clamp).float().mean().detach()

            psi = compute_aspo_surrogate(rho, adv_mc, cfg, cfg.epsilon_clip)
            policy_per_chunk = -psi.mean(dim=1)

        # Value loss (official FPO++ uses mean squared error).
        with autocast_context(device):
            v_pred = self.critic(states).float()
        returns_d = returns.reshape(-1)
        if v_pred.shape != returns_d.shape:
            raise ValueError(
                f"value/return shape mismatch: v_pred={tuple(v_pred.shape)}, "
                f"returns_d={tuple(returns_d.shape)}"
            )
        se = torch.square(v_pred - returns_d)
        if cfg.use_clipped_value_loss and "values" in batch:
            old_values = to_tensor(
                batch["values"], dtype=storage_dtype, device=device
            )
            v_clipped = old_values + (v_pred - old_values).clamp(
                -cfg.epsilon_clip, cfg.epsilon_clip
            )
            clipped_se = torch.square(v_clipped - returns_d)
            value_per_chunk = torch.maximum(se, clipped_se)
        else:
            value_per_chunk = se

        if reduction == "mean":
            if sample_mask is None:
                policy_loss = policy_per_chunk.mean()
                value_loss = value_per_chunk.mean()
            else:
                weight = sample_mask.to(policy_per_chunk.dtype)
                count = weight.sum().clamp_min(1)
                policy_loss = (policy_per_chunk * weight).sum() / count
                value_loss = (value_per_chunk * weight).sum() / count
            loss = policy_loss + cfg.vf_coef * value_loss
        elif reduction == "none":
            if sample_mask is None:
                loss = policy_per_chunk + cfg.vf_coef * value_per_chunk
            else:
                weight = sample_mask.to(policy_per_chunk.dtype)
                loss = weight * (
                    policy_per_chunk + cfg.vf_coef * value_per_chunk
                )
        else:
            raise ValueError(f"reduction={reduction!r} must be 'mean' or 'none'")

        if check_finite and not bool(torch.isfinite(loss).all()):
            raise SJFPOAborted(
                f"Non-finite loss: "
                f"policy_loss={float(policy_per_chunk.mean().detach())}, "
                f"value_loss={float(value_per_chunk.mean().detach())}"
            )

        if not compute_info:
            return loss, None

        info = self._compute_info(
            l_old=l_old,
            l_new=l_new,
            rho=rho,
            rho_bin=rho_bin,
            v_star_f=v_star_f,
            v_pred=v_pred,
            returns_d=returns_d,
            policy_per_chunk=policy_per_chunk,
            value_per_chunk=value_per_chunk,
            advantages=advantages,
            cond_stats=cond_stats,
            cfg=cfg,
        )
        return loss, info

    # -- diagnostics (single device->host transfer) --------------------

    @torch.inference_mode()
    def _compute_info(
        self,
        *,
        l_old,
        l_new,
        rho,
        rho_bin,
        v_star_f,
        v_pred,
        returns_d,
        policy_per_chunk,
        value_per_chunk,
        advantages,
        cond_stats,
        cfg,
    ) -> dict[str, float]:
        """All diagnostic reductions in one go, batched into a single sync.

        Only called when ``forward(compute_info=True)``; the training loop
        can skip it entirely when nothing will consume the result.
        """
        # Log-ratio on raw delta (never clamped-then-checked).
        delta = l_old - l_new
        log_rho_abs = delta.abs()
        log_rho_max = log_rho_abs.max()
        log_rho_clip_frac = (log_rho_abs > cfg.log_ratio_cap).float().mean()
        # Fraction of samples whose CFM loss exceeds cfm_clamp. Those samples
        # contribute no gradient at all, so a run where this stays high is
        # training the critic only.
        cfm_above_clamp_frac = (l_new > cfg.cfm_clamp).float().mean()

        # rho stats
        rho_mean = rho.mean()
        rho_std = rho.std(correction=0)

        # per-bin rho
        rho_bin_mean = rho_bin.mean(dim=(0, 1))
        n_bins = rho_bin_mean.numel()
        split = max(1, n_bins // 2)
        rho_bin_range = rho_bin_mean.max() - rho_bin_mean.min()
        rho_bin_low = rho_bin_mean[:split].mean()
        rho_bin_high = (
            rho_bin_mean[split:].mean() if n_bins > 1 else rho_bin_mean.mean()
        )
        rho_bin_outside = (
            ((rho_bin < 1.0 - cfg.epsilon_clip) | (rho_bin > 1.0 + cfg.epsilon_clip))
            .float()
            .mean()
        )
        eps_low = cfg.epsilon_clip_lowfreq if cfg.use_per_bin_clip else cfg.epsilon_clip
        eps_high = (
            cfg.epsilon_clip_highfreq if cfg.use_per_bin_clip else cfg.epsilon_clip
        )
        low_slice = rho_bin[..., :split]
        rho_bin_low_outside = (
            ((low_slice < 1.0 - eps_low) | (low_slice > 1.0 + eps_low)).float().mean()
        )
        if n_bins > 1:
            high_slice = rho_bin[..., split:]
            rho_bin_high_outside = (
                ((high_slice < 1.0 - eps_high) | (high_slice > 1.0 + eps_high))
                .float()
                .mean()
            )
        else:
            rho_bin_high_outside = rho_bin_low_outside

        # value-health stats
        returns_var = returns_d.var(correction=0).clamp_min(1e-8)
        explained_var = 1.0 - (returns_d - v_pred).var(correction=0) / returns_var
        value_error_max = torch.nn.functional.smooth_l1_loss(
            v_pred, returns_d, reduction="none", beta=1.0
        ).max()
        value_pred_max = v_pred.abs().max()
        return_max = returns_d.abs().max()
        v_star_rms = v_star_f.pow(2).mean().sqrt()

        # Concrete diagnostics (Step 3)
        l_old_std = l_old.std(correction=0)
        l_new_std = l_new.std(correction=0)
        corr_denom = l_old_std * l_new_std + 1e-8
        l_old_new_corr = (
            (l_old - l_old.mean()) * (l_new - l_new.mean())
        ).mean() / corr_denom

        # Pack into one tensor and transfer once.
        cond_keys = list(cond_stats.keys()) if cond_stats else []
        scalars = [
            policy_per_chunk.mean(),
            value_per_chunk.mean(),
            explained_var,
            delta.mean(),
            log_rho_max,
            log_rho_clip_frac,
            value_error_max,
            value_pred_max,
            return_max,
            rho_mean,
            rho_std,
            rho_bin_range,
            rho_bin_low,
            rho_bin_high,
            rho_bin_outside,
            rho_bin_low_outside,
            rho_bin_high_outside,
            v_star_rms,
            l_old.mean(),
            l_new.mean(),
            l_old_std,
            l_new_std,
            cfm_above_clamp_frac,
            l_old_new_corr,
            advantages.abs().mean(),
            returns_d.std(correction=0),
            v_pred.std(correction=0),
        ]
        scalars.extend(
            cond_stats[k].detach()
            if torch.is_tensor(cond_stats[k])
            else torch.tensor(float(cond_stats[k]), device=delta.device)
            for k in cond_keys
        )
        packed = torch.stack([s.detach().reshape(()) for s in scalars]).float().cpu()
        vals = packed.tolist()

        keys = [
            "policy_loss",
            "value_loss",
            "explained_var",
            "mean_log_ratio",
            "log_rho_max",
            "log_rho_clip_frac",
            "value_error_max",
            "value_pred_max",
            "return_max",
            "flow_rho_mean",
            "flow_rho_std",
            "flow_rho_bin_range",
            "flow_rho_bin_lowfreq",
            "flow_rho_bin_highfreq",
            "flow_rho_bin_outside_frac",
            "flow_rho_bin_low_outside_frac",
            "flow_rho_bin_high_outside_frac",
            "flow_v_star_rms",
            "l_old_mean",
            "l_new_mean",
            "l_old_std",
            "l_new_std",
            "cfm_above_clamp_frac",
            "l_old_new_corr",
            "adv_mean_abs",
            "value_target_std",
            "value_pred_std",
        ]
        info = dict(zip(keys, vals))
        for i, k in enumerate(cond_keys):
            info[f"cond_{k}"] = vals[len(keys) + i]
        return info

In [ ]:
%%writefile src/sjfpo/networks.py
"""Networks: window encoders, state conditioner (raw + spectral), flow heads, V(s) critic.

Changes in this revision (addresses review feedback):
* ``RunningMeanStd.forward`` pins the output dtype to the input dtype, so
  half-precision callers don't get silently promoted to fp32.
* ``RunningMeanStd.count`` is fp64 (avoids fp32 saturation at 2^24 frames);
  mean/var stay fp32.
* ``RunningMeanStd._uninitialized`` is registered with ``persistent=False``
  so pre-change checkpoints still load.
* Added ``RunningMeanStd.reset()`` to clear the one-way latch.
* Added a shape assertion in ``RunningMeanStd.update``.
* ``_RMSNorm`` uses the ``dtype=`` kwarg at construction instead of mutating
  ``.data`` (meta-device / ``to_empty`` friendly).
* ``condition.to(...)`` now moves device as well as dtype.
* ``temb`` broadcast replaces the explicit ``expand`` (scalar ``t`` runs the
  time MLP once, not B times).
* ``SelfAttention`` accepts an explicit ``[B, L]`` key-padding mask and
  validates its shape before constructing the SDPA mask.
* ``StateConditioner`` uses a per-frame raw-state MLP plus a temporal DCT
  branch; it does not instantiate an inner ``WindowEncoder``.
* Type hints on the dual-return forwards.

Performance notes (carried from the previous revision):
* Attention uses ``jax.nn.dot_product_attention``-equivalent SDPA on CUDA.
* Sinusoidal frequency tables are cached per-module per-device.
* ``SwiGLU`` fuses gate and up into a single GEMM.
* ``DiTBlock`` / ``SJFPOFlowModel`` return diagnostics only when
  ``return_stats=True`` — the default training path has no extra reductions.
"""

from __future__ import annotations

import math
from collections import OrderedDict

import torch
import torch.nn.functional as F
from torch import nn

from sjfpo.dct_utils import dct

# ---------------------------------------------------------------------------
# Norm
# ---------------------------------------------------------------------------


class _RMSNorm(nn.RMSNorm):
    """RMSNorm computed in FP32 with the input dtype preserved on output.

    The normalization parameters are initialized in FP32. Activations may be
    FP16/BF16 under autocast, but the result is always cast back to the input
    activation dtype so normalization does not change the model-wide policy.
    """

    def __init__(self, hidden_dim: int, eps: float = 1e-6):
        super().__init__(hidden_dim, eps=eps, dtype=torch.float32)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_dtype = x.dtype
        with torch.autocast(device_type=x.device.type, enabled=False):
            # Calling F.rms_norm directly also handles callers that converted
            # the module to a lower parameter dtype with ``module.to``.
            y = F.rms_norm(
                x.to(dtype=torch.float32),
                self.normalized_shape,
                self.weight.to(dtype=torch.float32),
                self.eps,
            )
        return y.to(dtype=input_dtype)


def _rms_norm(hidden_dim: int, eps: float = 1e-6) -> nn.RMSNorm:
    return _RMSNorm(hidden_dim, eps=eps)


# ---------------------------------------------------------------------------
# Primitives
# ---------------------------------------------------------------------------


class SwiGLU(nn.Module):
    """Gated SiLU feedforward: down(silu(gate(x)) * up(x)).

    Gate and up are fused into a single ``gate_up`` projection (one GEMM
    instead of two); torch's weight layout is ``[out, in]`` so the fused
    weight is ``cat([gate.weight, up.weight], dim=0)``.
    """

    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int):
        super().__init__()
        if in_dim < 1 or hidden_dim < 1 or out_dim < 1:
            raise ValueError("SwiGLU dimensions must be positive")
        self.gate_up = nn.Linear(in_dim, 2 * hidden_dim)
        self.down = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        g, u = self.gate_up(x).chunk(2, dim=-1)
        return self.down(F.silu(g) * u)


class CondMLP(nn.Module):
    """SiLU MLP for conditioning regression."""

    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int):
        super().__init__()
        if in_dim < 1 or hidden_dim < 1 or out_dim < 1:
            raise ValueError("CondMLP dimensions must be positive")
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(F.silu(self.fc1(x)))


class SinusoidalEncoding(nn.Module):
    """Fixed sinusoidal codes with cached frequency tables.

    Cache is a small per-module LRU keyed by ``torch.device``. Under
    ``torch.compile`` the first miss can still cause a graph break; callers
    that compile this module should warm the cache or move the table to a
    non-persistent buffer in a future optimization.
    """

    def __init__(self, dim: int, scale: float = 1.0):
        super().__init__()
        if dim < 2:
            raise ValueError(f"SinusoidalEncoding dim must be >= 2, got {dim}")
        self.dim = dim
        self.scale = 1.0 if scale is None else float(scale)
        self.half = dim // 2
        self._freq_cache: OrderedDict[torch.device, torch.Tensor] = OrderedDict()

    def _freqs(self, device: torch.device) -> torch.Tensor:
        device = torch.device(device)
        cached = self._freq_cache.get(device)
        if cached is not None:
            self._freq_cache.move_to_end(device)
            return cached
        i = torch.arange(self.half, device=device, dtype=torch.float32)
        cached = torch.exp(-math.log(10000.0) * i / max(self.half - 1, 1))
        self._freq_cache[device] = cached
        if len(self._freq_cache) > 4:
            self._freq_cache.popitem(last=False)
        return cached

    def clear_cache(self) -> None:
        """Drop cached device-specific frequency tables."""
        self._freq_cache.clear()

    def add_to(self, x: torch.Tensor, positions: torch.Tensor) -> torch.Tensor:
        """Add codes for ``positions`` ([..., L]) to ``x`` ([..., L, D])."""
        ang = (positions.to(dtype=torch.float32) * self.scale).unsqueeze(
            -1
        ) * self._freqs(x.device)
        pe = torch.cat([ang.sin(), ang.cos()], dim=-1)
        if pe.shape[-1] < self.dim:  # odd dim: pad last column with zeros
            pe = F.pad(pe, (0, self.dim - pe.shape[-1]))
        return x + pe.to(dtype=x.dtype)

    def encode_times(self, t: torch.Tensor) -> torch.Tensor:
        """Sinusoidal embedding for continuous timestep values."""
        if not isinstance(t, torch.Tensor):
            t = torch.as_tensor(t, dtype=torch.float32)
        t = t.to(dtype=torch.float32) * self.scale
        ang = t.unsqueeze(-1) * self._freqs(t.device)
        emb = torch.cat([ang.sin(), ang.cos()], dim=-1)
        if self.dim % 2:
            emb = F.pad(emb, (0, 1))
        return emb

    def positions(self, length: int, device) -> torch.Tensor:
        """0..L-1 index vector for slot/frame codes."""
        return torch.arange(length, dtype=torch.float32, device=device)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add position codes by slot index: [..., L, D] -> [..., L, D]."""
        return self.add_to(x, self.positions(x.shape[-2], x.device))


class SelfAttention(nn.Module):
    """Multi-head self-attention on SDPA.

    ``attention_mask`` is a key-padding mask with shape ``[B, L]``. A
    True/1 entry marks a valid key and False/0 marks an ignored key. Every
    batch item is forced to retain key 0 when its mask is otherwise empty so
    SDPA cannot produce NaNs; arbitrary query/key masks are not supported.
    """

    def __init__(self, hidden_dim: int, n_heads: int):
        super().__init__()
        if hidden_dim < 1:
            raise ValueError(f"hidden_dim must be positive, got {hidden_dim}")
        if n_heads < 1:
            raise ValueError(f"n_heads must be positive, got {n_heads}")
        if hidden_dim % n_heads != 0:
            raise ValueError(f"hidden_dim={hidden_dim} must split over {n_heads} heads")
        self.n_heads = n_heads
        self.head_dim = hidden_dim // n_heads
        self.qkv = nn.Linear(hidden_dim, 3 * hidden_dim)
        self.out = nn.Linear(hidden_dim, hidden_dim)

    def forward(
        self,
        x: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """[B, L, D] -> [B, L, D].

        ``attention_mask`` is a boolean/0-1 key-padding mask with shape
        ``[B, L]``; 1/True is valid and 0/False is ignored.
        """
        if x.dim() != 3:
            raise ValueError(f"x must have shape [B, L, D], got {tuple(x.shape)}")
        b, l, _ = x.shape
        if b < 1 or l < 1:
            raise ValueError(
                f"x must have positive batch and sequence sizes, got {(b, l)}"
            )
        h = self.n_heads
        d = self.head_dim

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        # [B, L, H*D] -> [B, H, L, D] via view+transpose.
        q = q.view(b, l, h, d).transpose(1, 2)
        k = k.view(b, l, h, d).transpose(1, 2)
        v = v.view(b, l, h, d).transpose(1, 2)

        mask = None
        if attention_mask is not None:
            # Hugging Face convention: 1/True is a valid key; 0/False is
            # ignored. PyTorch SDPA boolean masks use True to allow attention.
            keep = torch.as_tensor(
                attention_mask, dtype=torch.bool, device=x.device
            ).clone()
            if keep.shape != (b, l):
                raise ValueError(
                    f"attention_mask must have shape [B, L] = {(b, l)}, "
                    f"got {tuple(keep.shape)}"
                )
            # Ensure every batch item has one valid key to avoid SDPA NaNs.
            keep[:, 0] = True
            mask = keep[:, None, None, :]

        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        y = y.transpose(1, 2).reshape(b, l, h * d)
        return self.out(y)


class WindowEncoder(nn.Module):
    """Pre-norm residual block over a time window."""

    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        n_heads: int,
        mlp_ratio: float,
    ):
        super().__init__()
        if in_dim < 1 or hidden_dim < 2:
            raise ValueError("WindowEncoder dimensions must be positive")
        if (
            not math.isfinite(mlp_ratio)
            or mlp_ratio <= 0
            or int(hidden_dim * mlp_ratio) < 1
        ):
            raise ValueError("mlp_ratio must produce a positive hidden width")
        self.frame_proj = nn.Linear(in_dim, hidden_dim)
        self.pos_encoding = SinusoidalEncoding(hidden_dim)
        self.norm1 = _rms_norm(hidden_dim, eps=1e-6)
        self.attn1 = SelfAttention(hidden_dim, n_heads)
        self.norm2 = _rms_norm(hidden_dim, eps=1e-6)
        mlp_hidden = int(hidden_dim * mlp_ratio)
        self.mlp1 = SwiGLU(hidden_dim, mlp_hidden, hidden_dim)

    def forward(
        self,
        x: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """[B, L, in_dim] -> [B, L, hidden_dim].

        ``attention_mask`` is a boolean/0-1 key-padding mask with shape
        ``[B, L]``; 1/True is valid and 0/False is ignored.
        """
        h = self.pos_encoding(self.frame_proj(x))
        h = h + self.attn1(self.norm1(h), attention_mask)
        return h + self.mlp1(self.norm2(h))


# ---------------------------------------------------------------------------
# Running stats
# ---------------------------------------------------------------------------


class RunningMeanStd(nn.Module):
    """Per-dim running mean/variance over raw obs frames.

    ``mean``/``var`` stay fp32; ``count`` is fp64 so it doesn't saturate at
    2^24 frames on long runs. ``forward`` preserves the input dtype so
    half-precision callers don't get silently promoted.
    """

    def __init__(self, dim: int, eps: float = 1e-8, clip: float = 10.0):
        super().__init__()
        if dim <= 0:
            raise ValueError(f"dim={dim} must be positive")
        self.register_buffer("mean", torch.zeros(dim, dtype=torch.float32))
        self.register_buffer("var", torch.ones(dim, dtype=torch.float32))
        self.register_buffer("count", torch.tensor(0.0, dtype=torch.float64))
        # Non-persistent so pre-change checkpoints load without this key.
        self.register_buffer(
            "_uninitialized", torch.tensor(True, dtype=torch.bool), persistent=False
        )
        self.eps = eps
        self.clip = clip

    @torch.inference_mode()
    def update(self, x: torch.Tensor) -> None:
        """Fold raw frames into the running stats (any device, no grad)."""
        if x.shape[-1] != self.mean.numel():
            raise ValueError(
                f"update got last dim {x.shape[-1]}, expected {self.mean.numel()}"
            )
        batch = (
            x.detach()
            .reshape(-1, x.shape[-1])
            .to(self.mean.device, dtype=torch.float32)
        )
        b = batch.shape[0]
        if b == 0:
            return
        b_mean = batch.mean(dim=0)
        b_var = batch.var(dim=0, unbiased=False)
        delta = b_mean - self.mean
        total = self.count + b  # fp64
        self.mean.add_((delta * b / total).to(self.mean.dtype))
        # Chan et al. parallel variance merge; result cast back to fp32 by copy_.
        self.var.copy_(
            (
                self.count * self.var
                + b * b_var
                + delta.square() * (self.count * b / total)
            )
            / total
        )
        self.count.copy_(total)
        self._uninitialized.fill_(False)

    @torch.inference_mode()
    def reset(self) -> None:
        """Clear the one-way latch and reset stats to their initial values."""
        self.mean.zero_()
        self.var.fill_(1.0)
        self.count.zero_()
        self._uninitialized.fill_(True)

    def _load_from_state_dict(self, *args, **kwargs):
        super()._load_from_state_dict(*args, **kwargs)
        self._uninitialized.copy_(self.count.eq(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Normalize with frozen stats; identity until the first update."""
        norm = (x - self.mean.to(x.device)) * torch.rsqrt(
            self.var.to(x.device) + self.eps
        )
        norm = norm.clamp(-self.clip, self.clip).to(x.dtype)
        return torch.where(self._uninitialized.to(x.device), x, norm)


# ---------------------------------------------------------------------------
# DiT
# ---------------------------------------------------------------------------


def modulate(x: torch.Tensor, shift: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
    """adaLN modulation: x * (1 + scale) + shift."""
    return x * (1.0 + scale) + shift


class DiTBlock(nn.Module):
    """Pre-norm DiT block with adaLN-Zero conditioning.

    ``return_stats=True`` returns ``(out, stats_dict)``; the default path
    returns the tensor only, so no diagnostic reductions run in training.
    """

    def __init__(self, hidden_dim: int, n_heads: int, mlp_ratio: float):
        super().__init__()
        if hidden_dim < 2:
            raise ValueError("hidden_dim must be at least 2")
        if (
            not math.isfinite(mlp_ratio)
            or mlp_ratio <= 0
            or int(hidden_dim * mlp_ratio) < 1
        ):
            raise ValueError("mlp_ratio must produce a positive hidden width")
        self.norm1 = _rms_norm(hidden_dim)
        self.attn = SelfAttention(hidden_dim, n_heads)
        self.norm2 = _rms_norm(hidden_dim)
        mlp_hidden = int(hidden_dim * mlp_ratio)
        self.mlp = SwiGLU(hidden_dim, mlp_hidden, hidden_dim)
        self.adaLN = CondMLP(hidden_dim, hidden_dim, 6 * hidden_dim)
        # Zero-init modulation head: block starts as identity (adaLN-Zero).
        nn.init.constant_(self.adaLN.fc2.weight, 0.0)
        nn.init.constant_(self.adaLN.fc2.bias, 0.0)

    def forward(
        self, x: torch.Tensor, c: torch.Tensor, return_stats: bool = False
    ) -> torch.Tensor | tuple[torch.Tensor, dict[str, torch.Tensor]]:
        """Run DiT over ``[B,K,D]`` or coordinate-aware ``[B,K,D_a,H]``."""
        coordinate_axis = x.dim() == 4
        token_axis = not coordinate_axis and c.dim() == 3
        b = c.shape[0]
        if coordinate_axis:
            _, k, action_axis, _ = x.shape
            parts = self.adaLN(c).view(b, k, action_axis, 6, -1)
            (
                shift_msa,
                scale_msa,
                gate_msa,
                shift_mlp,
                scale_mlp,
                gate_mlp,
            ) = parts.unbind(dim=3)
            h = modulate(self.norm1(x), shift_msa, scale_msa)
            # Attention remains over the K action tokens, independently per
            # action dimension; only the batch/attention axis is reshaped.
            h_attn = h.permute(0, 2, 1, 3).reshape(
                b * action_axis, k, h.shape[-1]
            )
            attn_out = self.attn(h_attn)
            attn_out = attn_out.reshape(b, action_axis, k, -1).permute(0, 2, 1, 3)
            x = x + gate_msa * attn_out
            out = x + gate_mlp * self.mlp(
                modulate(self.norm2(x), shift_mlp, scale_mlp)
            )
        elif token_axis:
            _, k, _ = x.shape
            parts = self.adaLN(c).view(b, k, 6, -1)
            (
                shift_msa,
                scale_msa,
                gate_msa,
                shift_mlp,
                scale_mlp,
                gate_mlp,
            ) = parts.unbind(dim=2)
            h = modulate(self.norm1(x), shift_msa, scale_msa)
            x = x + gate_msa * self.attn(h)
            out = x + gate_mlp * self.mlp(
                modulate(self.norm2(x), shift_mlp, scale_mlp)
            )
        else:
            # reshape+unbind avoids building a [B, 1, 6H] intermediate.
            parts = self.adaLN(c).view(b, 6, -1)
            shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = parts.unbind(
                dim=1
            )
            h = modulate(self.norm1(x), shift_msa.unsqueeze(1), scale_msa.unsqueeze(1))
            x = x + gate_msa.unsqueeze(1) * self.attn(h)
            out = x + gate_mlp.unsqueeze(1) * self.mlp(
                modulate(self.norm2(x), shift_mlp.unsqueeze(1), scale_mlp.unsqueeze(1))
            )

        if not return_stats:
            return out

        with torch.inference_mode():
            stats = {
                "gate_msa": gate_msa.square().mean().sqrt(),
                "gate_mlp": gate_mlp.square().mean().sqrt(),
                "scale_msa": (1.0 + scale_msa).square().mean().sqrt(),
                "scale_mlp": (1.0 + scale_mlp).square().mean().sqrt(),
            }
        return out, stats


class StateConditioner(nn.Module):
    """Fuse a mean-pooled per-frame state summary with DCT-bin features.

    The raw path uses a per-frame MLP + mean-pool rather than an attention
    block: the spectral branch already encodes temporal structure explicitly,
    so the attention block is redundant for the chunk-level conditioning
    vector (the caller mean-pools the result anyway). ``bin_norm`` normalizes
    each DCT frequency bin across the observation-feature dimension before
    positional encoding and projection.
    """

    def __init__(self, obs_dim: int, hidden_dim: int):
        super().__init__()
        if obs_dim < 2:
            raise ValueError("StateConditioner obs_dim must be at least 2")
        if hidden_dim < 2:
            raise ValueError("hidden_dim must be at least 2")
        self.obs_dim = obs_dim
        # Raw path: per-frame MLP then mean-pool (replaces full attention window).
        self.frame_proj = nn.Linear(obs_dim, hidden_dim)
        self.frame_mlp = SwiGLU(hidden_dim, 4 * hidden_dim, hidden_dim)
        # Spectral path: unchanged.
        self.bin_norm = _rms_norm(obs_dim)
        self.bin_encoding = SinusoidalEncoding(obs_dim)
        self.key_proj = nn.Linear(obs_dim, hidden_dim)
        self.fuse = nn.Linear(2 * hidden_dim, hidden_dim)

    def forward(self, states: torch.Tensor) -> torch.Tensor:
        """[B, T, obs] -> [B, D]."""
        if states.dim() != 3:
            raise ValueError(
                f"states must have shape [B, T, obs], got {tuple(states.shape)}"
            )
        if states.shape[0] < 1 or states.shape[1] < 1:
            raise ValueError("states must have positive batch and time dimensions")
        if states.shape[2] != self.obs_dim:
            raise ValueError(
                f"states feature dimension must be {self.obs_dim}, got {states.shape[2]}"
            )
        raw = self.frame_mlp(self.frame_proj(states)).mean(dim=1)
        spec = self.key_proj(self.bin_encoding(self.bin_norm(dct(states, dim=1)))).mean(
            dim=1
        )
        return self.fuse(torch.cat([raw, spec], dim=-1))


class SJFPOFlowModel(nn.Module):
    """DiT velocity model over noisy-action tokens, conditioned via adaLN.

    ``return_stats=True`` returns ``(velocity, stats_dict)`` and computes
    per-block gate/scale RMS reductions; the default path returns only the
    velocity, with no diagnostic reductions in the graph.
    """

    def __init__(
        self,
        action_dim: int,
        action_horizon: int,
        hidden_dim: int,
        dit_depth: int,
        dit_heads: int,
        dit_mlp_ratio: float,
        time_embed_dim: int,
        timestep_scale: float,
    ):
        super().__init__()
        if action_dim < 1 or hidden_dim < 2:
            raise ValueError("action_dim must be positive and hidden_dim at least 2")
        if action_horizon < 1:
            raise ValueError("action_horizon must be positive")
        if dit_depth < 1 or dit_heads < 1 or time_embed_dim < 2:
            raise ValueError("dit_depth, dit_heads, and time_embed_dim must be positive")
        if (
            not math.isfinite(dit_mlp_ratio)
            or dit_mlp_ratio <= 0
            or int(hidden_dim * dit_mlp_ratio) < 1
        ):
            raise ValueError("dit_mlp_ratio must produce a positive hidden width")
        if not math.isfinite(timestep_scale) or timestep_scale <= 0:
            raise ValueError("timestep_scale must be finite and positive")
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        self.action_horizon = action_horizon
        self.time_embed_dim = time_embed_dim
        self.timestep_embedder = SinusoidalEncoding(
            self.time_embed_dim, scale=timestep_scale
        )
        self.in_proj = nn.Linear(action_dim, hidden_dim)
        # Learned positional embedding with one row per configured action
        # position. Reduced DCT token counts slice this table to their length.
        self.token_pos = nn.Parameter(torch.zeros(1, action_horizon, hidden_dim))
        nn.init.normal_(self.token_pos, std=0.02)
        self.time_mlp = CondMLP(time_embed_dim, 4 * hidden_dim, hidden_dim)
        self.blocks = nn.ModuleList(
            DiTBlock(hidden_dim, dit_heads, dit_mlp_ratio) for _ in range(dit_depth)
        )
        self.norm_out = _rms_norm(hidden_dim)
        self.adaLN_out = nn.Linear(hidden_dim, 2 * hidden_dim)
        nn.init.constant_(self.adaLN_out.weight, 0.0)
        nn.init.constant_(self.adaLN_out.bias, 0.0)
        self.out = nn.Linear(hidden_dim, action_dim)
        # Zero final layer: untrained policy predicts ~zero velocity.
        nn.init.constant_(self.out.weight, 0.0)
        nn.init.constant_(self.out.bias, 0.0)

    def forward(
        self,
        x_t: torch.Tensor,
        t: torch.Tensor | float,
        condition: torch.Tensor,
        return_stats: bool = False,
    ) -> torch.Tensor | tuple[torch.Tensor, dict[str, torch.Tensor]]:
        """Predict velocity from token actions and shared-per-token times.

        ``t`` is either a scalar, ``[B]``, or ``[B, K, 1]``. The per-token
        form gives each action token its own time, shared across its action
        dimensions.
        """
        if x_t.dim() != 3:
            raise ValueError(f"x_t must have shape [B, K, D], got {tuple(x_t.shape)}")
        if x_t.shape[0] < 1 or x_t.shape[1] < 1:
            raise ValueError("x_t must have positive batch and token dimensions")
        if x_t.shape[2] != self.action_dim:
            raise ValueError(
                f"x_t feature dimension must be {self.action_dim}, got {x_t.shape[2]}"
            )
        if condition.dim() != 2 or condition.shape[0] != x_t.shape[0]:
            raise ValueError(
                f"condition must have shape [{x_t.shape[0]}, {self.hidden_dim}], "
                f"got {tuple(condition.shape)}"
            )
        if condition.shape[1] != self.hidden_dim:
            raise ValueError(
                f"condition feature dimension must be {self.hidden_dim}, "
                f"got {condition.shape[1]}"
            )
        if x_t.shape[1] > self.action_horizon:
            raise ValueError(
                f"action token count {x_t.shape[1]} exceeds action_horizon "
                f"{self.action_horizon}"
            )

        t = torch.as_tensor(t, dtype=torch.float32, device=x_t.device)
        token_axis = t.dim() in (2, 3) and tuple(t.shape[:2]) == tuple(x_t.shape[:2])
        if token_axis:
            if t.dim() == 3:
                if t.shape[-1] != 1:
                    raise ValueError(
                        f"t must end in a singleton action-dimension axis, got {tuple(t.shape)}"
                    )
                t = t.squeeze(-1)
            temb = self.timestep_embedder.encode_times(t.reshape(-1))
            temb = temb.reshape(x_t.shape[0], x_t.shape[1], -1).to(
                device=x_t.device, dtype=x_t.dtype
            )
            c = self.time_mlp(temb) + condition.to(
                device=x_t.device, dtype=x_t.dtype
            )[:, None, :]
            token_pos = self.token_pos[:, : x_t.shape[1], :].to(
                device=x_t.device, dtype=x_t.dtype
            )
            h = self.in_proj(x_t) + token_pos
        elif t.dim() <= 1:
            t = t.reshape(-1)
            if t.numel() not in (1, x_t.shape[0]):
                raise ValueError(
                    f"t must contain 1 or {x_t.shape[0]} values, got {t.numel()}"
                )
            temb = self.timestep_embedder.encode_times(t).to(
                device=x_t.device, dtype=x_t.dtype
            )
            c = self.time_mlp(temb) + condition.to(
                device=x_t.device, dtype=x_t.dtype
            )
            token_pos = self.token_pos[:, : x_t.shape[1], :].to(
                device=x_t.device, dtype=x_t.dtype
            )
            h = self.in_proj(x_t) + token_pos
        else:
            raise ValueError(
                f"t must be scalar, [B], or [B, K, 1], got {tuple(t.shape)}"
            )

        if not return_stats:
            for block in self.blocks:
                h = block(h, c)
            if token_axis:
                shift_out, scale_out = self.adaLN_out(c).chunk(2, dim=-1)
                return self.out(modulate(self.norm_out(h), shift_out, scale_out))
            shift_out, scale_out = self.adaLN_out(c)[:, None, :].chunk(2, dim=-1)
            return self.out(modulate(self.norm_out(h), shift_out, scale_out))

        block_stats: list[dict[str, torch.Tensor]] = []
        for block in self.blocks:
            h, s = block(h, c, return_stats=True)
            block_stats.append(s)
        with torch.inference_mode():
            stats = {
                "temb": temb.square().mean().sqrt(),
                "fused": condition.square().mean().sqrt(),
                "c": c.square().mean().sqrt(),
                **{
                    k: torch.stack([s[k] for s in block_stats]).mean()
                    for k in ("gate_msa", "gate_mlp", "scale_msa", "scale_mlp")
                },
            }
        if token_axis:
            shift_out, scale_out = self.adaLN_out(c).chunk(2, dim=-1)
            out = self.out(modulate(self.norm_out(h), shift_out, scale_out))
        else:
            shift_out, scale_out = self.adaLN_out(c)[:, None, :].chunk(2, dim=-1)
            out = self.out(modulate(self.norm_out(h), shift_out, scale_out))
        return out, stats


class Critic(nn.Module):
    """State-value baseline V(s): MLP on current frame."""

    def __init__(self, obs_dim: int, hidden_dim: int):
        super().__init__()
        if obs_dim <= 0:
            raise ValueError("obs_dim must be positive")
        if hidden_dim < 1:
            raise ValueError("hidden_dim must be positive")
        self.obs_dim = obs_dim
        self.mlp = CondMLP(obs_dim, hidden_dim, 1)

    def forward(self, states: torch.Tensor) -> torch.Tensor:
        """[B, T, obs] -> [B] using the current frame."""
        if states.dim() != 3:
            raise ValueError(
                f"states must have shape [B, T, obs], got {tuple(states.shape)}"
            )
        if states.shape[0] < 1 or states.shape[1] < 1:
            raise ValueError("states must have positive batch and time dimensions")
        if states.shape[2] != self.obs_dim:
            raise ValueError(
                f"states feature dimension must be {self.obs_dim}, got {states.shape[2]}"
            )
        return self.mlp(states[:, -1, :]).squeeze(-1)

In [ ]:
%%writefile src/sjfpo/processor_sjfpo.py
"""SJFPO pre/post processors (LeRobot fashion): obs normalize, action unscale.

Note: ``ProcessorStep`` is not a ``torch.nn.Module`` — no ``register_buffer``.
Bounds are stored as plain attributes with a small per-(device, dtype) cache
so ``unnormalize`` avoids a per-call ``.to()``.
"""

from __future__ import annotations

import torch

from lerobot.configs import PipelineFeatureType, PolicyFeature
from lerobot.processor.pipeline import ProcessorStep, ProcessorStepRegistry
from lerobot.utils.constants import ACTION, OBS_STATE

from sjfpo.networks import RunningMeanStd


@ProcessorStepRegistry.register(name="sjfpo_observation_normalizer")
class SJFPOObservationProcessorStep(ProcessorStep):
    """Normalize ``observation.state`` with the shared live ``RunningMeanStd``.

    Holds a **reference** to the policy's normalizer, not a copy, so
    ``update_stats`` folds into the same buffers the policy reads at forward
    time.
    """

    def __init__(self, normalizer: RunningMeanStd):
        # ProcessorStep is not nn.Module; do not call super().__init__().
        self.normalizer = normalizer

    @torch.no_grad()
    def normalize(self, states: torch.Tensor) -> torch.Tensor:
        """Raw window -> normalized window (identity until the first update)."""
        return self.normalizer(states)

    def update_stats(self, states: torch.Tensor) -> None:
        """Fold raw training frames into the running stats (train batches only)."""
        self.normalizer.update(states)

    @torch.no_grad()
    def __call__(self, batch: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
        out = dict(batch)
        out[OBS_STATE] = self.normalize(batch[OBS_STATE])
        return out

    def transform_features(
        self, features: dict[PipelineFeatureType, dict[str, PolicyFeature]]
    ) -> dict[PipelineFeatureType, dict[str, PolicyFeature]]:
        return features


@ProcessorStepRegistry.register(name="sjfpo_action_unnormalizer")
class SJFPOActionPostprocessorStep(ProcessorStep):
    """Unscale policy action chunks from [-1, 1] to real env bounds.

    Bounds are updated atomically by ``set_action_bounds()``. A tiny
    per-``(device, dtype)`` cache keeps the pre-computed affine coefficients
    ``_scale`` / ``_bias`` local to whatever device the incoming chunk is on,
    so ``unnormalize`` is a single FMA with no per-call ``.to()``. Updating
    bounds clears both caches.
    """

    def __init__(self, action_low: torch.Tensor, action_high: torch.Tensor):
        self._scale_cache: dict[tuple[str, torch.dtype], torch.Tensor] = {}
        self._bias_cache: dict[tuple[str, torch.dtype], torch.Tensor] = {}
        self.set_action_bounds(action_low, action_high)

    def set_action_bounds(
        self,
        action_low: torch.Tensor,
        action_high: torch.Tensor,
    ) -> None:
        """Atomically replace bounds and invalidate cached affine coefficients."""
        low = torch.as_tensor(action_low, dtype=torch.float32).reshape(-1).clone()
        high = torch.as_tensor(action_high, dtype=torch.float32).reshape(-1).clone()
        if low.shape != high.shape:
            raise ValueError(
                f"action_low shape {tuple(low.shape)} != "
                f"action_high shape {tuple(high.shape)}"
            )
        if not bool((high >= low).all()):
            raise ValueError("action_high must be elementwise >= action_low")

        self.action_low = low
        self.action_high = high

        # Affine form: y = _scale * s + _bias
        #   = 0.5 (high - low) * s + 0.5 (high + low)
        self._scale_cpu = 0.5 * (high - low)
        self._bias_cpu = 0.5 * (high + low)
        self._scale_cache.clear()
        self._bias_cache.clear()

    def _coefs_for(self, ref: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        key = (str(ref.device), ref.dtype)
        scale = self._scale_cache.get(key)
        if scale is None:
            scale = self._scale_cpu.to(device=ref.device, dtype=ref.dtype)
            bias = self._bias_cpu.to(device=ref.device, dtype=ref.dtype)
            self._scale_cache[key] = scale
            self._bias_cache[key] = bias
        return scale, self._bias_cache[key]

    def to(self, device: torch.device | str) -> "SJFPOActionPostprocessorStep":
        """Optional eager move: pre-populate the cache for ``device``.

        Not required (``unnormalize`` will populate it on first call), but
        useful when you want the transfer to happen outside the hot path.
        """
        device = torch.device(device)
        self._scale_cpu.to(device=device)
        self._bias_cpu.to(device=device)
        # Prime the cache for the default float dtype (unchanged if already primed).
        key = (str(device), torch.float32)
        self._scale_cache.setdefault(
            key, self._scale_cpu.to(device=device, dtype=torch.float32)
        )
        self._bias_cache.setdefault(
            key, self._bias_cpu.to(device=device, dtype=torch.float32)
        )
        self.action_low = self.action_low.to(device=device)
        self.action_high = self.action_high.to(device=device)
        return self

    @torch.inference_mode()
    def unnormalize(self, scaled: torch.Tensor) -> torch.Tensor:
        """Scaled chunk -> real env actions (identity when bounds are ±1)."""
        scale, bias = self._coefs_for(scaled)
        return torch.addcmul(bias, scale, scaled)

    @torch.inference_mode()
    def __call__(self, batch: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
        out = dict(batch)
        out[ACTION] = self.unnormalize(batch[ACTION])
        return out

    def transform_features(
        self, features: dict[PipelineFeatureType, dict[str, PolicyFeature]]
    ) -> dict[PipelineFeatureType, dict[str, PolicyFeature]]:
        return features


def make_sjfpo_pre_post_processors(
    policy,
) -> tuple[SJFPOObservationProcessorStep, SJFPOActionPostprocessorStep]:
    """Build the pre/post pair; policy setter synchronizes action bounds.

    The observation step reads and updates the policy's ``obs_normalizer``
    buffers in place. The action step owns its validated affine cache, and
    ``SJFPOPolicy.set_action_bounds()`` updates it atomically.
    """
    return (
        SJFPOObservationProcessorStep(policy.obs_normalizer),
        SJFPOActionPostprocessorStep(policy.action_low, policy.action_high),
    )

In [ ]:
%%writefile src/sjfpo/sweep.py
"""Reentrant sweep executor: scan run dirs, execute pending runs in order."""

from __future__ import annotations

import atexit
import json
import time
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from sjfpo.config import RunSpec


class SweepError(RuntimeError):
    """Scan-time errors (corrupt manifest, missing files)."""

    pass


@dataclass(frozen=True)
class ScanResult:
    """Scan outcome: pending (missing+partial), skipped (complete), diverged."""

    pending: list[RunSpec]
    skipped: list[RunSpec]
    diverged: list[RunSpec]


def format_run_directory_name(spec: RunSpec) -> str:
    """Per-run subdir: {method}_{env}_seed{seed}/."""
    return f"{spec.method}_{spec.env}_seed{spec.seed}"


def read_run_manifest(run_dir: Path) -> dict[str, Any]:
    """Read and parse manifest.json, raising SweepError on failure."""
    manifest_path = run_dir / "manifest.json"
    try:
        return json.loads(manifest_path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise SweepError(f"corrupt manifest in {run_dir}: {e}") from e
    except OSError as e:
        raise SweepError(f"failed to read manifest in {run_dir}: {e}") from e


def classify_run_status(
    run_dir: Path,
    dir_name: str,
    manifest_reader: Callable[[Path], dict[str, Any]],
) -> str:
    """Classify one run dir: missing | diverged | complete | partial."""
    if not run_dir.is_dir():
        return "missing"
    if not (run_dir / "manifest.json").is_file():
        return "missing"
    try:
        manifest = manifest_reader(run_dir)
    except SweepError:
        raise SweepError(f"failed to scan run {dir_name}") from None
    if manifest.get("diverged", False) is True:
        return "diverged"
    if manifest.get("complete", False) is True:
        return "complete"
    return "partial"


def print_session_summary(
    completed_count: int,
    total_count: int,
    session_start_time: float,
    failed: list[str] | None = None,
) -> None:
    """Print completed/failed/remaining/hours session summary."""
    elapsed_hours = (time.time() - session_start_time) / 3600.0
    failed = list(failed or [])
    remaining = total_count - completed_count - len(failed)
    print(
        f"SESSION SUMMARY: {completed_count} completed, "
        f"{len(failed)} failed, {remaining} remaining, "
        f"{elapsed_hours:.2f} hours elapsed"
    )
    for name in failed:
        print(f"SESSION SUMMARY FAILED: {name}")


def scan_runs(
    matrix: tuple[RunSpec, ...],
    base_dir: str | Path,
    dir_name_fn: Callable[[RunSpec], str] = format_run_directory_name,
    manifest_reader: Callable[[Path], dict[str, Any]] = read_run_manifest,
    sort_key: Callable[[RunSpec], tuple[Any, ...]] = lambda s: (
        s.method,
        s.env,
        s.seed,
    ),
) -> ScanResult:
    """Scan the run matrix, classifying each run as pending/skipped/diverged."""
    base = Path(base_dir)
    pending: list[RunSpec] = []
    skipped: list[RunSpec] = []
    diverged: list[RunSpec] = []
    index_map = {spec: i for i, spec in enumerate(matrix)}
    for spec in matrix:
        run_dir = base / dir_name_fn(spec)
        status = classify_run_status(run_dir, dir_name_fn(spec), manifest_reader)
        if status == "missing" or status == "partial":
            pending.append(spec)
        elif status == "complete":
            skipped.append(spec)
        else:
            diverged.append(spec)
    pending.sort(key=lambda s: sort_key(s) + (index_map.get(s, 0),))
    return ScanResult(pending=pending, skipped=skipped, diverged=diverged)


def flush_pending_runs(
    pending_runs: list[RunSpec],
    base_dir: str | Path,
    run_fn: Callable[[RunSpec, Path], dict[str, Any]],
    session_start_time: float,
    dir_name_fn: Callable[[RunSpec], str] = format_run_directory_name,
) -> tuple[list[dict[str, Any]], list[str]]:
    """Execute pending runs sequentially; failures are recorded, never silent."""
    base = Path(base_dir)
    results: list[dict[str, Any]] = []
    failed: list[str] = []
    completed_count = 0
    try:
        for spec in pending_runs:
            name = dir_name_fn(spec)
            try:
                result = run_fn(spec, base)
                results.append(result)
                completed_count += 1
            except Exception as e:
                failed.append(name)
                print(f"ERROR in run {name}: {e}")
                continue
    finally:
        print_session_summary(
            completed_count, len(pending_runs), session_start_time, failed
        )
    return results, failed


def announce_scan_results(scan_result: ScanResult) -> None:
    """Print the skipped/pending/diverged scan summary."""
    print(
        f"{len(scan_result.skipped)} skipped (complete), "
        f"{len(scan_result.pending)} pending, "
        f"{len(scan_result.diverged)} diverged"
    )


def register_atexit_summary(pending_count: int, session_start: float) -> None:
    """Register belt-and-braces atexit summary for SIGTERM'd sessions."""

    def print_atexit_summary() -> None:
        elapsed_hours = (time.time() - session_start) / 3600.0
        print(
            f"SESSION SUMMARY (atexit): {pending_count} pending at start, "
            f"{elapsed_hours:.2f} hours elapsed"
        )

    atexit.register(print_atexit_summary)


def run_pending(
    matrix: tuple[RunSpec, ...],
    base_dir: str | Path,
    run_fn: Callable[[RunSpec, Path], dict[str, Any]],
    scan_fn: Callable[[tuple[RunSpec, ...], str | Path], ScanResult] = scan_runs,
    dir_name_fn: Callable[[RunSpec], str] = format_run_directory_name,
) -> dict[str, Any]:
    """Scan, report, execute pending runs, register atexit handler."""
    base = Path(base_dir)
    session_start = time.time()

    scan_result = scan_fn(matrix, base)

    announce_scan_results(scan_result)

    register_atexit_summary(len(scan_result.pending), session_start)

    results, failed = flush_pending_runs(
        scan_result.pending, base, run_fn, session_start, dir_name_fn
    )

    return {
        "completed": len(results),
        "skipped": len(scan_result.skipped),
        "diverged": len(scan_result.diverged),
        "failed": failed,
        "results": results,
    }

In [ ]:
%%writefile src/sjfpo/train_sjfpo.py
"""SJFPO online training entry (LeRobot-native, MuJoCo-only).

``generate_rollout`` gathers a batch, ``forward`` computes the loss; this loop
only steps the optimizer, evaluates, and checkpoints.

Performance notes
-----------------
* Vectorized training envs use ``AsyncVectorEnv`` (parallel worker processes)
  when available; falls back to ``SyncVectorEnv`` on pickling failures.
* Grad-norm-after-clip is derived analytically (``min(pre, clip_val)``);
  pre-clip norms are accumulated on device and converted to a Python float
  once per rollout, not once per minibatch.
* ``zero_grad(set_to_none=True)`` frees grad buffers between minibatches.
* The ``forward`` diagnostic pass is requested only when wandb will consume it.
* Eval and checkpoint are triggered by explicit step counters, so no eval or
  save is skipped when ``collect_steps`` overshoots the boundary.
"""

from __future__ import annotations

import dataclasses
import json
import sys
import time
from collections import deque
from tqdm import tqdm
from pathlib import Path
from typing import Any

import numpy as np
import torch
from lerobot.configs.types import FeatureType, PolicyFeature
from lerobot.utils.constants import ACTION, DONE, OBS_STATE, REWARD, TRUNCATED
from lerobot.utils.random_utils import set_seed

from sjfpo.checkpointing import load_policy_checkpoint, save_policy_checkpoint
from sjfpo.configuration_sjfpo import SJFPOConfig
from sjfpo.evaluator import Evaluator, check_divergence, mean_chunk_jerk
from sjfpo.modeling_sjfpo import SJFPOAborted, SJFPOPolicy
from lerobot.utils.constants import CHECKPOINTS_DIR
from sjfpo.utils import maybe_compile, resolve_device, to_numpy_1d
from sjfpo.wandb_logging import (
    finish_run,
    init_wandb_run_for_method,
    log_eval_summary,
    log_metrics,
)


# Consecutive updates with most CFM samples above ``cfm_clamp`` before aborting.
ABOVE_CLAMP_ABORT_UPDATES = 5


def mini_batch_indices(
    n: int,
    num_minibatches: int,
    device: torch.device | str,
    num_epochs: int = 1,
):
    """Yield shuffled minibatch indices for each epoch."""
    if n < 1:
        raise ValueError(f"n={n} must be >= 1")
    n_mb = max(1, min(int(num_minibatches), n))
    device = torch.device(device)
    perm = torch.randperm(n, device=device)
    for _ in range(max(1, int(num_epochs))):
        for i in range(n_mb):
            start = (i * n) // n_mb
            stop = ((i + 1) * n) // n_mb
            yield perm[start:stop]


class _EvalPolicy:
    """Single-action eval adapter over ``SJFPOPolicy.select_action``."""

    def __init__(self, policy: SJFPOPolicy):
        self.policy = policy

    def reset(self) -> None:
        self.policy.reset()

    def __call__(self, obs: np.ndarray) -> np.ndarray:
        was_training = self.policy.training
        self.policy.eval()
        try:
            action = self.policy.select_action({"observation.state": obs})
        finally:
            self.policy.train(was_training)
        return to_numpy_1d(action)  # atleast_1d: 1-D envs must stay [1]


def _compile_in_place(policy: SJFPOPolicy) -> None:
    """Apply ``maybe_compile`` to CUDA submodules the training path touches."""
    device = next(policy.parameters()).device
    policy.actor = maybe_compile(policy.actor, device=device)
    policy.old_actor = maybe_compile(policy.old_actor, device=device)
    policy.critic = maybe_compile(policy.critic, device=device)


def build_policy(
    obs_dim: int,
    action_dim: int,
    cfg: SJFPOConfig,
    device=None,
    action_low: np.ndarray | None = None,
    action_high: np.ndarray | None = None,
) -> SJFPOPolicy:
    """Attach runtime observation/action features, then build the policy."""
    device = torch.device(device) if device is not None else resolve_device()
    policy_cfg = dataclasses.replace(
        cfg,
        input_features={
            "observation.state": PolicyFeature(FeatureType.STATE, (obs_dim,))
        },
        output_features={"action": PolicyFeature(FeatureType.ACTION, (action_dim,))},
    )
    policy = SJFPOPolicy(policy_cfg).to(device)
    _compile_in_place(policy)
    if action_low is not None and action_high is not None:
        policy.set_action_bounds(action_low, action_high)
    return policy


def _build_optimizer(
    policy: SJFPOPolicy, cfg: SJFPOConfig, params: list[dict], lr: float
):
    kwargs = dataclasses.asdict(cfg.get_optimizer_preset(lr=lr))
    kwargs.pop("grad_clip_norm", None)  # applied manually after backward
    if next(policy.parameters()).is_cuda:
        kwargs["fused"] = True
    return torch.optim.AdamW(params, **kwargs)


def build_optimizers(
    policy: SJFPOPolicy, cfg: SJFPOConfig
) -> tuple[torch.optim.Optimizer, torch.optim.Optimizer]:
    """Actor and critic get separate optimizers, LRs and grad-norm clips.

    A single shared group lets the critic's value gradient (orders of magnitude
    larger than the policy's) own the one grad-norm clip, so the policy step is
    both clipped and outnumbered.
    """
    return (
        _build_optimizer(
            policy, cfg, policy.get_actor_optim_params()["params"], cfg.policy_lr
        ),
        _build_optimizer(
            policy, cfg, policy.get_critic_optim_params()["params"], cfg.critic_lr
        ),
    )


def sample_minibatch(
    batch: dict[str, torch.Tensor],
    indices: torch.Tensor,
    out: dict[str, torch.Tensor] | None = None,
) -> dict[str, torch.Tensor]:
    """Slice dim-0 rows; every per-chunk batch value is ``[B, ...]``.

    ``mc_tau`` ``[B,M,K,1]`` and ``mc_eps`` ``[B,M,K,d]`` are stored per
    chunk/MC draw and are sliced with the minibatch. ``normalizer_observations``
    is passed through because
    it is one row per executed environment step, not per chunk.
    """
    if out is None:
        out = {}
    for k, v in batch.items():
        if k == "normalizer_observations":
            # The normalizer is updated once from the complete rollout batch.
            out[k] = v
        else:
            out[k] = v[indices]
    return out


def pad_minibatch_indices(
    indices: torch.Tensor, size: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Pad a minibatch by repeating its last row and return a validity mask."""
    valid = int(indices.numel())
    if size < valid:
        raise ValueError(f"compile batch size {size} is smaller than minibatch {valid}")
    if valid == size:
        mask = torch.ones(size, dtype=torch.bool, device=indices.device)
        return indices, mask
    padding = indices[-1:].expand(size - valid)
    padded = torch.cat([indices, padding])
    mask = torch.arange(size, device=indices.device) < valid
    return padded, mask


def random_baseline(
    evaluator: Evaluator, env: Any, max_episodes: int = 30
) -> float:
    """Mean random-policy return over 30 episodes for divergence gating.

    Returns ``0.0`` unchanged — callers gate on ``baseline > 0`` before
    treating the value as meaningful (see ``check_divergence``).
    """
    baseline_evaluator = Evaluator(
        env,
        eval_seed=evaluator.eval_seed,
        max_episodes=max_episodes,
        max_steps=evaluator.max_steps,
    )
    results = baseline_evaluator.evaluate(
        lambda obs: env.action_space.sample(), out_dir=None
    )
    if not results:
        return 1.0
    return float(np.mean([r["return"] for r in results]))


def _make_vec_env(env_name: str, num_envs: int):
    """Build a vectorized training env, preferring async workers.

    ``AsyncVectorEnv`` parallelizes env stepping across worker processes —
    the single biggest rollout-throughput win for ``num_envs > 1``. Falls
    back to ``SyncVectorEnv`` if the async path is unavailable in the
    current runtime (e.g. no /dev/shm, pickling of the env factory fails).
    """
    import gymnasium as gym

    env_fns = [lambda: gym.make(env_name) for _ in range(num_envs)]
    try:
        return gym.vector.AsyncVectorEnv(env_fns)
    except Exception:
        return gym.vector.SyncVectorEnv(env_fns)


# ---------------------------------------------------------------------------
# Main loop
# ---------------------------------------------------------------------------


def train_sjfpo_run(
    env: Any,
    out_dir: str | Path,
    cfg: SJFPOConfig | None = None,
    *,
    seed: int = 3407,
    total_steps: int = 2000,
    eval_steps: int = 1000,
    save_steps: int = 1000,
    obs_dim: int | None = None,
    action_dim: int | None = None,
    resume_from_checkpoint: str | Path | None = None,
    wandb_enabled: bool = True,
) -> dict[str, Any]:
    """Run a single SJFPO training run, returning a summary with artifacts."""
    cfg = cfg or SJFPOConfig()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    unwrapped = getattr(env, "unwrapped", env)
    spec = getattr(unwrapped, "spec", None)
    env_name = getattr(spec, "id", "unknown") if spec is not None else "unknown"
    if obs_dim is None or action_dim is None:
        obs_dim = int(np.prod(env.observation_space.shape))
        action_dim = int(np.prod(env.action_space.shape))

    set_seed(seed)
    device = resolve_device()

    # Build vectorized training env *before* CUDA init so forked workers don't
    # inherit the parent's CUDA context (CleanRL / SB3 ordering).
    train_env = env
    if cfg.num_envs > 1 and env_name != "unknown":
        train_env = _make_vec_env(env_name, cfg.num_envs)

    action_low = np.atleast_1d(np.asarray(env.action_space.low, dtype=np.float32))
    action_high = np.atleast_1d(np.asarray(env.action_space.high, dtype=np.float32))

    global_step = 0
    if resume_from_checkpoint is not None:
        loaded = load_policy_checkpoint(
            resume_from_checkpoint,
            device=device,
            action_low=action_low,
            action_high=action_high,
        )
        if len(loaded) == 4:
            policy, actor_optimizer, critic_optimizer, counters = loaded
        else:
            policy, _legacy_optimizer, counters = loaded
            actor_optimizer, critic_optimizer = build_optimizers(policy, policy.config)
        global_step = int(counters["global_step"])
        policy.train()
        # Re-apply compilation — from_pretrained builds an eager policy.
        _compile_in_place(policy)
        (out_dir / "resume_info.json").write_text(
            json.dumps(
                {
                    "resume_from_checkpoint": str(resume_from_checkpoint),
                    "restored_global_step": global_step,
                },
                indent=2,
            ),
            encoding="utf-8",
        )
    else:
        policy = build_policy(
            obs_dim,
            action_dim,
            cfg,
            device=device,
            action_low=action_low,
            action_high=action_high,
        )
        actor_optimizer, critic_optimizer = build_optimizers(policy, cfg)

    print(
        f"SJFPO policy: actor_depth={cfg.dit_depth} hidden={cfg.hidden_dim} "
        f"params={sum(p.numel() for p in policy.parameters()):,}",
        file=sys.stderr,
    )
    # -- Eval / checkpoint infrastructure -----------------------------------
    evaluator = Evaluator(env, eval_seed=10000 + seed, max_episodes=12)
    eval_policy = _EvalPolicy(policy)
    r_rand = random_baseline(evaluator, env)
    wandb_enabled = init_wandb_run_for_method(
        method="sjfpo",
        env_name=env_name,
        seed=seed,
        total_steps=total_steps,
        cfg_dict=dataclasses.asdict(cfg),
        wandb_enabled=wandb_enabled,
    )

    # Next step at which each periodic action fires.
    next_eval = eval_steps
    next_save = save_steps

    eval_count = 0
    consecutive_below = 0
    above_clamp_updates = 0
    diverged = False
    best_return: float | None = None
    steps_to_best: int | None = None
    recent_eval_returns: deque[float] = deque(maxlen=10)
    all_eval_returns: list[float] = []
    artifacts: list[str] = []
    elapsed = 0.0
    fps = 0.0
    t_start = time.time()

    # Hoisted once — indexing the optimizer's param groups every minibatch is
    # pure Python overhead on the critical path.
    actor_params = [
        p for group in actor_optimizer.param_groups for p in group["params"]
    ]
    critic_params = [
        p for group in critic_optimizer.param_groups for p in group["params"]
    ]

    try:
        while global_step < total_steps:
            steps_to_go = total_steps - global_step
            collect_steps = min(cfg.chunks_per_update * cfg.action_horizon, steps_to_go)
            policy.snapshot_behavior_policy()
            batch = policy.generate_rollout(
                train_env,
                collect_steps,
                action_low=action_low,
                action_high=action_high,
            )
            # Count executed env steps via executed chunk prefixes.
            step_delta = int((~batch["action_is_pad"]).sum())
            if step_delta <= 0:
                raise RuntimeError(
                    "rollout returned zero executed steps; aborting to avoid "
                    "an infinite training loop."
                )
            global_step += step_delta

            rollout_jerk = None
            if "action" in batch and "action_is_pad" in batch:
                rollout_jerk = mean_chunk_jerk(
                    batch["action"].detach().cpu().numpy(),
                    batch["action_is_pad"].detach().cpu().numpy(),
                )
            action_abs_max = float(batch["action"].abs().max())
            action_magnitude = float(batch["action"].abs().mean())
            if action_abs_max > 1.05:
                raise ValueError(
                    f"Unnormalized actions detected: max_abs={action_abs_max:.3f} "
                    "(expected scaled [-1, 1]; check rollout/action bounds)"
                )

            policy.train()

            infos: list[dict[str, float]] = []
            actor_pre_norms: list[torch.Tensor] = []
            critic_pre_norms: list[torch.Tensor] = []
            above_clamp_fracs: list[torch.Tensor] = []
            n_chunks = batch["advantages"].shape[0]
            n_minibatches = min(cfg.num_minibatches, n_chunks)
            expected_chunks = (
                cfg.chunks_per_update * cfg.action_horizon + cfg.n_action_steps - 1
            ) // cfg.n_action_steps
            if cfg.compile_batch_size > 0:
                compile_batch_size = cfg.compile_batch_size
            else:
                compile_batch_size = max(
                    1,
                    (expected_chunks + n_minibatches - 1) // n_minibatches,
                )
            compile_batch_size = max(
                compile_batch_size,
                (n_chunks + n_minibatches - 1) // n_minibatches,
            )
            # Match Accelerate's accumulation contract: backward on every
            # microbatch, synchronize/step only at the accumulation boundary,
            # and always flush a final partial accumulation group.
            microbatches = list(
                mini_batch_indices(
                    n_chunks,
                    cfg.num_minibatches,
                    batch["advantages"].device,
                    num_epochs=cfg.num_epochs,
                )
            )
            accumulation_steps = cfg.grad_accum_steps
            actor_optimizer.zero_grad(set_to_none=True)
            critic_optimizer.zero_grad(set_to_none=True)
            minibatch: dict[str, torch.Tensor] = {}
            finite_error = torch.zeros(
                (), device=batch["advantages"].device, dtype=torch.bool
            )
            for micro_step, mb_idx in enumerate(microbatches):
                group_start = micro_step - (micro_step % accumulation_steps)
                group_size = min(accumulation_steps, len(microbatches) - group_start)
                padded_indices, sample_mask = pad_minibatch_indices(
                    mb_idx, compile_batch_size
                )
                padded = int(padded_indices.numel()) != int(mb_idx.numel())
                minibatch = sample_minibatch(batch, padded_indices, minibatch)
                minibatch["sample_mask"] = sample_mask
                loss, epoch_info = policy(
                    minibatch,
                    compute_info=wandb_enabled and not padded,
                    check_finite=False,
                )
                finite_error |= ~torch.isfinite(loss)
                if loss.dim() != 0:
                    raise ValueError(
                        f"Loss must be scalar before backward; got {tuple(loss.shape)}."
                    )
                (loss / group_size).backward()
                is_boundary = (
                    (micro_step + 1) % accumulation_steps == 0
                    or micro_step + 1 == len(microbatches)
                )
                if is_boundary:
                    if bool(finite_error):
                        raise SJFPOAborted(
                            "Non-finite loss detected during gradient accumulation"
                        )
                    finite_error.zero_()
                    pre_clip_actor = torch.nn.utils.clip_grad_norm_(
                        actor_params, cfg.grad_clip_norm
                    )
                    actor_pre_norms.append(pre_clip_actor.detach())
                    pre_clip_critic = torch.nn.utils.clip_grad_norm_(
                        critic_params, cfg.grad_clip_norm
                    )
                    critic_pre_norms.append(pre_clip_critic.detach())
                    above_clamp_fracs.append(policy.cfm_above_clamp_frac)
                    actor_optimizer.step()
                    critic_optimizer.step()
                    actor_optimizer.zero_grad(set_to_none=True)
                    critic_optimizer.zero_grad(set_to_none=True)
                    policy.step_count += 1
                if epoch_info is not None:
                    infos.append(epoch_info)

            policy.update_ema(global_step)

            # Update once per collected rollout, after the policy/value update.
            # These are raw post-step frames, not overlapping warmup windows.
            policy.update_observation_stats(batch["normalizer_observations"])

            # Aggregate across epochs/minibatches.
            if infos:
                info = {k: sum(d[k] for d in infos) / len(infos) for k in infos[0]}
            else:
                info = None
            actor_grad_norm_mean_pre = (
                float(torch.stack(actor_pre_norms).mean()) if actor_pre_norms else 0.0
            )
            critic_grad_norm_mean_pre = (
                float(torch.stack(critic_pre_norms).mean()) if critic_pre_norms else 0.0
            )
            grad_norm_mean_pre = actor_grad_norm_mean_pre + critic_grad_norm_mean_pre
            grad_norm_mean_post = min(grad_norm_mean_pre, cfg.grad_clip_norm)

            if wandb_enabled and info is not None:
                data = {f"train/{k}": v for k, v in info.items()}
                data["train/actor_grad_norm_preclip"] = actor_grad_norm_mean_pre
                data["train/critic_grad_norm_preclip"] = critic_grad_norm_mean_pre
                data["train/grad_norm_preclip"] = grad_norm_mean_pre
                data["train/grad_norm_after_clip"] = grad_norm_mean_post
                if rollout_jerk is not None:
                    data["train/rollout_jerk"] = rollout_jerk
                data["train/action_magnitude"] = action_magnitude
                data["train/total_loss"] = (
                    info["policy_loss"] + cfg.vf_coef * info["value_loss"]
                )
                data.update(
                    {
                        "gae/advantages_mean": float(batch["advantages"].mean()),
                        "gae/advantages_std": float(batch["advantages"].std()),
                        "gae/returns_mean": float(batch["returns"].mean()),
                        "gae/values_mean": float(batch["values"].mean()),
                    }
                )
                log_metrics(data, step=global_step)

            # clamp() has zero gradient above its max, so CFM samples above
            # cfm_clamp contribute nothing: rho stays exactly 1 and the policy
            # receives no gradient while the critic keeps training. Abort
            # instead of burning a run that can only plateau.
            above_clamp_frac = (
                float(torch.stack(above_clamp_fracs).mean())
                if above_clamp_fracs
                else 0.0
            )
            if above_clamp_frac > 0.5:
                above_clamp_updates += 1
                if above_clamp_updates >= ABOVE_CLAMP_ABORT_UPDATES:
                    raise SJFPOAborted(
                        f"{ABOVE_CLAMP_ABORT_UPDATES} consecutive updates with >50% "
                        f"of CFM samples above cfm_clamp={cfg.cfm_clamp} "
                        f"(frac={above_clamp_frac:.2f}, "
                        f"cfm_loss_reduction={cfg.cfm_loss_reduction!r}). The policy "
                        "ratio is pinned at 1 and gets no gradient. Raise "
                        "cfm_clamp, or use a reduction whose scale fits under it."
                    )
            else:
                above_clamp_updates = 0

            # -- Periodic eval ------------------------------------------
            diverged_now = False
            diverge_msg = ""
            if global_step >= next_eval or global_step >= total_steps:
                eval_results = evaluator.evaluate(
                    eval_policy, out_dir=out_dir, append_csv=True
                )
                mean_return = float(np.mean([r["return"] for r in eval_results]))
                all_eval_returns.append(mean_return)
                recent_eval_returns.append(mean_return)
                eval_count += 1
                if best_return is None or mean_return > best_return:
                    best_return = mean_return
                    steps_to_best = global_step
                if wandb_enabled:
                    log_eval_summary(
                        mean_return,
                        eval_results,
                        all_eval_returns,
                        eval_count,
                        global_step,
                        r_rand=r_rand,
                        consecutive_below=consecutive_below,
                        best_return=best_return,
                        steps_to_best=steps_to_best,
                    )
                consecutive_below, diverged_now = check_divergence(
                    mean_return, r_rand, consecutive_below
                )
                if diverged_now:
                    diverge_msg = (
                        f"DIVERGED at step {global_step}: "
                        f"mean_return={mean_return:.2f} "
                        f"< 0.10*R_rand={0.10 * r_rand:.2f}"
                    )
                # Advance the trigger past the current step.
                next_eval = max(global_step + eval_steps, next_eval + eval_steps)

            if diverged_now:
                diverged = True
                if wandb_enabled:
                    log_metrics(
                        {"eval/diverged": 1, "eval/diverge_step": global_step},
                        step=global_step,
                    )
                print(diverge_msg)
                break

            # -- Periodic checkpoint ------------------------------------
            if global_step >= next_save or global_step >= total_steps:
                ckpt_dir = out_dir / CHECKPOINTS_DIR / f"step_{global_step}"
                save_policy_checkpoint(
                    ckpt_dir,
                    policy=policy,
                    actor_optimizer=actor_optimizer,
                    critic_optimizer=critic_optimizer,
                    global_step=global_step,
                    ema_state=policy.ema_state_dict(),
                )
                artifacts.append(str(ckpt_dir / "model.safetensors"))
                next_save = max(global_step + save_steps, next_save + save_steps)

        elapsed = time.time() - t_start
        fps = global_step / max(elapsed, 1e-6)
        if wandb_enabled:
            log_metrics(
                {"timing/fps": fps, "timing/elapsed_s": elapsed},
                step=global_step,
            )
    finally:
        if train_env is not env:
            train_env.close()
        if wandb_enabled:
            finish_run(exit_code=1 if diverged else 0)

    ckpt_dir = out_dir / CHECKPOINTS_DIR / "final"
    save_policy_checkpoint(
        ckpt_dir,
        policy=policy,
        actor_optimizer=actor_optimizer,
        critic_optimizer=critic_optimizer,
        global_step=global_step,
        ema_state=policy.ema_state_dict(),
    )
    artifacts.append(str(ckpt_dir / "model.safetensors"))

    csv_path = out_dir / "eval_history.csv"
    if csv_path.exists():
        artifacts.append(str(csv_path))

    return {
        "method": "sjfpo",
        "env_name": env_name,
        "seed": seed,
        "reward_mean_last10": (
            float(np.mean(recent_eval_returns)) if recent_eval_returns else 0.0
        ),
        "fps": fps,
        "artifacts": artifacts,
        "diverged": diverged,
        "global_step": global_step,
    }

In [ ]:
%%writefile src/sjfpo/utils.py
"""Shared stateless utilities (no sjfpo.* imports — circular-import prevention)."""

from __future__ import annotations

import contextlib
import os
import sys
from functools import lru_cache
from pathlib import Path

import numpy as np
import torch

from lerobot.processor.converters import from_tensor_to_numpy, to_tensor


# ---------------------------------------------------------------------------
# Runtime probes (lazy, cached)
# ---------------------------------------------------------------------------


@lru_cache(maxsize=1)
def cuda_available() -> bool:
    """Whether CUDA is usable. Cached — first call triggers CUDA init."""
    return torch.cuda.is_available()


@lru_cache(maxsize=1)
def is_rtx_1000_pro_blackwell() -> bool:
    """True when running on an NVIDIA RTX PRO 1000 Blackwell GPU."""
    if not cuda_available():
        return False
    try:
        name = torch.cuda.get_device_name(0).lower()
    except Exception:
        return False
    return "rtx pro 1000" in name and "blackwell" in name


@lru_cache(maxsize=1)
def _colab_drive_present() -> bool:
    return Path("/content/drive").exists()


# ---------------------------------------------------------------------------
# Device / dtype resolution
# ---------------------------------------------------------------------------


def resolve_device() -> torch.device:
    """CUDA when available, else CPU. ``SJFPO_DEVICE`` ∈ {auto, cpu, cuda}."""
    want = os.environ.get("SJFPO_DEVICE", "auto").lower()
    if want == "cpu":
        return torch.device("cpu")
    if want == "cuda":
        return torch.device("cuda") if cuda_available() else torch.device("cpu")
    if want != "auto":
        raise ValueError(
            f"SJFPO_DEVICE={want!r} unknown; expected 'auto', 'cpu', or 'cuda'"
        )
    return torch.device("cuda") if cuda_available() else torch.device("cpu")


def resolve_storage_dtype(
    device: str | torch.device | None = None,
) -> torch.dtype:
    """Storage dtype for the actual execution device.

    CPU always uses FP32. RTX PRO 1000 Blackwell uses BF16 storage when the
    execution device is CUDA; all other devices use FP32.
    """
    if device is not None and torch.device(device).type == "cpu":
        return torch.float32
    return torch.bfloat16 if is_rtx_1000_pro_blackwell() else torch.float32


def resolve_compute_dtype() -> torch.dtype:
    """Matmul/autocast dtype with explicit environment override.

    ``SJFPO_DTYPE`` is parsed before the Blackwell fast path so ``fp32`` is
    honored there as well. BF16 is only selected when CUDA supports it.
    """
    want = os.environ.get("SJFPO_DTYPE", "auto").lower()
    if want not in {"auto", "fp32", "bf16"}:
        raise ValueError(
            f"SJFPO_DTYPE={want!r} unknown; expected 'auto', 'fp32', or 'bf16'"
        )
    if want == "fp32":
        return torch.float32
    if not cuda_available():
        return torch.float32
    if want == "bf16":
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
    if is_rtx_1000_pro_blackwell():
        return torch.bfloat16
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32


def autocast_context(device=None):
    """bf16 autocast on CUDA; no-op elsewhere (CPU, fp32, or non-CUDA device)."""
    if not cuda_available():
        return contextlib.nullcontext()
    if resolve_compute_dtype() != torch.bfloat16:
        return contextlib.nullcontext()
    dev_type = device.type if isinstance(device, torch.device) else "cuda"
    if dev_type != "cuda":
        return contextlib.nullcontext()
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16)


# ---------------------------------------------------------------------------
# torch.compile wrapper
# ---------------------------------------------------------------------------


def maybe_compile(
    module: torch.nn.Module, device: str | torch.device | None = None
) -> torch.nn.Module:
    """Compile with inductor after build, before train/infer.

    On CUDA by default. ``SJFPO_COMPILE=0`` opts out (e.g., for debugging or
    when the module is called with a fresh shape per iteration and the
    recompile cost dominates).
    """
    if os.environ.get("SJFPO_COMPILE", "1") == "0":
        return module
    target = (
        next(module.parameters()).device
        if device is None
        else torch.device(device)
    )
    if target.type != "cuda" or not cuda_available():
        return module
    # bf16 support is a proxy for "worth compiling"; Blackwell always qualifies.
    if not torch.cuda.is_bf16_supported() and not is_rtx_1000_pro_blackwell():
        return module
    # Symbolic batch dim: train minibatches (n_mb * num_mc), remainder splits,
    # and inference (B=1) share one graph instead of recompiling per B.
    # capture_scalar_outputs keeps guarded Python scalars (e.g. Critic's
    # executed-step check) in the graph instead of forcing an eager fallback.
    with torch._dynamo.config.patch(capture_scalar_outputs=True):
        compiled = torch.compile(module, dynamic=True)
    return compiled


# ---------------------------------------------------------------------------
# Environment / path discovery
# ---------------------------------------------------------------------------


def detect_runtime_env() -> str:
    """``kaggle`` | ``colab`` | ``local`` (env vars win over path probes)."""
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    return "local"


def _runtime_root(
    kaggle: str,
    colab_drive: str,
    colab_local: str,
    local: str,
    runtime: str | None = None,
) -> Path:
    """Shared branch logic for the three ``default_*`` path helpers."""
    runtime = runtime or detect_runtime_env()
    if runtime == "kaggle":
        return Path(kaggle)
    if runtime == "colab":
        drive = Path(colab_drive)
        return drive if _colab_drive_present() else Path(colab_local)
    return Path(local)


def default_runs_base(runtime: str | None = None) -> Path:
    """Per-run sweep directory (kaggle / colab-drive-or-content / local .omo)."""
    return _runtime_root(
        "/kaggle/working/runs",
        "/content/drive/MyDrive/spectral-joint-policy/runs",
        "/content/runs",
        ".omo/evidence/runs",
        runtime,
    )


def default_analysis_output(runtime: str | None = None) -> Path:
    """Analysis output directory (mirrors ``default_runs_base``)."""
    return _runtime_root(
        "/kaggle/working/analysis",
        "/content/drive/MyDrive/spectral-joint-policy/analysis",
        "/content/analysis",
        ".omo/evidence/analysis",
        runtime,
    )


def default_smoke_dir(runtime: str | None = None) -> Path:
    """Smoke-gate output directory (mirrors ``default_runs_base``)."""
    return _runtime_root(
        "/kaggle/working/smoke",
        "/content/drive/MyDrive/spectral-joint-policy/smoke",
        "/content/smoke",
        ".omo/evidence/smoke",
        runtime,
    )


# ---------------------------------------------------------------------------
# Tensor / array interop
# ---------------------------------------------------------------------------


def to_device_async(x: torch.Tensor, device) -> torch.Tensor:
    """Host-to-device move that overlaps with CPU work on CUDA.

    Pinned staging + ``non_blocking=True`` lets the DMA engine copy while
    Python keeps running. The caller is responsible for syncing before
    reading the result (any ``.item()`` / ``.cpu()`` / numpy conversion
    will sync implicitly).
    """
    device = torch.device(device)
    if device.type == "cuda" and x.device != device:
        if not x.is_pinned():
            x = x.pin_memory()
        return x.to(device, non_blocking=True)
    return x.to(device)


def filter_low_freq(x: torch.Tensor, k: int, dim: int = 1) -> torch.Tensor:
    """Keep only the ``k`` lowest-frequency bins along ``dim``."""
    if k < 1 or k > x.shape[dim]:
        raise ValueError(f"k={k} out of range for dim {dim} with size {x.shape[dim]}")
    s = [slice(None)] * x.dim()
    s[dim] = slice(None, k)
    return x[tuple(s)]


def to_numpy_1d(x: torch.Tensor | np.ndarray | float) -> np.ndarray:
    """Numpy with ``atleast_1d``: singletons stay ``[1]``, never 0-d/scalar."""
    return np.atleast_1d(from_tensor_to_numpy(x))


def to_tensor_1d(
    x: torch.Tensor | np.ndarray | float,
    dtype: torch.dtype | None = torch.float32,
    device: torch.device | str | None = None,
) -> torch.Tensor:
    """Tensor with ``atleast_1d``: singletons stay ``[1]``, never 0-d."""
    return torch.atleast_1d(to_tensor(x, dtype=dtype, device=device))

In [ ]:
%%writefile src/sjfpo/wandb_logging.py
"""wandb logging — direct wandb usage (installed and enabled)."""

from __future__ import annotations

import os

import numpy as np
import wandb


# Direct wandb usage (library installed and enabled).
# Enabled only when the caller passes ``wandb_enabled=True``.


# ---------------------------------------------------------------------------
# Run-name helpers
# ---------------------------------------------------------------------------


def compact_value(v) -> str:
    """Short wandb-safe rendering: ints plain, floats via ``:g``."""
    if isinstance(v, bool):
        return str(int(v))
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float):
        if v != v:
            return "nan"
        if v == float("inf"):
            return "inf"
        if v == float("-inf"):
            return "-inf"
        return f"{v:g}"
    return str(v)


def build_run_name(method: str, env_name: str, seed: int, cfg_dict: dict) -> str:
    tags: list[str] = []
    get = cfg_dict.get
    if (v := get("top_p_spectral_tokens")) is not None:
        tags.append(f"top{compact_value(v)}")
    if (v := get("hidden_dim")) is not None and (d := get("dit_depth")) is not None:
        tags.append(f"d{v}x{d}")
    if (v := get("num_epochs", get("learning_epochs"))) is not None:
        tags.append(f"ep{v}")
    if (v := get("policy_lr", get("lr"))) is not None:
        tags.append(f"lr{v:g}" if isinstance(v, float) else f"lr{v}")
    if (v := get("epsilon_clip", get("ratio_clip"))) is not None:
        tags.append(f"eps{compact_value(v)}")
    if get("use_per_bin_clip"):
        tags.append(
            f"bin{compact_value(get('epsilon_clip_lowfreq'))}"
            f"-{compact_value(get('epsilon_clip_highfreq'))}"
        )
    if (v := get("num_mc_samples")) is not None:
        tags.append(f"mc{v}")
    base = f"{method}_{env_name}_s{seed}"
    return base if not tags else f"{base}_{'_'.join(tags)}"


# ---------------------------------------------------------------------------
# Run lifecycle
# ---------------------------------------------------------------------------


def init_wandb_run_for_method(
    *,
    method: str,
    env_name: str,
    seed: int,
    total_steps: int,
    cfg_dict: dict,
    wandb_enabled: bool = True,
) -> bool:
    if not wandb_enabled:
        return False
    wandb.init(
        project=method,
        group=f"{method}_{env_name}",
        name=build_run_name(method, env_name, seed, cfg_dict),
        tags=[method, "mujoco"],
        config={**cfg_dict, "method": method, "seed": seed, "env": env_name},
        job_type="smoke" if total_steps <= 10000 else "train",
        mode=os.environ.get("WANDB_MODE", "online"),
        reinit="finish_previous",
    )
    return True


def finish_run(exit_code: int = 0) -> None:
    wandb.finish(exit_code=exit_code)


# ---------------------------------------------------------------------------
# Metric logging
# ---------------------------------------------------------------------------


def log_metrics(data: dict, step: int) -> None:
    wandb.log(data, step=step)


def log_eval_summary(
    mean_return: float,
    eval_results: list[dict],
    all_eval_returns: list[float],
    eval_count: int,
    global_step: int,
    *,
    r_rand: float | None = None,
    consecutive_below: int | None = None,
    best_return: float | None = None,
    steps_to_best: int | None = None,
) -> None:
    returns = [r["return"] for r in eval_results]
    lengths = [r["length"] for r in eval_results]
    successes = [r["success"] for r in eval_results if r.get("success") is not None]
    jerks = [r["jerk"] for r in eval_results if r.get("jerk") is not None]
    data = {
        "eval/mean_return": float(mean_return),
        "eval/min_return": float(np.min(returns)) if returns else 0.0,
        "eval/max_return": float(np.max(returns)) if returns else 0.0,
        "eval/std_return": float(np.std(returns)) if returns else 0.0,
    }
    if best_return is not None:
        data["eval/best_mean_return"] = float(best_return)
    if steps_to_best is not None:
        data["eval/steps_to_best"] = int(steps_to_best)
    if r_rand is not None:
        data["eval/random_baseline"] = float(r_rand)
    if len(all_eval_returns) >= 2 and all_eval_returns[-1] == mean_return:
        prev = all_eval_returns[-2]
        data["eval/reward_improvement"] = float(mean_return - prev)
        data["eval/reward_improvement_pct"] = float(
            (mean_return - prev) / max(abs(prev), 1e-8)
        )
    if consecutive_below is not None:
        data["eval/consecutive_below"] = int(consecutive_below)

    data.update(
        {
            "eval/mean_length": float(np.mean(lengths)) if lengths else 0.0,
            "eval/max_length": float(np.max(lengths)) if lengths else 0.0,
            "eval/std_length": float(np.std(lengths)) if lengths else 0.0,
            "eval/episode_count": int(eval_count),
        }
    )
    if successes:
        data["eval/success_rate"] = float(np.mean(successes))
    if jerks:
        data["eval/mean_jerk"] = float(np.mean(jerks)) if jerks else 0.0
        data["eval/max_jerk"] = float(np.max(jerks)) if jerks else 0.0
    if eval_results:
        data["eval/episodes"] = wandb.Table(
            columns=["episode", "return", "length", "success", "jerk"],
            data=[[r["episode"], r["return"], r["length"], r.get("success"), r.get("jerk")]
                  for r in eval_results],
        )
    log_metrics(data, step=global_step)


def log_eval_video(frames: list, global_step: int, *, fps: int = 30) -> None:
    if not frames:
        return
    try:
        video = np.stack([np.asarray(f, dtype=np.uint8) for f in frames])
    except (ValueError, TypeError):
        return
    if video.ndim != 4:
        return
    try:
        log_metrics({"eval/video": wandb.Video(video, fps=fps, format="mp4")}, step=global_step)
    except Exception:
        return

In [ ]:
%%writefile scripts/train.py
"""Single-run training CLI for SJFPO (PyTorch) and SJFPO (JAX).

One run per invocation; draccus builds the run config (singular values only).
Sweeps live in scripts/sweep.sh: one call here per env x seed.

    python scripts/train.py --method sjfpo --env Hopper-v5 --seed 0 --total_steps 2000
    python scripts/train.py --method sjfpo_jax --env Hopper-v5 --seed 0 --total_steps 2000
    python scripts/train.py --method sjfpo --env Hopper-v5 --sjfpo.policy_lr 3e-4
    python scripts/train.py --method sjfpo --env Hopper-v5 --seed 0 \
        --resume outputs/sjfpo_Hopper-v5_seed0/checkpoint_1000 \
        --total_steps 4000

Flags use underscores (draccus convention, same as lerobot-train).

Precedence: CLI flags > --config YAML (flat, legacy) > dataclass defaults.
The flat YAML only fills method fields the CLI left at defaults. Tuple/dict
fields (hidden, ...) are YAML-only (draccus takes one token each).

Exit codes: 0 success, 1 usage error, 2 runtime/divergence abort,
4 required artifact missing (draccus parse errors exit 2).
"""

from __future__ import annotations

import dataclasses
import json
import math
import os
import sys
import traceback
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import draccus

# Make src/ importable before any sjfpo.* import.
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from sjfpo.config import (  # noqa: E402
    KNOWN_ENVS,
    lookup_env_dims,
)
from sjfpo.configuration_sjfpo import SJFPOConfig  # noqa: E402


VALID_ENVS = KNOWN_ENVS


# ---------------------------------------------------------------------------
# Torch CUDA tuning (skipped when torch is not installed — JAX-only installs)
# ---------------------------------------------------------------------------


def configure_torch_cuda() -> bool:
    """Apply torch CUDA tuning when torch is importable and CUDA is present.

    Returns True when torch is usable. Never raises: a JAX-only environment
    simply skips this. Importing torch is deferred to this function so the
    JAX path doesn't pay the import cost.
    """
    try:
        import torch  # noqa: F401
    except ImportError:
        return False
    if not torch.cuda.is_available():
        return False
    torch.backends.cuda.matmul.fp32_precision = "ieee"
    torch.backends.cudnn.fp32_precision = "ieee"
    torch.backends.cudnn.conv.fp32_precision = "tf32"
    torch.backends.cudnn.rnn.fp32_precision = "tf32"
    torch.backends.cudnn.benchmark = True
    # TF32 for leftover fp32 GEMMs; autocast still handles bf16 matmuls.
    torch.set_float32_matmul_precision("high")
    # Memory allocation hint (harmless if already set by user env).
    os.environ.setdefault(
        "PYTORCH_CUDA_ALLOC_CONF",
        "max_split_size_mb:128,expandable_segments:True",
    )
    return True


# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------


@dataclass
class TrainConfig:
    """One training run (singular values only; sweep axes live in sweep.sh)."""

    method: Literal["sjfpo", "sjfpo_jax"] = "sjfpo"
    env: str | None = None
    seed: int = 3407
    total_steps: int | None = None
    eval_steps: int | None = None
    save_steps: int | None = None
    obs_dim: int | None = None
    action_dim: int | None = None
    out_dir: str = "outputs/"
    wandb_project: str | None = None
    wandb_enabled: bool = True
    wandb_run_name: str | None = None
    config: str | None = None  # legacy flat YAML overlay for the method config
    # Resume an sjfpo / sjfpo_jax checkpoint dir (checkpoint_<step>/ or
    # checkpoint_final/): continues policy + optimizer + counters into a FRESH
    # run dir (<base>_resume[/2...]). total_steps stays an absolute env-step
    # budget. sjfpo_jax resume requires a JAX-side checkpoint (see
    # sjfpo_jax.checkpointing).
    resume: str | None = None
    sjfpo: SJFPOConfig = field(default_factory=SJFPOConfig)


# ---------------------------------------------------------------------------
# YAML overlay + helpers
# ---------------------------------------------------------------------------


def load_yaml_dict(path: str | Path) -> dict:
    """Load YAML via safe_load only; friendly error when pyyaml is missing."""
    try:
        import yaml
    except ImportError as exc:
        raise SystemExit(
            "ERROR: pyyaml is required for --config (pip install pyyaml)"
        ) from exc
    with open(path, encoding="utf-8") as fh:
        data = yaml.safe_load(fh)
    if data is None:
        return {}
    if not isinstance(data, dict):
        raise SystemExit(f"ERROR: config {path} must be a YAML mapping")
    return data


def coerce_value(cfg_cls: type, raw: dict) -> dict:
    """Keep only fields the target dataclass knows; coerce list->tuple."""
    fields = {f.name for f in dataclasses.fields(cfg_cls)}
    out = {k: v for k, v in raw.items() if k in fields}
    for key in ("hidden",):
        if isinstance(out.get(key), list):
            out[key] = tuple(out[key])
    return out


def apply_flat_yaml(
    method_cfg, yaml_dict: dict, cli_fields: set[str] | None = None
):
    """Apply flat YAML after the dataclass defaults and before CLI overrides."""
    raw = coerce_value(type(method_cfg), yaml_dict)
    if not raw:
        return method_cfg
    fresh = type(method_cfg)()
    cli_touched = {
        f.name
        for f in dataclasses.fields(method_cfg)
        if getattr(method_cfg, f.name) != getattr(fresh, f.name)
    }
    cli_touched.update(cli_fields or ())
    overrides = {k: v for k, v in raw.items() if k not in cli_touched}
    return dataclasses.replace(method_cfg, **overrides) if overrides else method_cfg


def _sjfpo_cli_fields(argv: list[str]) -> set[str]:
    """Return SJFPO config fields explicitly present on the command line."""
    fields: set[str] = set()
    prefix = "--sjfpo."
    for index, token in enumerate(argv):
        if not token.startswith(prefix):
            continue
        name = token[len(prefix) :].split("=", 1)[0]
        if name:
            fields.add(name)
    return fields


def select_method_cfg(
    args: TrainConfig, yaml_dict: dict, cli_fields: set[str] | None = None
):
    """Nested draccus sub-config for the method, overlaid with flat YAML.

    ``sjfpo_jax`` reuses the sjfpo hyperparameter block (identical fields,
    identical semantics; only the execution backend differs).
    """
    method_cfg = {
        "sjfpo": args.sjfpo,
        "sjfpo_jax": args.sjfpo,
    }[args.method]
    return apply_flat_yaml(method_cfg, yaml_dict, cli_fields=cli_fields)


def default_intervals(
    total: int, args_val: int | None, yaml_dict: dict, key: str, cap: int
) -> int:
    """CLI > YAML > capped default (the cap keeps smoke runs sane)."""
    if args_val:
        return args_val
    return int(yaml_dict.get(key, min(cap, total)))


def resumed_cfg_overrides(cfg) -> list[str]:
    """Scalar fields differing from dataclass defaults.

    Used to warn with --resume: the checkpoint config wins there, so CLI
    overrides of these fields are silently ignored. Only plain scalars are
    compared (dict/tensor features have no meaningful CLI equality).
    """
    try:
        fresh = type(cfg)()
    except Exception:
        return []
    touched = []
    for f in dataclasses.fields(cfg):
        try:
            cur, default = getattr(cfg, f.name), getattr(fresh, f.name)
        except Exception:
            continue
        if isinstance(cur, (bool, int, float, str)) or cur is None:
            if isinstance(default, (bool, int, float, str)) or default is None:
                if cur != default:
                    touched.append(f.name)
    return sorted(touched)


def resolve_run_dir(
    out_base: str | Path, method: str, env: str, seed: int, resume: str | None
) -> Path:
    """Run dir for one invocation; resumed runs get a fresh suffixed dir.

    ``<method>_<env>_seed<seed>`` normally, else the first free
    ``<base>_resume``, ``<base>_resume2``, ... so a resume never overwrites
    the original run (or an earlier resume).
    """
    base = Path(out_base) / f"{method}_{env}_seed{seed}"
    if not resume:
        return base
    candidate = Path(str(base) + "_resume")
    i = 2
    while candidate.exists():
        candidate = Path(f"{base}_resume{i}")
        i += 1
    return candidate


# ---------------------------------------------------------------------------
# Run dispatch
# ---------------------------------------------------------------------------


def run_single(
    method: str,
    env_id: str,
    seed: int,
    out_dir: Path,
    cfg,
    total_steps: int,
    eval_steps: int,
    save_steps: int,
    obs_dim: int | None,
    action_dim: int | None,
    resume_from_checkpoint: str | None = None,
    wandb_enabled: bool = True,
) -> dict:
    """Dispatch one run to the matching backend's training entry."""
    import gymnasium as gym

    if env_id.startswith("AdroitHand"):
        try:
            import gymnasium_robotics  # noqa: F401 (registers Adroit env ids)
        except ImportError:
            raise RuntimeError(
                f"{env_id} needs gymnasium-robotics (pip install gymnasium-robotics)"
            )

    env = gym.make(env_id)
    try:
        # Registry fills unset dims; the live space is ground truth on mismatch.
        reg_obs, reg_act = lookup_env_dims(env_id)
        live_obs = int(math.prod(env.observation_space.shape))
        live_act = int(math.prod(env.action_space.shape))
        if (reg_obs, reg_act) != (live_obs, live_act):
            raise ValueError(
                f"registry dims {(reg_obs, reg_act)} != live {env_id} dims "
                f"{(live_obs, live_act)}; update configs/envs.yaml"
            )
        obs_dim = reg_obs if obs_dim is None else obs_dim
        action_dim = reg_act if action_dim is None else action_dim
        if (obs_dim, action_dim) != (live_obs, live_act):
            raise ValueError(
                f"requested dims {(obs_dim, action_dim)} != live {env_id} dims "
                f"{(live_obs, live_act)}"
            )

        # ---- JAX path -----------------------------------------------------
        if method == "sjfpo_jax":
            # Lazy import so torch-only users don't pay the JAX import cost.
            from sjfpo_jax.train_sjfpo import train_sjfpo_run as train_jax_run

            return train_jax_run(
                env=env,
                out_dir=out_dir,
                cfg=cfg,
                seed=seed,
                total_steps=total_steps,
                eval_steps=eval_steps,
                save_steps=save_steps,
                obs_dim=obs_dim,
                action_dim=action_dim,
                resume_from_checkpoint=resume_from_checkpoint,
                wandb_enabled=wandb_enabled,
            )

        # ---- Torch path ---------------------------------------------------
        from sjfpo.train_sjfpo import train_sjfpo_run

        if resume_from_checkpoint is not None and method != "sjfpo":
            raise ValueError(f"--resume is sjfpo-only, got method={method!r}")

        return train_sjfpo_run(
            env=env,
            out_dir=out_dir,
            cfg=cfg,
            seed=seed,
            total_steps=total_steps,
            eval_steps=eval_steps,
            save_steps=save_steps,
            obs_dim=obs_dim,
            action_dim=action_dim,
            resume_from_checkpoint=resume_from_checkpoint,
            wandb_enabled=wandb_enabled,
        )
    finally:
        env.close()


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------


def main(argv: list[str] | None = None) -> int:
    # Torch CUDA tuning is a no-op (and torch is not imported) if torch is
    # unavailable — the sjfpo_jax path works in a JAX-only install.
    configure_torch_cuda()

    argv = list(sys.argv[1:] if argv is None else argv)
    args = draccus.parse(config_class=TrainConfig, args=argv)
    if not args.env:
        print("ERROR: --env is required", file=sys.stderr)
        return 1
    if args.env not in VALID_ENVS:
        print(
            f"ERROR: unknown env {args.env!r}; choices: {', '.join(VALID_ENVS)}",
            file=sys.stderr,
        )
        return 1

    yaml_dict: dict = load_yaml_dict(args.config) if args.config else {}
    total = args.total_steps or int(yaml_dict.get("total_steps", 2000))

    # JAX eval is 12 full episodes: sparse defaults (capped by total for smokes).
    is_jax = args.method == "sjfpo_jax"
    eval_s = default_intervals(
        total, args.eval_steps, yaml_dict, "eval_steps", 20000 if is_jax else 1000
    )
    ckpt_i = default_intervals(
        total, args.save_steps, yaml_dict, "save_steps", 25000 if is_jax else 1000
    )
    cfg = select_method_cfg(args, yaml_dict, _sjfpo_cli_fields(argv))

    if args.resume and args.method not in ("sjfpo", "sjfpo_jax"):
        print(
            f"ERROR: --resume is sjfpo-only, got --method {args.method!r}",
            file=sys.stderr,
        )
        return 1
    if args.resume and not Path(args.resume).is_dir():
        print(f"ERROR: --resume dir not found: {args.resume}", file=sys.stderr)
        return 1
    if args.resume:
        overridden = resumed_cfg_overrides(cfg)
        if overridden:
            print(
                "WARNING: --resume restores the checkpoint config; "
                f"CLI overrides ignored for: {', '.join(overridden)}",
                file=sys.stderr,
            )

    # Same layout the old matrix used (sweep.format_run_directory_name),
    # suffixed when resuming so the source run is never overwritten.
    run_dir = resolve_run_dir(
        args.out_dir, args.method, args.env, args.seed, args.resume
    )
    run_dir.mkdir(parents=True, exist_ok=True)

    try:
        result = run_single(
            args.method,
            args.env,
            args.seed,
            run_dir,
            cfg,
            total,
            eval_s,
            ckpt_i,
            args.obs_dim,
            args.action_dim,
            args.resume,
            args.wandb_enabled,
        )
    except Exception as exc:
        traceback.print_exc(file=sys.stderr)
        print(
            json.dumps(
                {
                    "method": args.method,
                    "env": args.env,
                    "seed": args.seed,
                    "error": str(exc),
                }
            )
        )
        return 2

    for artifact in result.get("artifacts", []):
        if not Path(artifact).exists():
            print(f"MISSING ARTIFACT: {artifact}", file=sys.stderr)
            return 4
    print(json.dumps(result))
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile configs/envs.yaml
# State-based MuJoCo prototype ladder. `train.py --env <id>` resolves
# obs_dim/action_dim here; --obs-dim/--act-dim still override, and the live
# env space is validated against the registry at startup (mismatch = loud
# error, usually a gym version change — re-measure and update).
# Dims below MEASURED with gymnasium 1.x + mujoco on 2026-09-14.
CartPole-v1: {obs_dim: 4, action_dim: 2, tier: toy, requires: gymnasium}
Pendulum-v1: {obs_dim: 3, action_dim: 1, tier: smoke, requires: gymnasium}
MountainCar-v0: {obs_dim: 2, action_dim: 3, tier: toy, requires: gymnasium}
InvertedPendulum-v5: {obs_dim: 4, action_dim: 1, tier: smoke}
InvertedDoublePendulum-v5: {obs_dim: 9, action_dim: 1, tier: smoke}
Reacher-v5: {obs_dim: 10, action_dim: 2, tier: smoke}
Hopper-v5: {obs_dim: 11, action_dim: 3, tier: core}
HalfCheetah-v5: {obs_dim: 17, action_dim: 6, tier: core}
Walker2d-v5: {obs_dim: 17, action_dim: 6, tier: core}
Swimmer-v5: {obs_dim: 8, action_dim: 2, tier: core}
Pusher-v5: {obs_dim: 23, action_dim: 7, tier: hard}
Ant-v5: {obs_dim: 105, action_dim: 8, tier: hard}
Humanoid-v5: {obs_dim: 348, action_dim: 17, tier: hard}
AdroitHandDoor-v1: {obs_dim: 39, action_dim: 28, tier: dexterous, requires: gymnasium-robotics}
AdroitHandPen-v1: {obs_dim: 45, action_dim: 24, tier: dexterous, requires: gymnasium-robotics}
AdroitHandHammer-v1: {obs_dim: 46, action_dim: 26, tier: dexterous, requires: gymnasium-robotics}

In [ ]:
%%writefile configs/shared.yaml
# Shared experiment defaults — mirrors src/sjfpo/config.py RUN_MATRIX axes + budgets.
# Source of truth: config.py RunSpec/RUN_MATRIX (15 runs: 3 envs x 5 seeds).
# MuJoCo-only (flat Box). fp32-only. CUDA-default with CPU fallback.
run_matrix:
  methods: [sjfpo]
  envs: [HalfCheetah-v5, Walker2d-v5, Hopper-v5]
  seeds: [0, 1, 2, 3, 4]
  total_steps: 1000000
  eval_steps: 50000
  save_steps: 25000
tiny:
  env: Hopper-v5
  seed: 0
  steps:
    sjfpo: 20000
  eval_steps: 10000
  save_steps: 10000
eval:
  eval_steps: 50000
  save_steps: 25000
  max_episodes: 12
output:
  out_dir: outputs/
# wandb: enabled by default; SJFPO_DISABLE_WANDB=1 is the only off switch (project = algo name at runtime).
# Never log secrets. See src/sjfpo/wandb_logging.py.

In [ ]:
%%writefile configs/sjfpo.yaml
# SJFPO configuration. Keep values aligned with SJFPOConfig.
# Runtime overrides may come from CLI flags; this file is the experiment YAML source.

# -- Observation / action geometry -------------------------------------
n_obs_steps: 16
action_horizon: 16
n_action_steps: 16
action_perturb_std: 0.02
hidden_dim: 128

# -- Monte-Carlo / rollout ---------------------------------------------
num_mc_samples: 16
num_euler_steps: 16
num_inference_steps: 16
chunks_per_update: 256
num_epochs: 16
num_minibatches: 4
compile_batch_size: 0
num_envs: 64

# -- Optimization ------------------------------------------------------
gamma: 0.999
gae_lambda: 0.995
epsilon_clip: 0.05
use_dct_transform: false
trust_region_mode: aspo
cfm_loss_reduction: mean
ema_decay: 0.95
ema_warmup_steps: 500
use_per_bin_clip: false
epsilon_clip_lowfreq: 0.02
epsilon_clip_highfreq: 0.10
cfm_clamp: 20.0
log_ratio_cap: 10.0
advantage_clamp: 100.0
normalize_advantage: true
use_clipped_value_loss: false
min_timestep: 0.005
max_timestep: 0.995
timestep_sampling_mode: fpo_beta
noise_sampling_mode: normal
timestep_inverse_cdf_beta: 1.0
timestep_beta_alpha: 2.0
timestep_beta_beta: 2.0

# -- DiT backbone ------------------------------------------------------
dit_depth: 4
dit_heads: 4
dit_mlp_ratio: 4.0
time_embed_dim: 128
timestep_scale: 1000.0

# -- Losses / optimization --------------------------------------------
vf_coef: 1.0
grad_clip_norm: 0.5
top_p_spectral_tokens: 1.0
policy_lr: 3.0e-4
critic_lr: 9.0e-4
optimizer_weight_decay: 1.0e-2
use_gradient_checkpointing: false
grad_accum_steps: 1

In [ ]:
%%writefile requirements-colab.txt
# Deep Learning
torch
torchvision
torchaudio
torchcodec
jax
jaxlib
accelerate
diffusers
datasets
huggingface-hub
safetensors
bitsandbytes
einops
einx

# Reinforcement Learning / Robot Learning
gymnasium
gymnasium-robotics
pettingzoo
mujoco
skrl
lerobot

# Scientific / Vision
numpy
scipy
pandas
matplotlib
scikit-learn
scikit-image
opencv-python-headless
pillow
imageio
imageio-ffmpeg
av
moviepy

# Data / Storage
pyarrow

# Experiment Tracking / Config
wandb
tensorboard
draccus


In [ ]:
# Detect runtime and choose install source.
ENV_NAME = detect_env()
if ENV_NAME == "colab":
    !uv pip install -r requirements-colab.txt
else:
    WHEELS_DIR = "/kaggle/input/sjfpo-packages"
    !pip install -q --no-index --find-links=$WHEELS_DIR -r requirements-linux.txt

In [ ]:
# wandb login from secret store (runs before any training cell).
_wandb_key = load_secret("WANDB_API_KEY")
ifwandb_key:
    !wandb login $_wandb_key
else:
    print("WANDB_API_KEY not found; training will run without wandb sync.")

In [ ]:
# Smoke: 200k steps, single SJFPO run on Hopper.
!python scripts/train.py     --method sjfpo     --env Hopper-v5     --seed 3407     --total_steps 200000     --eval_steps 50     --save_steps 10000     --use_per_bin_clip true     --top_p_spectral_tokens 1.0

In [ ]:
# Full run: 200k steps with 50 eval / 10k save.
!python scripts/train.py     --method sjfpo     --env Hopper-v5     --seed 3407     --total_steps 200000     --eval_steps 50     --save_steps 10000     --use_per_bin_clip true     --top_p_spectral_tokens 1.0